# Explainable GeoAI analysis of autism prevalence

This notebook contains the analysis workflow for the manuscript **“A Multi-scale Explainable GeoAI Framework for Discovering Spatiotemporal Heterogeneity in Environmental Profiles Associated with Autism Prevalence.”**

### Analysis overview
- Study period: 2010–2022
- Unit of analysis: New York State school-district-year
- Missing-data approach: complete-case analysis; no outcome or predictor imputation
- Final model: LightGBM selected after comparison of 12 machine-learning algorithms
- Explainability: SHAP and LIME
- Spatial analyses: dominant-predictor mapping, environmental signatures, Local Moran's I, and spatial-block cross-validation
- Reproducibility seed: 42

This public-release version improves organization, documentation, and path handling **without changing the analytical logic, model specifications, random seeds, or output filenames** used in the revised analysis.

## Repository layout and path configuration

The notebook uses paths relative to the GitHub repository rather than author-specific absolute paths. The expected layout is:

```text
ASD-Explainable-GeoAI/
├── ASD_GeoAI_revision_analysis_2026_v1.ipynb
├── data/
│   ├── autism_fulldata041926.csv
│   ├── processed/
│   └── spatial/
│       └── tl_2020_36_unsd/
│           ├── tl_2020_36_unsd.shp
│           └── ... other shapefile sidecar files
└── results/
    ├── tables/
    └── figures/
```

Run the notebook from the repository root. If it is opened from a `notebooks/` subdirectory, the configuration cell below automatically checks the parent directory. Users running the code from another location can set `PROJECT_ROOT` to their cloned repository directory.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------
# Project-wide paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

# Support the common case in which the notebook is moved into
# a repository-level "notebooks" directory.
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_FILE = DATA_DIR / "autism_fulldata041926.csv"
PROCESSED_DIR = DATA_DIR / "processed"
SPATIAL_DIR = DATA_DIR / "spatial" / "tl_2020_36_unsd"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

# Create directories generated by this workflow.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data:     {RAW_DATA_FILE}")
print(f"Tables:       {TABLES_DIR}")
print(f"Figures:      {FIGURES_DIR}")

## 1. Prepare the observed ASD dataset

Read the source data, restrict to 2010–2022, calculate ASD prevalence per 1,000 students, and retain observed values without imputation.

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. Input and output paths
# ============================================================

# Raw source dataset. All project paths are defined once in the
# configuration cell above and are relative to the repository root.
input_file = str(RAW_DATA_FILE)

# Processed-data directory and first derived dataset.
processed_dir = str(PROCESSED_DIR)
output_file = str(PROCESSED_DIR / "autism_observed_data.csv")

# ============================================================
# 2. Read original data
# ============================================================

df = pd.read_csv(input_file)

# Clean column names
df.columns = df.columns.str.strip()

print("=" * 70)
print("ORIGINAL DATA")
print("=" * 70)

print("Input file:")
print(input_file)

print("\nNumber of rows:", len(df))
print("Number of columns:", len(df.columns))


# ============================================================
# 3. Check required columns
# ============================================================

required_cols = [
    "Year",
    "GEOID",
    "District Name",
    "Autism",
    "Total Students"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print("\nAll required columns are present.")


# ============================================================
# 4. Convert key variables to numeric
# ============================================================

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

df["Autism"] = pd.to_numeric(
    df["Autism"],
    errors="coerce"
)

df["Total Students"] = pd.to_numeric(
    df["Total Students"],
    errors="coerce"
)


# ============================================================
# 5. Keep study period 2010–2022
# ============================================================

df = df[
    (df["Year"] >= 2010) &
    (df["Year"] <= 2022)
].copy()

print("\nRecords during 2010–2022:", len(df))


# ============================================================
# 6. Check original ASD data
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL ASD DATA CHECK")
print("=" * 70)

n_total = len(df)

n_missing_autism = df["Autism"].isna().sum()
n_missing_students = df["Total Students"].isna().sum()

print(
    "Missing Autism:",
    n_missing_autism,
    f"({n_missing_autism / n_total * 100:.2f}%)"
)

print(
    "Missing Total Students:",
    n_missing_students,
    f"({n_missing_students / n_total * 100:.2f}%)"
)

print(
    "Autism = 0:",
    (df["Autism"] == 0).sum()
)

print(
    "Autism < 0:",
    (df["Autism"] < 0).sum()
)

print(
    "Total Students <= 0:",
    (df["Total Students"] <= 0).sum()
)


# ============================================================
# 7. Calculate autism prevalence per 1,000 STUDENTS
# ============================================================

# Autism prevalence per 1,000 students:
#
# Autism / Total Students × 1000
#
# IMPORTANT:
# - NO IMPUTATION
# - Missing Autism remains missing
# - Missing Total Students remains missing
# - Missing Autism is NOT treated as zero

df["Autism_prev1000"] = np.where(
    (
        df["Autism"].notna()
        & df["Total Students"].notna()
        & (df["Total Students"] > 0)
    ),
    df["Autism"] / df["Total Students"] * 1000,
    np.nan
)


# ============================================================
# 8. QC calculated prevalence
# ============================================================

print("\n" + "=" * 70)
print("AUTISM PREVALENCE QC")
print("=" * 70)

print(
    "Observed prevalence records:",
    df["Autism_prev1000"].notna().sum()
)

print(
    "Missing prevalence records:",
    df["Autism_prev1000"].isna().sum()
)

print(
    "Zero prevalence records:",
    (df["Autism_prev1000"] == 0).sum()
)

print("\nAutism prevalence summary:")
print(
    df["Autism_prev1000"].describe()
)


# ============================================================
# 9. Check missing/zero prevalence by year
# ============================================================

annual_check = (
    df.groupby("Year")
    .agg(
        Total_districts=(
            "GEOID",
            "size"
        ),
        Observed_ASD=(
            "Autism_prev1000",
            "count"
        ),
        Missing_ASD=(
            "Autism_prev1000",
            lambda x: x.isna().sum()
        ),
        Zero_prevalence=(
            "Autism_prev1000",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

annual_check["Missing_percent"] = (
    annual_check["Missing_ASD"]
    / annual_check["Total_districts"]
    * 100
)

annual_check["Zero_percent_among_observed"] = (
    annual_check["Zero_prevalence"]
    / annual_check["Observed_ASD"]
    * 100
)

print("\n" + "=" * 70)
print("ANNUAL CHECK")
print("=" * 70)

print(
    annual_check.to_string(index=False)
)


# ============================================================
# 10. Calculate annual overall prevalence
#     per 1,000 STUDENTS
# ============================================================

annual_results = []

for year, sub in df.groupby("Year"):

    # Use only genuinely observed ASD records
    valid = sub[
        sub["Autism"].notna()
        & sub["Total Students"].notna()
        & (sub["Total Students"] > 0)
    ].copy()

    total_autism = valid["Autism"].sum()
    total_students = valid["Total Students"].sum()

    overall_prev = (
        total_autism / total_students * 1000
        if total_students > 0
        else np.nan
    )

    annual_results.append({
        "Year": int(year),
        "Observed districts": len(valid),
        "Total Autism": total_autism,
        "Total Students": total_students,
        "Overall prevalence per 1,000 students": overall_prev
    })

annual_prevalence = pd.DataFrame(
    annual_results
)

print("\n" + "=" * 70)
print("ANNUAL OVERALL ASD PREVALENCE")
print("=" * 70)

print(
    annual_prevalence.to_string(index=False)
)


# ============================================================
# 11. Save processed dataset
# ============================================================

# Save in the EXISTING "processed data" folder.
#
# All original columns are retained.
# Autism_prev1000 is added.
# NO imputation is performed.

df.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 70)
print("DATA SAVED")
print("=" * 70)

print(output_file)


# ============================================================
# 12. Final preview
# ============================================================

print("\n" + "=" * 70)
print("PREVIEW")
print("=" * 70)

print(
    df[
        [
            "Year",
            "GEOID",
            "District Name",
            "Autism",
            "Total Students",
            "Autism_prev1000"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nDONE.")
print("Autism prevalence = Autism / Total Students × 1,000 students")
print("NO IMPUTATION WAS PERFORMED.")

## 2. Figure S2: temporal distribution of ASD prevalence

Create the annual boxplots using observed district-level ASD prevalence.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os

# ==============================
# 1. File paths
# ==============================

# Observed, non-imputed dataset
# NO imputation
file_path = str(PROCESSED_DIR / 'autism_observed_data.csv')

# Figure S2 output
output_fig = str(FIGURES_DIR / 'Figure_S2_autism_prevalence_by_year_final.tiff')

# ==============================
# 2. Style
# ==============================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9
})

sns.set_style("white")

# ==============================
# 3. Load observed data
# ==============================
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

df["Autism_prev1000"] = pd.to_numeric(
    df["Autism_prev1000"],
    errors="coerce"
)

# Only observed ASD prevalence
# Missing ASD records are excluded.
# True zero values remain in the analysis.
df = df.dropna(
    subset=["Year", "Autism_prev1000"]
).copy()

df["Year"] = df["Year"].astype(int)

year_order = sorted(
    df["Year"].unique()
)

# ==============================
# 4. Figure S2
# ==============================
fig, ax = plt.subplots(
    figsize=(10.5, 4.4)
)

sns.boxplot(
    data=df,
    x="Year",
    y="Autism_prev1000",
    order=year_order,
    color="#d9d9d9",
    width=0.6,
    linewidth=1.0,
    showfliers=False,
    boxprops=dict(
        edgecolor="#3a3a3a"
    ),
    whiskerprops=dict(
        color="#3a3a3a"
    ),
    capprops=dict(
        color="#3a3a3a"
    ),
    medianprops=dict(
        color="#b22222",
        linewidth=1.3
    ),
    ax=ax
)

# ==============================
# 5. Axis
# ==============================
ax.set_ylabel(
    "Autism prevalence (per 1,000 students)"
)

ax.set_xlabel("")

# ==============================
# 6. Grid
# ==============================
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.6,
    color="#e0e0e0",
    alpha=0.8
)

ax.grid(
    axis="x",
    visible=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ==============================
# 7. Legend
# ==============================
legend_elements = [
    Line2D(
        [0],
        [0],
        color="#b22222",
        lw=1.3,
        label="Median"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper left",
    frameon=False
)

# ==============================
# 8. Save Figure S2
# ==============================
plt.tight_layout()

plt.savefig(
    output_fig,
    dpi=300,
    format="tiff"
)

plt.show()

print("INPUT DATA:")
print(file_path)

print("\nFigure S2 saved to:")
print(output_fig)

## 3. Table S4: annual prevalence-category distribution

Summarize school districts by ASD prevalence category and calculate statewide annual prevalence.

In [ ]:
import pandas as pd
import numpy as np
import os

# =========================
# 1. File paths
# =========================

file_path = str(PROCESSED_DIR / 'autism_observed_data.csv')

output_path = str(TABLES_DIR / 'Table_S4_autism_distribution_2010_2022.csv')

# =========================
# 2. Read data
# =========================

df = pd.read_csv(file_path)

# Clean column names
df.columns = df.columns.str.strip()

# =========================
# 3. Keep needed columns
# =========================

needed_cols = [
    "GEOID",
    "Year",
    "Autism",
    "Total Students",
    "Autism_prev1000"
]

missing_cols = [
    col for col in needed_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

df = df[needed_cols].copy()

# =========================
# 4. Convert types
# =========================

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
).astype("Int64")

df["Autism"] = pd.to_numeric(
    df["Autism"],
    errors="coerce"
)

df["Total Students"] = pd.to_numeric(
    df["Total Students"],
    errors="coerce"
)

df["Autism_prev1000"] = pd.to_numeric(
    df["Autism_prev1000"],
    errors="coerce"
)

# Keep 2010–2022
df = df[
    (df["Year"] >= 2010) &
    (df["Year"] <= 2022)
].copy()

# =========================
# 5. Use observed ASD records only
# =========================

# Missing ASD outcome is excluded.
# True zero prevalence remains included.
df = df.dropna(
    subset=[
        "Year",
        "Autism",
        "Total Students",
        "Autism_prev1000"
    ]
).copy()

# Exclude invalid enrollment if any
df = df[
    df["Total Students"] > 0
].copy()

print("Observed district-year records used:", len(df))

# =========================
# 6. Classification
# =========================
#
# Display labels:
# 0
# 0–5
# 5–10
# 10–15
# 15–20
# >20
#
# Actual mathematical intervals:
# 0
# (0, 5]
# (5, 10]
# (10, 15]
# (15, 20]
# >20

def classify_prevalence(x):
    if x == 0:
        return "0"
    elif 0 < x <= 5:
        return "0–5"
    elif 5 < x <= 10:
        return "5–10"
    elif 10 < x <= 15:
        return "10–15"
    elif 15 < x <= 20:
        return "15–20"
    elif x > 20:
        return ">20"
    else:
        return np.nan

df["category"] = df["Autism_prev1000"].apply(
    classify_prevalence
)

# =========================
# 7. Check classification
# =========================

unclassified = df[
    df["category"].isna()
]

print("Unclassified records:", len(unclassified))

if len(unclassified) > 0:
    print(
        unclassified[
            [
                "GEOID",
                "Year",
                "Autism_prev1000"
            ]
        ].head(20)
    )

# =========================
# 8. Annual summary
# =========================

category_order = [
    "0",
    "0–5",
    "5–10",
    "10–15",
    "15–20",
    ">20"
]

results = []

for year, sub in df.groupby("Year"):

    total_districts = len(sub)

    # -------------------------------------
    # Overall prevalence per 1,000 STUDENTS
    #
    # NOT mean district prevalence
    #
    # sum(Autism) / sum(Total Students) × 1000
    # -------------------------------------

    total_autism = sub["Autism"].sum()
    total_students = sub["Total Students"].sum()

    overall_prev = (
        total_autism /
        total_students *
        1000
    )

    # Count districts in each category
    cat_counts = (
        sub["category"]
        .value_counts()
        .reindex(
            category_order,
            fill_value=0
        )
    )

    row = {
        "Year": int(year),
        "Total districts": total_districts
    }

    for cat in category_order:

        count = int(cat_counts[cat])

        pct = (
            count /
            total_districts *
            100
        )

        row[cat] = (
            f"{count} ({pct:.1f}%)"
        )

    row["Overall prevalence (‰)"] = round(
        overall_prev,
        2
    )

    results.append(row)

# =========================
# 9. Create Table S4
# =========================

summary_table = (
    pd.DataFrame(results)
    .sort_values("Year")
    .reset_index(drop=True)
)

# =========================
# 10. Check totals
# =========================

print("\nTABLE S4")
print(summary_table.to_string(index=False))

print("\nDistrict counts by year:")

for year, sub in df.groupby("Year"):

    category_total = (
        sub["category"]
        .value_counts()
        .sum()
    )

    print(
        int(year),
        "Total districts =",
        len(sub),
        "| Category total =",
        category_total
    )

# =========================
# 11. Save
# =========================

summary_table.to_csv(
    output_path,
    index=False
)

print("\nSaved to:")
print(output_path)

## 4. Construct the complete-case candidate-predictor dataset

Create the age 5–14 variable, retain 14 candidate predictors, and apply complete-case restriction.

In [ ]:
import pandas as pd

# ==============================
# 1. File paths
# ==============================

input_path = str(PROCESSED_DIR / 'autism_observed_data.csv')

output_path = str(PROCESSED_DIR / 'autism_selected_features.csv')


# ==============================
# 2. Load observed data
# ==============================

df = pd.read_csv(input_path)

# Clean column names
df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print("Input file:")
print(input_path)

print("\nOriginal dimensions:")
print(df.shape)


# ==============================
# 3. Check required columns
# ==============================

required_columns = [
    "GEOID",
    "Year",
    "Autism_prev1000",

    # Air pollution
    "PM25",

    # Climate
    "tmax_c",
    "ppt_mm",

    # Socioeconomic
    "Below_poverty_percent",
    "Bachelor_plus_percent",

    # Demographic
    "median_age",
    "under5_percent",
    "age5to9_percent",
    "age10to14_percent",
    "age15to19_percent",

    # Urbanization
    "Pop_density",

    # Race / ethnicity
    "asian_percent",
    "hispanic_percent",

    # Land cover
    "Developed_percent",
    "Forest_percent"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nAll required columns are present.")


# ==============================
# 4. Convert variables to numeric
# ==============================

numeric_columns = [
    "Year",
    "Autism_prev1000",
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "median_age",
    "under5_percent",
    "age5to9_percent",
    "age10to14_percent",
    "age15to19_percent",
    "Pop_density",
    "asian_percent",
    "hispanic_percent",
    "Developed_percent",
    "Forest_percent"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ==============================
# 5. Keep study period 2010–2022
# ==============================

df = df[
    (df["Year"] >= 2010) &
    (df["Year"] <= 2022)
].copy()

print("\nRecords during 2010–2022:")
print(len(df))


# ==============================
# 6. Construct age 5–14 variable
# ==============================

df["age5to14_percent"] = (
    df["age5to9_percent"]
    + df["age10to14_percent"]
)


# ==============================
# 7. Define 14 predictors
# ==============================

predictors_14 = [
    # Air pollution
    "PM25",

    # Climate
    "tmax_c",
    "ppt_mm",

    # Socioeconomic
    "Below_poverty_percent",
    "Bachelor_plus_percent",

    # Demographic
    "median_age",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",

    # Urbanization
    "Pop_density",

    # Race / ethnicity
    "asian_percent",
    "hispanic_percent",

    # Land cover
    "Developed_percent",
    "Forest_percent"
]

print("\nNumber of candidate predictors:")
print(len(predictors_14))


# ==============================
# 8. Select analysis columns
# ==============================

selected_columns = [
    "GEOID",
    "Year",
    "Autism_prev1000"
] + predictors_14

df_selected = df[
    selected_columns
].copy()


# ==============================
# 9. Check missing values
#    BEFORE complete-case deletion
# ==============================

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE COMPLETE-CASE SELECTION")
print("=" * 70)

missing_summary = (
    df_selected
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary)

n_before = len(df_selected)

print("\nRows before complete-case selection:")
print(n_before)


# ==============================
# 10. Complete-case selection
# ==============================

# No imputation.
# Remove observations missing:
# - ASD outcome
# - any of the 14 candidate predictors
#
# GEOID and Year are also required.

df_selected = df_selected.dropna(
    subset=selected_columns
).copy()

n_after = len(df_selected)
n_removed = n_before - n_after

print("\n" + "=" * 70)
print("COMPLETE-CASE SELECTION")
print("=" * 70)

print("Rows before:", n_before)
print("Rows removed:", n_removed)
print("Rows remaining:", n_after)

print(
    "Percentage removed:",
    f"{n_removed / n_before * 100:.2f}%"
)


# ==============================
# 11. Check final missing values
# ==============================

print("\n" + "=" * 70)
print("FINAL MISSING-VALUE CHECK")
print("=" * 70)

final_missing = df_selected.isna().sum()

print(final_missing)

print(
    "\nTotal missing values in final dataset:",
    final_missing.sum()
)


# ==============================
# 12. Check sample size by year
# ==============================

annual_n = (
    df_selected
    .groupby("Year")
    .size()
    .reset_index(name="N")
)

print("\n" + "=" * 70)
print("FINAL SAMPLE SIZE BY YEAR")
print("=" * 70)

print(
    annual_n.to_string(index=False)
)


# ==============================
# 13. Save complete-case dataset
# ==============================

df_selected.to_csv(
    output_path,
    index=False
)


# ==============================
# 14. Final summary
# ==============================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print("\nVariables:")
print(df_selected.columns.tolist())

print("\nNumber of predictors:")
print(len(predictors_14))

print("\nFinal dimensions:")
print(df_selected.shape)

print("\nOutput saved to:")
print(output_path)

print("\nNO IMPUTATION WAS PERFORMED.")

## 5. Figure S3: predictor correlation screening

Calculate the 14-predictor correlation matrix used to identify redundant variables.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ==============================
# 1. File paths
# ==============================

# Complete-case dataset containing
# 14 candidate predictors
input_path = str(PROCESSED_DIR / 'autism_selected_features.csv')

# Figure S3
output_fig = str(FIGURES_DIR / 'Figure_S3_correlation_heatmap.tiff')

# Correlation matrix data
output_csv = str(TABLES_DIR / 'correlation_matrix_14_predictors.csv')


# ==============================
# 2. Global settings
# ==============================

plt.rcParams.update({
    "font.family": "Arial",
    "axes.unicode_minus": True,
    "font.size": 11,
    "mathtext.default": "regular"
})

sns.set_style("white")


# ==============================
# 3. Load complete-case data
# ==============================

df = pd.read_csv(input_path)

# Clean column names
df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_path)

print("\nData dimensions:")
print(df.shape)


# ==============================
# 4. Define the 14 predictors
# ==============================

predictors_14 = [
    # Air pollution
    "PM25",

    # Climate
    "tmax_c",
    "ppt_mm",

    # Socioeconomic
    "Below_poverty_percent",
    "Bachelor_plus_percent",

    # Demographic
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",

    # Urbanization
    "Pop_density",

    # Race / ethnicity
    "hispanic_percent",
    "asian_percent",

    # Land cover
    "Developed_percent",
    "Forest_percent"
]


# ==============================
# 5. Check variables
# ==============================

missing_variables = [
    col for col in predictors_14
    if col not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Missing predictors: {missing_variables}"
    )

print("\nNumber of predictors:")
print(len(predictors_14))

print("\nPredictors:")
print(predictors_14)


# ==============================
# 6. Keep ONLY the 14 predictors
# ==============================

# IMPORTANT:
# Autism_prev1000 is NOT included here,
# because this analysis is predictor-predictor
# correlation screening.

df_corr = df[predictors_14].copy()


# ==============================
# 7. Final missing-value check
# ==============================

print("\n" + "=" * 70)
print("MISSING-VALUE CHECK")
print("=" * 70)

print(df_corr.isna().sum())

print(
    "\nTotal missing values:",
    df_corr.isna().sum().sum()
)


# ==============================
# 8. Rename variables for figure
# ==============================

rename_dict = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Developed_percent":
        "Developed land cover (%)",

    "Forest_percent":
        "Forest land cover (%)"
}

df_corr = df_corr.rename(
    columns=rename_dict
)


# ==============================
# 9. Reorder variables
# ==============================

variable_order = [

    r"PM$_{2.5}$ concentration (µg/m³)",

    "Mean daily maximum temperature (°C)",
    "Annual precipitation (mm)",

    "Families below the poverty line (%)",
    "Bachelor’s degree or higher (%)",

    "Population under 5 years (%)",
    "Population aged 5–14 years (%)",
    "Population aged 15–19 years (%)",
    "Median age (years)",

    "Population density (persons/km²)",

    "Hispanic or Latino population (%)",
    "Asian population (%)",

    "Developed land cover (%)",
    "Forest land cover (%)"
]

df_corr = df_corr[
    variable_order
].copy()


# ==============================
# 10. Compute Pearson correlation
# ==============================

corr = df_corr.corr(
    method="pearson"
)


# ==============================
# 11. Save full correlation matrix
# ==============================

corr.to_csv(
    output_csv
)

print("\n" + "=" * 70)
print("CORRELATION MATRIX SAVED")
print("=" * 70)

print(output_csv)


# ==============================
# 12. Specifically check
#     population density vs developed land
# ==============================

r_pop_dev = corr.loc[
    "Population density (persons/km²)",
    "Developed land cover (%)"
]

print("\n" + "=" * 70)
print("KEY CORRELATION")
print("=" * 70)

print(
    "Population density vs Developed land cover:"
)

print(
    f"Pearson r = {r_pop_dev:.3f}"
)


# ==============================
# 13. Find all correlations |r| >= 0.8
# ==============================

print("\nStrong correlations (|r| >= 0.80):")

strong_pairs = []

for i in range(len(corr.columns)):
    for j in range(i):

        r = corr.iloc[i, j]

        if abs(r) >= 0.80:

            strong_pairs.append(
                (
                    corr.index[i],
                    corr.columns[j],
                    r
                )
            )

if strong_pairs:

    for var1, var2, r in strong_pairs:

        print(
            f"{var1}  <->  {var2}: r = {r:.3f}"
        )

else:
    print("None")


# ==============================
# 14. Mask upper triangle + diagonal
# ==============================

mask = np.triu(
    np.ones_like(
        corr,
        dtype=bool
    )
)


# ==============================
# 15. Annotate only strong correlations
# ==============================

annot = corr.copy().round(2).astype(str)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):

        if (
            abs(corr.iloc[i, j]) < 0.80
            or mask[i, j]
        ):
            annot.iloc[i, j] = ""


# ==============================
# 16. Plot Figure S3
# ==============================

plt.figure(
    figsize=(10.5, 8.8)
)

ax = sns.heatmap(
    corr,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    annot=annot,
    fmt="",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={
        "shrink": 0.8,
        "label": "Pearson correlation coefficient"
    }
)

# Keep labels; remove tick marks
ax.tick_params(
    axis="x",
    rotation=45,
    labelrotation=45,
    length=0
)

ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)

plt.xticks(
    ha="right"
)

plt.yticks(
    rotation=0
)

# No title
plt.tight_layout()


# ==============================
# 17. Save Figure S3
# ==============================

plt.savefig(
    output_fig,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()


# ==============================
# 18. Final output
# ==============================

print("\n" + "=" * 70)
print("FIGURE S3 SAVED")
print("=" * 70)

print(output_fig)

## 6. Table S6: variance inflation factors

Calculate VIFs for the 13 predictors retained after correlation screening.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt

# ==============================
# 1. File paths
# ==============================

# Complete-case dataset with 14 candidate predictors
input_path = str(PROCESSED_DIR / 'autism_selected_features.csv')

# Save final VIF table
output_vif_csv = str(TABLES_DIR / 'Table_S6_VIF_results.csv')

# Save VIF figure
output_vif_fig = str(FIGURES_DIR / 'VIF_plot_13_predictors.tiff')


# ==============================
# 2. Load data
# ==============================

df = pd.read_csv(input_path)
df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_path)

print("\nData dimensions:")
print(df.shape)


# ==============================
# 3. Final 13 predictors
# ==============================

# Developed_percent is excluded
# based on its high correlation with Pop_density.

selected_predictors = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]

print("\nNumber of predictors:")
print(len(selected_predictors))


# ==============================
# 4. Check required variables
# ==============================

missing_predictors = [
    col for col in selected_predictors
    if col not in df.columns
]

if missing_predictors:
    raise ValueError(
        f"Missing predictors: {missing_predictors}"
    )


# ==============================
# 5. Prepare VIF dataset
# ==============================

X = df[selected_predictors].copy()

# Convert to numeric
for col in selected_predictors:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

print("\nMissing values before VIF:")
print(X.isna().sum())

# Dataset should already be complete-case,
# but keep this as a safety check.
X = X.dropna().copy()

print("\nData shape for VIF analysis:")
print(X.shape)


# ==============================
# 6. Add constant
# ==============================

X_const = sm.add_constant(X)


# ==============================
# 7. Compute VIF
# ==============================

vif_data = pd.DataFrame()

vif_data["Variable"] = X_const.columns

vif_data["VIF"] = [
    variance_inflation_factor(
        X_const.values,
        i
    )
    for i in range(X_const.shape[1])
]


# ==============================
# 8. Remove constant
# ==============================

vif_data = vif_data[
    vif_data["Variable"] != "const"
].copy()

# Round
vif_data["VIF"] = vif_data["VIF"].round(2)

# Sort descending
vif_data = (
    vif_data
    .sort_values(
        by="VIF",
        ascending=False
    )
    .reset_index(drop=True)
)


# ==============================
# 9. Rename variables
# ==============================

rename_dict = {

    "PM25":
        "PM2.5 concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}

vif_data["Variable"] = (
    vif_data["Variable"]
    .map(rename_dict)
)


# ==============================
# 10. Save VIF table
# ==============================

vif_data.to_csv(
    output_vif_csv,
    index=False
)

print("\n" + "=" * 70)
print("VIF RESULTS")
print("=" * 70)

print(vif_data.to_string(index=False))

print("\nVIF table saved to:")
print(output_vif_csv)


# ==============================
# 11. Check threshold
# ==============================

max_vif = vif_data["VIF"].max()

print("\nMaximum VIF:")
print(max_vif)

if max_vif < 5:
    print("All predictors have VIF < 5.")
else:
    print("Warning: At least one predictor has VIF >= 5.")


# ==============================
# 12. Plot VIF
# ==============================

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

plt.figure(
    figsize=(8.5, 6.5)
)

plt.barh(
    vif_data["Variable"][::-1],
    vif_data["VIF"][::-1]
)

# Threshold lines
plt.axvline(
    x=5,
    linestyle="--",
    linewidth=1
)

plt.axvline(
    x=10,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Variance inflation factor (VIF)"
)

plt.ylabel("")

plt.tight_layout()


# ==============================
# 13. Save figure
# ==============================

plt.savefig(
    output_vif_fig,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()

print("\nVIF plot saved to:")
print(output_vif_fig)

## 7. Build the final modeling dataset

Retain the outcome, identifiers, and 13 final predictors used throughout model development.

In [ ]:
import pandas as pd

# ==============================
# 1. File paths
# ==============================

# Complete-case dataset with 14 candidate predictors
input_path = str(PROCESSED_DIR / 'autism_selected_features.csv')

# Final modeling dataset with 13 predictors
output_path = str(PROCESSED_DIR / 'autism_model_final.csv')


# ==============================
# 2. Load data
# ==============================

df = pd.read_csv(input_path)
df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_path)

print("\nInput dimensions:")
print(df.shape)


# ==============================
# 3. Define final variables
# ==============================

# Developed_percent is removed
# because of its strong correlation
# with population density.

final_columns = [

    # Identification
    "GEOID",
    "Year",

    # Outcome
    "Autism_prev1000",

    # Final 13 predictors
    "PM25",
    "tmax_c",
    "ppt_mm",

    "Below_poverty_percent",
    "Bachelor_plus_percent",

    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",

    "Pop_density",

    "hispanic_percent",
    "asian_percent",

    "Forest_percent"
]


# ==============================
# 4. Check columns
# ==============================

missing_columns = [
    col for col in final_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ==============================
# 5. Keep final columns
# ==============================

df_final = df[
    final_columns
].copy()


# ==============================
# 6. Final missing-value check
# ==============================

print("\n" + "=" * 70)
print("MISSING-VALUE CHECK")
print("=" * 70)

print(
    df_final.isna().sum()
)

print(
    "\nTotal missing values:",
    df_final.isna().sum().sum()
)


# ==============================
# 7. Drop missing values
#    Safety check only
# ==============================

# autism_selected_features.csv
# is already complete-case,
# so this should remove 0 rows.

n_before = len(df_final)

df_final = (
    df_final
    .dropna()
    .reset_index(drop=True)
)

n_after = len(df_final)

print("\nRows before dropna:", n_before)
print("Rows after dropna:", n_after)
print("Rows removed:", n_before - n_after)


# ==============================
# 8. Save final dataset
# ==============================

df_final.to_csv(
    output_path,
    index=False
)


# ==============================
# 9. Final summary
# ==============================

print("\n" + "=" * 70)
print("FINAL MODELING DATASET")
print("=" * 70)

print("\nColumns:")
print(df_final.columns.tolist())

print("\nNumber of final predictors:")
print(13)

print("\nFinal data shape:")
print(df_final.shape)

print("\nSaved to:")
print(output_path)

print("\nDeveloped_percent was removed.")
print("NO IMPUTATION WAS PERFORMED.")

## 8. Table S7: descriptive statistics

Generate descriptive statistics for the outcome and final predictor set.

In [ ]:
import pandas as pd

# ==============================
# 1. File paths
# ==============================

# Final modeling dataset:
# Autism prevalence + 13 final predictors
input_path = str(PROCESSED_DIR / 'autism_model_final.csv')

# Table S7 output
output_path = str(TABLES_DIR / 'Table_S7_descriptive_statistics.csv')


# ==============================
# 2. Load data
# ==============================

df = pd.read_csv(input_path)

# Clean column names
df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_path)

print("\nData dimensions:")
print(df.shape)


# ==============================
# 3. Define variable order
#    1 outcome + 13 final predictors
# ==============================

variable_order = [

    # ---- Outcome ----
    "Autism_prev1000",

    # ---- Air pollution ----
    "PM25",

    # ---- Climate ----
    "tmax_c",
    "ppt_mm",

    # ---- Socioeconomic ----
    "Below_poverty_percent",
    "Bachelor_plus_percent",

    # ---- Demographic ----
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",

    # ---- Urbanization ----
    "Pop_density",

    # ---- Race / ethnicity ----
    "hispanic_percent",
    "asian_percent",

    # ---- Land cover ----
    "Forest_percent"
]


# ==============================
# 4. Check variables
# ==============================

missing_variables = [
    col for col in variable_order
    if col not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Missing variables: {missing_variables}"
    )

print("\nAll required variables are present.")

print("\nNumber of variables summarized:")
print(len(variable_order))

print("(1 outcome + 13 final predictors)")


# ==============================
# 5. Convert to numeric
# ==============================

for col in variable_order:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ==============================
# 6. Missing-value check
# ==============================

print("\n" + "=" * 70)
print("MISSING-VALUE CHECK")
print("=" * 70)

missing_summary = df[variable_order].isna().sum()

print(missing_summary)

print(
    "\nTotal missing values:",
    missing_summary.sum()
)


# ==============================
# 7. Descriptive statistics
# ==============================

desc = (
    df[variable_order]
    .agg([
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .T
    .reset_index()
)

desc.columns = [
    "Variable",
    "Mean",
    "SD",
    "Min",
    "Median",
    "Max"
]


# ==============================
# 8. Round to 1 decimal place
# ==============================

numeric_columns = [
    "Mean",
    "SD",
    "Min",
    "Median",
    "Max"
]

desc[numeric_columns] = (
    desc[numeric_columns]
    .round(1)
)


# ==============================
# 9. Rename variables
#    for publication
# ==============================

rename_dict = {

    "Autism_prev1000":
        "Autism prevalence (per 1,000 students)",

    "PM25":
        "PM2.5 concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}

desc["Variable"] = (
    desc["Variable"]
    .map(rename_dict)
)


# ==============================
# 10. Save Table S7
# ==============================

desc.to_csv(
    output_path,
    index=False
)


# ==============================
# 11. Final output
# ==============================

print("\n" + "=" * 70)
print("TABLE S7: DESCRIPTIVE STATISTICS")
print("=" * 70)

print(
    desc.to_string(index=False)
)

print("\nSaved to:")
print(output_path)

## 9. Compare 12 machine-learning algorithms

Use the fixed train/test split and model-specific tuning procedures to identify the best held-out model.

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    KFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


warnings.filterwarnings("ignore")


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


# ============================================================
# 2. File paths
# ============================================================

# Final complete-case modeling dataset
# 8200 district-year observations
# ASD prevalence + 13 final predictors
# NO IMPUTATION
data_path = str(PROCESSED_DIR / 'autism_model_final.csv')

# Model-comparison results
output_results = str(TABLES_DIR / 'model_comparison_results_final.csv')


# ============================================================
# 3. Load data
# ============================================================

df = pd.read_csv(data_path)

df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(data_path)

print("\nData shape:")
print(df.shape)


# ============================================================
# 4. Define outcome and predictors
# ============================================================

y = df["Autism_prev1000"].copy()

X = df.drop(
    columns=[
        "Autism_prev1000",
        "GEOID",
        "Year"
    ]
).copy()


# ============================================================
# 5. Data checks
# ============================================================

print("\nPredictors used:")
print(X.columns.tolist())

print("\nNumber of predictors:")
print(X.shape[1])

print("\nData shape:", df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

print(
    "\nMissing values in X:",
    X.isna().sum().sum()
)

print(
    "Missing values in y:",
    y.isna().sum()
)

if X.shape[1] != 13:
    raise ValueError(
        f"Expected 13 predictors, but found {X.shape[1]}."
    )

if X.isna().sum().sum() > 0 or y.isna().sum() > 0:
    raise ValueError(
        "Missing values detected in final modeling dataset."
    )


# ============================================================
# 6. Train-test split
#    80% training / 20% held-out testing
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("\nTrain-test split:")
print("Training N:", len(X_train))
print("Testing N:", len(X_test))


# ============================================================
# 7. Five-fold CV setting
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 8. Helper functions
# ============================================================

def rmse_func(y_true, y_pred):

    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def evaluate_basic_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    model_name,
    use_cv=True
):

    """
    Untuned/basic models:

    1. Five-fold CV on training data
    2. Fit on full training set
    3. Evaluate on held-out test set
    """

    result = {
        "Model": model_name
    }

    if use_cv:

        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="r2",
            n_jobs=None
        )

        result["CV_R2_Mean"] = (
            cv_scores.mean()
        )

        result["CV_R2_SD"] = (
            cv_scores.std()
        )

    else:

        result["CV_R2_Mean"] = np.nan
        result["CV_R2_SD"] = np.nan

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    result["Test_R2"] = r2_score(
        y_test,
        y_pred
    )

    result["Test_RMSE"] = rmse_func(
        y_test,
        y_pred
    )

    result["Best_Params"] = ""

    return result


def evaluate_tuned_model(
    search_obj,
    X_train,
    X_test,
    y_train,
    y_test,
    model_name
):

    """
    Tuned models:

    1. RandomizedSearchCV with five-fold CV
       on training set
    2. Select best estimator
    3. Evaluate best estimator on held-out
       test set
    """

    search_obj.fit(
        X_train,
        y_train
    )

    best_model = (
        search_obj.best_estimator_
    )

    y_pred = best_model.predict(
        X_test
    )

    # Index of best parameter combination
    best_index = search_obj.best_index_

    # Mean and SD across the five CV folds
    cv_mean = (
        search_obj.cv_results_[
            "mean_test_score"
        ][best_index]
    )

    cv_sd = (
        search_obj.cv_results_[
            "std_test_score"
        ][best_index]
    )

    result = {

        "Model":
            model_name,

        "CV_R2_Mean":
            cv_mean,

        "CV_R2_SD":
            cv_sd,

        "Test_R2":
            r2_score(
                y_test,
                y_pred
            ),

        "Test_RMSE":
            rmse_func(
                y_test,
                y_pred
            ),

        "Best_Params":
            str(
                search_obj.best_params_
            )
    }

    return result


# ============================================================
# 9. Initialize results
# ============================================================

results = []


# ============================================================
# 10. Model 1: Linear Regression
# ============================================================

lr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LinearRegression()
    )
])

results.append(
    evaluate_basic_model(
        lr_model,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Linear Regression"
    )
)


# ============================================================
# 11. Model 2: KNN Regressor
# ============================================================

knn_pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KNeighborsRegressor()
    )
])

knn_param_dist = {

    "model__n_neighbors":
        [3, 5, 7, 9, 11, 15],

    "model__weights":
        ["uniform", "distance"],

    "model__p":
        [1, 2]
}

knn_search = RandomizedSearchCV(
    estimator=knn_pipe,
    param_distributions=knn_param_dist,
    n_iter=10,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        knn_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="KNN Regressor"
    )
)


# ============================================================
# 12. Model 3: Neural Network
# ============================================================

mlp_pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        MLPRegressor(
            max_iter=3000,
            random_state=RANDOM_STATE
        )
    )
])

mlp_param_dist = {

    "model__hidden_layer_sizes":
        [
            (50,),
            (100,),
            (50, 50),
            (100, 50)
        ],

    "model__activation":
        ["relu", "tanh"],

    "model__alpha":
        [0.0001, 0.001, 0.01],

    "model__learning_rate_init":
        [0.001, 0.01]
}

mlp_search = RandomizedSearchCV(
    estimator=mlp_pipe,
    param_distributions=mlp_param_dist,
    n_iter=12,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        mlp_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Neural Network"
    )
)


# ============================================================
# 13. Model 4: Decision Tree
# ============================================================

dt_model = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

results.append(
    evaluate_basic_model(
        dt_model,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Decision Tree"
    )
)


# ============================================================
# 14. Model 5: Random Forest
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_basic_model(
        rf_model,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Random Forest"
    )
)


# ============================================================
# 15. Model 6: Support Vector Regressor
# ============================================================

svr_pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVR()
    )
])

svr_param_dist = {

    "model__kernel":
        ["rbf", "linear"],

    "model__C":
        [0.1, 1, 10, 50, 100],

    "model__epsilon":
        [0.01, 0.05, 0.1, 0.2],

    "model__gamma":
        ["scale", "auto"]
}

svr_search = RandomizedSearchCV(
    estimator=svr_pipe,
    param_distributions=svr_param_dist,
    n_iter=20,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        svr_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Support Vector Regressor"
    )
)


# ============================================================
# 16. Model 7: Lasso Regression
# ============================================================

lasso_pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Lasso(
            random_state=RANDOM_STATE,
            max_iter=5000
        )
    )
])

lasso_param_dist = {

    "model__alpha":
        [
            0.0001,
            0.001,
            0.01,
            0.1,
            1
        ]
}

lasso_search = RandomizedSearchCV(
    estimator=lasso_pipe,
    param_distributions=lasso_param_dist,
    n_iter=5,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        lasso_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Lasso Regression"
    )
)


# ============================================================
# 17. Model 8: Ridge Regression
# ============================================================

ridge_pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge(
            random_state=RANDOM_STATE
        )
    )
])

ridge_param_dist = {

    "model__alpha":
        [
            0.01,
            0.1,
            1,
            10,
            100
        ]
}

ridge_search = RandomizedSearchCV(
    estimator=ridge_pipe,
    param_distributions=ridge_param_dist,
    n_iter=5,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        ridge_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Ridge Regression"
    )
)


# ============================================================
# 18. Model 9: AdaBoost
# ============================================================

ada_model = AdaBoostRegressor(
    random_state=RANDOM_STATE
)

ada_param_dist = {

    "n_estimators":
        [50, 100, 200, 300],

    "learning_rate":
        [0.01, 0.05, 0.1, 0.5, 1.0],

    "loss":
        [
            "linear",
            "square",
            "exponential"
        ]
}

ada_search = RandomizedSearchCV(
    estimator=ada_model,
    param_distributions=ada_param_dist,
    n_iter=12,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        ada_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="AdaBoost"
    )
)


# ============================================================
# 19. Model 10: Gradient Boosting
# ============================================================

gb_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

results.append(
    evaluate_basic_model(
        gb_model,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="Gradient Boosting"
    )
)


# ============================================================
# 20. Model 11: LightGBM
# ============================================================

lgbm_model = LGBMRegressor(
    random_state=RANDOM_STATE,
    verbose=-1
)

lgbm_param_dist = {

    "n_estimators":
        [100, 200, 300, 500],

    "learning_rate":
        [0.01, 0.05, 0.1],

    "max_depth":
        [-1, 3, 5, 7],

    "num_leaves":
        [15, 31, 50, 80],

    "subsample":
        [0.8, 1.0],

    "colsample_bytree":
        [0.8, 1.0]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=lgbm_param_dist,
    n_iter=20,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        lgbm_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="LightGBM"
    )
)


# ============================================================
# 21. Model 12: XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_param_dist = {

    "n_estimators":
        [100, 200, 300, 500, 700],

    "max_depth":
        [3, 4, 5, 6, 8],

    "learning_rate":
        [0.01, 0.05, 0.08, 0.1, 0.2],

    "subsample":
        [0.8, 1.0],

    "colsample_bytree":
        [0.6, 0.8, 1.0],

    "reg_alpha":
        [0, 0.001, 0.01, 0.1],

    "reg_lambda":
        [0.5, 1, 2],

    "gamma":
        [0, 0.1, 0.2]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_dist,
    n_iter=25,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

results.append(
    evaluate_tuned_model(
        xgb_search,
        X_train,
        X_test,
        y_train,
        y_test,
        model_name="XGBoost"
    )
)


# ============================================================
# 22. Organize results
# ============================================================

results_df = pd.DataFrame(
    results
)

numeric_cols = [
    "CV_R2_Mean",
    "CV_R2_SD",
    "Test_R2",
    "Test_RMSE"
]

for col in numeric_cols:

    results_df[col] = (
        results_df[col]
        .round(4)
    )

# Sort by held-out test R²
results_df = (
    results_df
    .sort_values(
        by="Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# 23. Save results
# ============================================================

results_df.to_csv(
    output_results,
    index=False
)


# ============================================================
# 24. Print results
# ============================================================

pd.set_option(
    "display.max_colwidth",
    None
)

print("\n" + "=" * 70)
print("MODEL COMPARISON RESULTS")
print("=" * 70)

print(
    results_df[
        [
            "Model",
            "CV_R2_Mean",
            "CV_R2_SD",
            "Test_R2",
            "Test_RMSE",
            "Best_Params"
        ]
    ].to_string(index=False)
)


# ============================================================
# 25. Best model
# ============================================================

best_model = results_df.iloc[0]

print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)

print(
    best_model[
        [
            "Model",
            "CV_R2_Mean",
            "CV_R2_SD",
            "Test_R2",
            "Test_RMSE",
            "Best_Params"
        ]
    ]
)


# ============================================================
# 26. XGBoost best parameters
# ============================================================

xgb_row = results_df[
    results_df["Model"] == "XGBoost"
]

if len(xgb_row) > 0:

    print("\n" + "=" * 70)
    print("XGBOOST BEST PARAMETERS")
    print("=" * 70)

    print(
        xgb_row[
            "Best_Params"
        ].values[0]
    )


# ============================================================
# 27. Final output
# ============================================================

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(output_results)

## 10. Model-comparison figure

Visualize held-out R² and RMSE for the 12 candidate algorithms.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ==============================
# 1. Input and output paths
# ==============================

# Model comparison results generated from the ML analysis
input_path = str(TABLES_DIR / 'model_comparison_results_final.csv')

# Figure 4
output_file = str(FIGURES_DIR / 'Figure4_ML_model_comparison.tiff')


# ==============================
# 2. Read model results
# ==============================

results = pd.read_csv(input_path)

print("Columns:")
print(results.columns.tolist())

print("\nModel results:")
print(results)


# ==============================
# 3. Extract Test R² and Test RMSE
# ==============================

model_results = results[
    ["Model", "Test_R2", "Test_RMSE"]
].copy()

model_results = model_results.rename(
    columns={
        "Test_R2": "R2",
        "Test_RMSE": "RMSE"
    }
)


# ==============================
# 4. Sort models by Test R²
# ==============================

plot_df = (
    model_results
    .sort_values(by="R2", ascending=False)
    .reset_index(drop=True)
)

y_pos = range(len(plot_df))


# ==============================
# 5. Automatically identify best model
# ==============================

highlight_model = plot_df.iloc[0]["Model"]
highlight_color = "#b22222"

print("\nBest-performing model:")
print(highlight_model)


# ==============================
# 6. Global style
# ==============================

plt.rcParams.update({
    "font.family": "Arial",
    "axes.unicode_minus": True,
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 11.5,
    "ytick.labelsize": 11.5,
    "legend.fontsize": 10.5,
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black"
})


# ==============================
# 7. Create figure
# ==============================

fig, ax_bottom = plt.subplots(figsize=(10.8, 6.4))
ax_top = ax_bottom.twiny()


# ==============================
# 8. Axis ranges
# ==============================

# Updated to accommodate negative R²
r2_min, r2_max = -0.10, 0.65
rmse_min, rmse_max = 4.2, 7.1

ax_top.set_xlim(r2_min, r2_max)
ax_bottom.set_xlim(rmse_min, rmse_max)


# ==============================
# 9. Plot settings
# ==============================

r2_x_offset = 0.012
rmse_y_offset = 0.35

marker_size_regular = 44
marker_size_best = 62

r2_label_size = 10.5
rmse_label_size = 10.5


# ==============================
# 10. Plot R² and RMSE
# ==============================

for i, row in plot_df.iterrows():

    is_best = row["Model"] == highlight_model

    r2_color = "#b22222" if is_best else "#1f77b4"
    rmse_color = "#b22222" if is_best else "#7f7f7f"

    size = marker_size_best if is_best else marker_size_regular

    # R²
    ax_top.scatter(
        row["R2"],
        i,
        s=size,
        color=r2_color,
        zorder=3
    )

    ax_top.text(
        row["R2"] + r2_x_offset,
        i,
        f"{row['R2']:.4f}",
        va="center",
        ha="left",
        fontsize=r2_label_size,
        color=r2_color,
        fontweight="bold" if is_best else "normal"
    )

    # RMSE
    ax_bottom.scatter(
        row["RMSE"],
        i,
        s=size,
        facecolors="white",
        edgecolors=rmse_color,
        linewidths=1.4,
        marker="s",
        zorder=3
    )

    ax_bottom.text(
        row["RMSE"],
        i + rmse_y_offset,
        f"{row['RMSE']:.4f}",
        va="center",
        ha="center",
        fontsize=rmse_label_size,
        color=rmse_color,
        fontweight="bold" if is_best else "normal"
    )


# ==============================
# 11. Y axis
# ==============================

ax_bottom.set_yticks(list(y_pos))
ax_bottom.set_yticklabels(plot_df["Model"])

ax_bottom.invert_yaxis()

ax_bottom.set_ylim(
    len(plot_df) - 0.45,
    -0.65
)


# ==============================
# 12. Axis labels
# ==============================

ax_top.set_xlabel("R²", color="black")
ax_bottom.set_xlabel("RMSE", color="black")


# ==============================
# 13. Grid
# ==============================

ax_bottom.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    color="#D0D0D0",
    alpha=0.8
)

ax_bottom.grid(axis="y", visible=False)


# ==============================
# 14. Bottom axis spines
# ==============================

ax_bottom.spines["left"].set_visible(True)
ax_bottom.spines["left"].set_color("black")
ax_bottom.spines["left"].set_linewidth(1.0)

ax_bottom.spines["bottom"].set_visible(True)
ax_bottom.spines["bottom"].set_color("black")
ax_bottom.spines["bottom"].set_linewidth(1.0)

ax_bottom.spines["top"].set_visible(False)
ax_bottom.spines["right"].set_visible(False)


# ==============================
# 15. Top axis spines
# ==============================

ax_top.spines["top"].set_visible(True)
ax_top.spines["top"].set_color("black")
ax_top.spines["top"].set_linewidth(1.0)

ax_top.spines["bottom"].set_visible(False)
ax_top.spines["left"].set_visible(False)
ax_top.spines["right"].set_visible(False)


# ==============================
# 16. Tick settings
# ==============================

ax_bottom.tick_params(
    axis="x",
    colors="black",
    width=0.9,
    length=4
)

ax_bottom.tick_params(
    axis="y",
    width=0.9,
    length=4
)

ax_top.tick_params(
    axis="x",
    colors="black",
    width=0.9,
    length=4
)


# ==============================
# 17. Highlight best model label
# ==============================

for tick_label in ax_bottom.get_yticklabels():

    if tick_label.get_text() == highlight_model:
        tick_label.set_color(highlight_color)
        tick_label.set_fontweight("bold")
    else:
        tick_label.set_color("black")
        tick_label.set_fontweight("normal")


# ==============================
# 18. Legend
# ==============================

legend_elements = [

    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor="#1f77b4",
        markeredgecolor="#1f77b4",
        markersize=6.5,
        label="R²"
    ),

    Line2D(
        [0], [0],
        marker="s",
        color="none",
        markerfacecolor="white",
        markeredgecolor="#7f7f7f",
        markersize=6.5,
        label="RMSE"
    )
]

ax_bottom.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)


# ==============================
# 19. Save
# ==============================

plt.subplots_adjust(right=0.82)

plt.savefig(
    output_file,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:")
print(output_file)

## 11. Final LightGBM model and global SHAP analysis

Fit the selected LightGBM model and generate global SHAP importance outputs and figures.

In [ ]:
import pandas as pd
import numpy as np
import os
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 0. Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


# ============================================================
# 1. File paths
# ============================================================

# Final complete-case modeling dataset
# 8200 district-year observations
# ASD prevalence + 13 final predictors
# NO IMPUTATION

file_path = str(PROCESSED_DIR / 'autism_model_final.csv')


# Existing output folders
tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)


# Output files
output_performance = os.path.join(
    tables_dir,
    "Table_LightGBM_model_performance_final.csv"
)

output_shap_summary = os.path.join(
    tables_dir,
    "Table_SHAP_global_importance_direction_final.csv"
)

output_shap_matrix = os.path.join(
    tables_dir,
    "Table_SHAP_values_matrix_final.csv"
)

output_combined = os.path.join(
    figures_dir,
    "Figure_SHAP_combined_final.tiff"
)

output_standard_summary = os.path.join(
    figures_dir,
    "Figure_SHAP_summary_standard_final.tiff"
)

output_standard_bar = os.path.join(
    figures_dir,
    "Figure_SHAP_bar_standard_final.tiff"
)


# ============================================================
# 2. Load final modeling dataset
# ============================================================

df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(file_path)

print("\nData shape:")
print(df.shape)


# ============================================================
# 3. Define outcome and final 13 predictors
# ============================================================

target_col = "Autism_prev1000"

feature_cols = [

    # Air pollution
    "PM25",

    # Climate
    "tmax_c",
    "ppt_mm",

    # Socioeconomic
    "Below_poverty_percent",
    "Bachelor_plus_percent",

    # Demographic
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",

    # Urbanization
    "Pop_density",

    # Race / ethnicity
    "hispanic_percent",
    "asian_percent",

    # Land cover
    "Forest_percent"
]


model_data = df[
    ["GEOID", "Year", target_col] + feature_cols
].copy()


# ============================================================
# 4. Data checks
# ============================================================

X = model_data[feature_cols].copy()

y = model_data[target_col].copy()


print("\nPredictors:")
print(X.columns.tolist())

print("\nNumber of predictors:")
print(X.shape[1])

print("\nX shape:")
print(X.shape)

print("\ny shape:")
print(y.shape)

print("\nMissing values in predictors:")
print(X.isna().sum())

print("\nMissing values in outcome:")
print(y.isna().sum())

print("\nInfinite values in predictors:")
print(np.isinf(X).sum())


if X.shape[1] != 13:

    raise ValueError(
        f"Expected 13 predictors, but found {X.shape[1]}."
    )


if X.isna().sum().sum() > 0 or y.isna().sum() > 0:

    raise ValueError(
        "Missing values detected in final modeling dataset."
    )


# ============================================================
# 5. Train-test split
#
# IMPORTANT:
# Same split as model-comparison analysis
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=RANDOM_STATE
)


print("\nTrain-test split:")

print("Training N:", len(X_train))

print("Testing N:", len(X_test))


# ============================================================
# 6. Final LightGBM model
#
# Best parameters obtained from model comparison:
#
# subsample = 0.8
# num_leaves = 50
# n_estimators = 500
# max_depth = -1
# learning_rate = 0.1
# colsample_bytree = 1.0
# ============================================================

final_model = lgb.LGBMRegressor(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=-1,

    num_leaves=50,

    subsample=0.8,

    colsample_bytree=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbose=-1
)


# ============================================================
# 7. Fit training data and evaluate on held-out test set
# ============================================================

final_model.fit(
    X_train,
    y_train
)


y_pred = final_model.predict(
    X_test
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)


r2 = r2_score(
    y_test,
    y_pred
)


print("\n" + "=" * 70)

print("FINAL LIGHTGBM MODEL EVALUATION")

print("=" * 70)

print(f"Test RMSE: {rmse:.4f}")

print(f"Test R²: {r2:.4f}")


# Save performance
model_perf = pd.DataFrame({

    "Metric": [
        "Test RMSE",
        "Test R2"
    ],

    "Value": [
        rmse,
        r2
    ]
})


model_perf.to_csv(
    output_performance,
    index=False
)


# ============================================================
# 8. Refit LightGBM on FULL dataset for SHAP analysis
# ============================================================

final_model.fit(
    X,
    y
)


# ============================================================
# 9. Calculate SHAP values
# ============================================================

explainer = shap.TreeExplainer(
    final_model
)


shap_values = explainer.shap_values(
    X
)


# Convert to DataFrame
shap_df = pd.DataFrame(

    shap_values,

    columns=feature_cols,

    index=X.index
)


# ============================================================
# 10. Export SHAP matrix
#
# GEOID + Year + SHAP values
# ============================================================

shap_matrix_export = pd.concat(

    [

        model_data[
            ["GEOID", "Year"]
        ].reset_index(drop=True),

        shap_df.reset_index(drop=True)

    ],

    axis=1
)


# ============================================================
# 11. Rename variables for publication
# ============================================================

feature_name_map = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}


X_renamed = X.rename(
    columns=feature_name_map
)


shap_df_renamed = shap_df.rename(
    columns=feature_name_map
)


display_features = list(
    X_renamed.columns
)


# ============================================================
# 12. Calculate SHAP global importance and direction
# ============================================================

results = []


for raw_feature, display_feature in zip(
    feature_cols,
    display_features
):

    mean_abs_shap = np.abs(
        shap_df[raw_feature]
    ).mean()


    mean_shap = shap_df[
        raw_feature
    ].mean()


    corr = np.corrcoef(

        X[raw_feature],

        shap_df[raw_feature]

    )[0, 1]


    if pd.isna(corr):

        direction = "Unclear"

    elif np.isclose(
        corr,
        0,
        atol=1e-6
    ):

        direction = "Neutral"

    elif corr > 0:

        direction = "Positive"

    else:

        direction = "Negative"


    results.append({

        "Feature":
            display_feature,

        "MeanAbsSHAP":
            mean_abs_shap,

        "MeanSHAP":
            mean_shap,

        "Feature_SHAP_Correlation":
            corr,

        "Overall_Direction":
            direction
    })


shap_summary = pd.DataFrame(
    results
).sort_values(

    by="MeanAbsSHAP",

    ascending=False

).reset_index(drop=True)


# Round numeric results
shap_summary["MeanAbsSHAP"] = (
    shap_summary["MeanAbsSHAP"].round(4)
)

shap_summary["MeanSHAP"] = (
    shap_summary["MeanSHAP"].round(4)
)

shap_summary[
    "Feature_SHAP_Correlation"
] = (
    shap_summary[
        "Feature_SHAP_Correlation"
    ].round(4)
)


print("\n" + "=" * 70)

print("SHAP GLOBAL IMPORTANCE + DIRECTION")

print("=" * 70)

print(shap_summary)


# ============================================================
# 13. Save SHAP tables
# ============================================================

shap_summary.to_csv(
    output_shap_summary,
    index=False
)


shap_matrix_export.to_csv(
    output_shap_matrix,
    index=False
)


# ============================================================
# 14. Combined SHAP figure
# ============================================================

feature_order = (
    shap_summary["Feature"].tolist()
)


n_feat = len(feature_order)


y_positions = np.arange(
    n_feat
)[::-1]


plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        11.5
})


fig, (ax1, ax2) = plt.subplots(

    1,
    2,

    figsize=(16, 6.0),

    gridspec_kw={
        "width_ratios": [2.5, 1.5],
        "wspace": 0.10
    }
)


# ============================================================
# 14a. SHAP summary scatter
# ============================================================

for i, feature in enumerate(
    feature_order
):

    y0 = y_positions[i]


    shap_vals = (
        shap_df_renamed[
            feature
        ].values
    )


    feat_vals = (
        X_renamed[
            feature
        ].values
    )


    valid = ~(
        np.isnan(shap_vals)
        |
        np.isnan(feat_vals)
    )


    shap_vals = shap_vals[
        valid
    ]

    feat_vals = feat_vals[
        valid
    ]


    # Standardize feature values to 0–1
    # for consistent color mapping

    vmin = np.nanpercentile(
        feat_vals,
        5
    )

    vmax = np.nanpercentile(
        feat_vals,
        95
    )


    feat_vals_clip = np.clip(
        feat_vals,
        vmin,
        vmax
    )


    if vmax > vmin:

        feat_color = (
            feat_vals_clip - vmin
        ) / (
            vmax - vmin
        )

    else:

        feat_color = np.full_like(
            feat_vals_clip,
            0.5,
            dtype=float
        )


    # Fixed jitter for reproducibility
    rng = np.random.default_rng(
        RANDOM_STATE + i
    )


    jitter = rng.normal(
        0,
        0.08,
        size=len(shap_vals)
    )


    sc = ax1.scatter(

        shap_vals,

        np.full_like(
            shap_vals,
            y0,
            dtype=float
        ) + jitter,

        c=feat_color,

        cmap="coolwarm",

        vmin=0,

        vmax=1,

        s=12,

        alpha=0.75,

        edgecolors="none"
    )


ax1.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)


ax1.set_yticks(
    y_positions
)


ax1.set_yticklabels(
    feature_order,
    fontsize=11
)


ax1.set_xlabel(
    "SHAP value (effect on predicted autism prevalence)",
    fontsize=13
)


ax1.set_ylabel("")


ax1.set_title(
    "(a) SHAP summary",
    fontsize=14
)


ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)


ax1.tick_params(
    axis="x",
    labelsize=11
)


# Colorbar
cbar = plt.colorbar(
    sc,
    ax=ax1,
    pad=0.01
)


cbar.set_label(
    "Feature value",
    fontsize=11
)


ticks = np.linspace(
    0,
    1,
    6
)


cbar.set_ticks(
    ticks
)


cbar.set_ticklabels(
    [
        f"{t:.1f}"
        for t in ticks
    ]
)


cbar.ax.tick_params(
    labelsize=10
)


ax1.spines[
    "top"
].set_visible(False)


ax1.spines[
    "right"
].set_visible(False)


# ============================================================
# 14b. SHAP feature importance
# ============================================================

bar_data = (
    shap_summary
    .set_index("Feature")
    .loc[feature_order]
)


bar_values = (
    bar_data[
        "MeanAbsSHAP"
    ].values
)


bar_colors = plt.cm.viridis(
    np.linspace(
        0.35,
        0.92,
        n_feat
    )
)


ax2.barh(

    y_positions,

    bar_values,

    height=0.52,

    color=bar_colors
)


xmax = bar_values.max()

right_margin = xmax * 0.20

text_offset = xmax * 0.02


ax2.set_xlim(
    0,
    xmax + right_margin
)


for y_pos, v in zip(
    y_positions,
    bar_values
):

    ax2.text(

        v + text_offset,

        y_pos,

        f"{v:.2f}",

        va="center",

        ha="left",

        fontsize=10.5,

        clip_on=True
    )


ax2.set_yticks(
    y_positions
)


ax2.set_yticklabels([])


ax2.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=13
)


ax2.set_ylabel("")


ax2.set_title(
    "(b) SHAP feature importance",
    fontsize=14
)


ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)


ax2.tick_params(
    axis="x",
    labelsize=11
)


ax2.spines[
    "top"
].set_visible(False)


ax2.spines[
    "right"
].set_visible(False)


# Same vertical limits
ax1.set_ylim(
    -0.45,
    n_feat - 0.55
)


ax2.set_ylim(
    -0.45,
    n_feat - 0.55
)


plt.tight_layout()


plt.savefig(

    output_combined,

    dpi=300,

    format="tiff",

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 15. Standard SHAP summary plot
# ============================================================

plt.figure()


shap.summary_plot(

    shap_values,

    X_renamed,

    show=False
)


plt.tight_layout()


plt.savefig(

    output_standard_summary,

    dpi=300,

    format="tiff",

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 16. Standard SHAP importance bar plot
# ============================================================

plt.figure()


shap.summary_plot(

    shap_values,

    X_renamed,

    plot_type="bar",

    show=False
)


plt.tight_layout()


plt.savefig(

    output_standard_bar,

    dpi=300,

    format="tiff",

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 17. Final output summary
# ============================================================

print("\n" + "=" * 70)

print("FILES SAVED")

print("=" * 70)


print("\nTables:")

print(output_performance)

print(output_shap_summary)

print(output_shap_matrix)


print("\nFigures:")

print(output_combined)

print(output_standard_summary)

print(output_standard_bar)

## 12. SHAP dependence analysis

Generate predictor-level SHAP dependence outputs for the final LightGBM model.

In [ ]:
import pandas as pd
import numpy as np
import os
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# 1. File paths
# ============================================================

# Final complete-case modeling dataset
# 8200 observations
# Autism prevalence + final 13 predictors
# NO IMPUTATION
file_path = str(PROCESSED_DIR / 'autism_model_final.csv')

# Existing output folders
tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)


# ============================================================
# 2. Load data
# ============================================================

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

target_col = "Autism_prev1000"


# ============================================================
# 3. Define predictors
#    Ordered by final LightGBM SHAP importance
# ============================================================

feature_cols = [
    "PM25",
    "Pop_density",
    "hispanic_percent",
    "Bachelor_plus_percent",
    "Forest_percent",
    "tmax_c",
    "age15to19_percent",
    "Below_poverty_percent",
    "median_age",
    "age5to14_percent",
    "under5_percent",
    "ppt_mm",
    "asian_percent"
]


# ============================================================
# 4. Publication-friendly names
# ============================================================

feature_name_map = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m$^3$)",

    "Pop_density":
        r"Population density (persons/km$^2$)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "Forest_percent":
        "Forest land cover (%)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "median_age":
        "Median age (years)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "asian_percent":
        "Asian population (%)"
}


# Shorter panel titles
panel_title_map = {

    "PM25":
        r"PM$_{2.5}$ concentration",

    "Pop_density":
        "Population density",

    "hispanic_percent":
        "Hispanic or Latino population",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher",

    "Forest_percent":
        "Forest land cover",

    "tmax_c":
        "Mean daily maximum temperature",

    "age15to19_percent":
        "Population aged 15–19 years",

    "Below_poverty_percent":
        "Families below the poverty line",

    "median_age":
        "Median age",

    "age5to14_percent":
        "Population aged 5–14 years",

    "under5_percent":
        "Population under 5 years",

    "ppt_mm":
        "Annual precipitation",

    "asian_percent":
        "Asian population"
}


# ============================================================
# 5. Prepare modeling data
# ============================================================

model_data = df[
    ["GEOID", "Year", target_col] + feature_cols
].copy()

# Safety check
model_data = model_data.dropna().copy()

X = model_data[feature_cols].copy()
y = model_data[target_col].copy()


print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print("Data shape:", model_data.shape)
print("Number of predictors:", len(feature_cols))

print("\nMissing values:")
print(model_data.isna().sum())

print("\nPredictor order:")
for i, f in enumerate(feature_cols, start=1):
    print(i, f)


# ============================================================
# 6. Fit final LightGBM model
#
# Best parameters from the final model comparison
# ============================================================

final_model = lgb.LGBMRegressor(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=-1,

    num_leaves=50,

    subsample=0.8,

    colsample_bytree=1.0,

    random_state=42,

    n_jobs=-1,

    verbose=-1
)

# SHAP interpretation uses model refitted on full dataset
final_model.fit(X, y)


# ============================================================
# 7. Calculate SHAP values
# ============================================================

explainer = shap.TreeExplainer(final_model)

shap_values = explainer.shap_values(X)

shap_df = pd.DataFrame(
    shap_values,
    columns=feature_cols,
    index=X.index
)


# ============================================================
# 8. Calculate current SHAP importance
#    and verify ranking
# ============================================================

importance_df = pd.DataFrame({

    "Feature": feature_cols,

    "MeanAbsSHAP": [
        np.abs(shap_df[f]).mean()
        for f in feature_cols
    ]
})

importance_df = (
    importance_df
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("LIGHTGBM SHAP IMPORTANCE")
print("=" * 70)

print(importance_df.to_string(index=False))


# ============================================================
# 9. Define top 6 automatically
# ============================================================

top6_features = (
    importance_df
    .head(6)["Feature"]
    .tolist()
)

print("\nTop 6 predictors:")
print(top6_features)


# ============================================================
# 10. Export descriptive SHAP dependence data
# ============================================================

summary_rows = []
quartile_rows = []

# Use final SHAP ranking
ranked_features = importance_df["Feature"].tolist()

for rank, feature in enumerate(
    ranked_features,
    start=1
):

    display_name = feature_name_map[feature]

    x = X[feature]
    s = shap_df[feature]

    valid = ~(x.isna() | s.isna())

    x = x[valid]
    s = s[valid]

    corr = np.corrcoef(
        x,
        s
    )[0, 1]

    q25, q50, q75 = np.percentile(
        x,
        [25, 50, 75]
    )

    low_mean = s[
        x <= q25
    ].mean()

    mid_mean = s[
        (x > q25) &
        (x <= q75)
    ].mean()

    high_mean = s[
        x > q75
    ].mean()

    summary_rows.append({

        "Rank":
            rank,

        "Raw_variable":
            feature,

        "Predictor":
            display_name,

        "Feature_min":
            x.min(),

        "Feature_Q25":
            q25,

        "Feature_median":
            q50,

        "Feature_Q75":
            q75,

        "Feature_max":
            x.max(),

        "SHAP_min":
            s.min(),

        "SHAP_mean":
            s.mean(),

        "SHAP_max":
            s.max(),

        "Feature_SHAP_correlation":
            corr,

        "Mean_SHAP_lowest_quartile":
            low_mean,

        "Mean_SHAP_middle_50_percent":
            mid_mean,

        "Mean_SHAP_highest_quartile":
            high_mean,

        "SHAP_change_highest_vs_lowest_quartile":
            high_mean - low_mean
    })

    q_bins = pd.qcut(
        x,
        q=4,
        duplicates="drop"
    )

    temp = pd.DataFrame({

        "Rank":
            rank,

        "Raw_variable":
            feature,

        "Predictor":
            display_name,

        "Feature_value":
            x,

        "SHAP_value":
            s,

        "Quartile":
            q_bins.astype(str)
    })

    q_summary = temp.groupby(
        [
            "Rank",
            "Raw_variable",
            "Predictor",
            "Quartile"
        ],
        observed=True
    ).agg(

        Feature_min=(
            "Feature_value",
            "min"
        ),

        Feature_max=(
            "Feature_value",
            "max"
        ),

        Feature_mean=(
            "Feature_value",
            "mean"
        ),

        Mean_SHAP=(
            "SHAP_value",
            "mean"
        ),

        Median_SHAP=(
            "SHAP_value",
            "median"
        ),

        N=(
            "SHAP_value",
            "size"
        )

    ).reset_index()

    quartile_rows.append(
        q_summary
    )


summary_table = pd.DataFrame(
    summary_rows
)

quartile_table = pd.concat(
    quartile_rows,
    ignore_index=True
)


# ============================================================
# 11. Save SHAP dependence tables
# ============================================================

output_summary = os.path.join(
    tables_dir,
    "Table_SHAP_dependence_summary_all13.csv"
)

output_quartiles = os.path.join(
    tables_dir,
    "Table_SHAP_dependence_quartiles_all13.csv"
)

output_point_level = os.path.join(
    tables_dir,
    "Table_SHAP_dependence_point_level_all13.csv"
)


summary_table.to_csv(
    output_summary,
    index=False
)

quartile_table.to_csv(
    output_quartiles,
    index=False
)


# Point-level SHAP data
point_level = model_data[
    ["GEOID", "Year", target_col]
].copy()

for feature in ranked_features:

    point_level[
        feature + "_value"
    ] = X[feature].values

    point_level[
        feature + "_SHAP"
    ] = shap_df[feature].values


point_level.to_csv(
    output_point_level,
    index=False
)


# ============================================================
# 12. Plot style
# ============================================================

plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        13
})


panel_labels_13 = [
    "(a)", "(b)", "(c)", "(d)", "(e)",
    "(f)", "(g)", "(h)", "(i)", "(j)",
    "(k)", "(l)", "(m)"
]


# ============================================================
# 13. Individual dependence plots
#     for all 13 predictors
# ============================================================

for rank, (label, feature) in enumerate(
    zip(
        panel_labels_13,
        ranked_features
    ),
    start=1
):

    fig, ax = plt.subplots(
        figsize=(6.8, 5.0)
    )

    shap.dependence_plot(

        feature,

        shap_values,

        X,

        interaction_index=None,

        show=False,

        ax=ax,

        alpha=0.60,

        dot_size=10
    )

    ax.set_xlabel(
        feature_name_map[feature],
        fontsize=14
    )

    ax.set_ylabel(
        "SHAP value",
        fontsize=14
    )

    ax.set_title(

        f"{label} {panel_title_map[feature]}",

        fontsize=16,

        fontweight="normal",

        loc="center",

        pad=8
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=0.9,
        alpha=0.8
    )

    ax.grid(
        alpha=0.25,
        linestyle="--",
        linewidth=0.6
    )

    ax.tick_params(
        axis="both",
        labelsize=13
    )

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)

    plt.tight_layout()

    individual_file = os.path.join(
        figures_dir,
        f"SHAP_dependence_{rank:02d}_{feature}.tiff"
    )

    plt.savefig(
        individual_file,
        dpi=300,
        format="tiff",
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 14. Main manuscript figure
#     Top 6 LightGBM SHAP predictors
# ============================================================

panel_labels_6 = [
    "(a)",
    "(b)",
    "(c)",
    "(d)",
    "(e)",
    "(f)"
]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 8.6)
)

axes = axes.flatten()


for ax, label, feature in zip(
    axes,
    panel_labels_6,
    top6_features
):

    plt.sca(ax)

    shap.dependence_plot(

        feature,

        shap_values,

        X,

        interaction_index=None,

        show=False,

        ax=ax,

        alpha=0.60,

        dot_size=10
    )

    ax.set_xlabel(
        feature_name_map[feature],
        fontsize=14
    )

    ax.set_ylabel(
        "SHAP value",
        fontsize=14
    )

    ax.set_title(

        f"{label} {panel_title_map[feature]}",

        fontsize=16,

        fontweight="normal",

        loc="center",

        pad=8
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=0.9,
        alpha=0.8
    )

    ax.grid(
        alpha=0.25,
        linestyle="--",
        linewidth=0.6
    )

    ax.tick_params(
        axis="both",
        labelsize=13
    )

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)


plt.tight_layout(
    w_pad=2.0,
    h_pad=2.2
)


main_fig_file = os.path.join(
    figures_dir,
    "Figure_SHAP_dependence_top6_main.tiff"
)


plt.savefig(
    main_fig_file,
    dpi=600,
    format="tiff",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 15. Supplementary figure
#     All 13 predictors
# ============================================================

fig, axes = plt.subplots(
    5,
    3,
    figsize=(18.5, 22.0)
)

axes = axes.flatten()


for ax, label, feature in zip(
    axes,
    panel_labels_13,
    ranked_features
):

    plt.sca(ax)

    shap.dependence_plot(

        feature,

        shap_values,

        X,

        interaction_index=None,

        show=False,

        ax=ax,

        alpha=0.60,

        dot_size=9
    )

    ax.set_xlabel(
        feature_name_map[feature],
        fontsize=16
    )

    ax.set_ylabel(
        "SHAP value",
        fontsize=16
    )

    ax.set_title(

        f"{label} {panel_title_map[feature]}",

        fontsize=19,

        fontweight="normal",

        loc="center",

        pad=8
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=0.8,
        alpha=0.8
    )

    ax.grid(
        alpha=0.25,
        linestyle="--",
        linewidth=0.5
    )

    ax.tick_params(
        axis="both",
        labelsize=16
    )

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)


# Hide unused two panels
for ax in axes[
    len(ranked_features):
]:

    ax.axis("off")


plt.tight_layout(
    w_pad=1.8,
    h_pad=2.0
)


supp_fig_file = os.path.join(
    figures_dir,
    "Figure_S_SHAP_dependence_all13.tiff"
)


plt.savefig(
    supp_fig_file,
    dpi=600,
    format="tiff",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 16. Final output
# ============================================================

print("\n" + "=" * 70)
print("SAVED OUTPUTS")
print("=" * 70)

print("\nTop 6 predictors used in main figure:")

for i, feature in enumerate(
    top6_features,
    start=1
):

    print(
        f"{i}. {feature_name_map[feature]}"
    )


print("\nTables:")

print(output_summary)
print(output_quartiles)
print(output_point_level)


print("\nFigures:")

print(main_fig_file)
print(supp_fig_file)

print(
    "\nIndividual dependence plots saved in:"
)

print(figures_dir)

## 13. Prepare local SHAP outputs

Reshape point-level SHAP values for mapping, dominant-predictor summaries, and downstream spatial analyses.

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. File paths
# ============================================================

# LightGBM point-level SHAP results
input_path = str(TABLES_DIR / 'Table_SHAP_dependence_point_level_all13.csv')

# Existing results/tables folder
output_dir = str(TABLES_DIR)

# Step 1 outputs
output_main = os.path.join(
    output_dir,
    "Table_local_SHAP_for_mapping.csv"
)

output_shap_matrix = os.path.join(
    output_dir,
    "Table_local_SHAP_matrix_raw.csv"
)

output_friendly = os.path.join(
    output_dir,
    "Table_local_SHAP_for_mapping_friendly.csv"
)

output_selected = os.path.join(
    output_dir,
    "Table_local_SHAP_for_mapping_selected_years.csv"
)

# Step 2 outputs
output_plot_all = os.path.join(
    output_dir,
    "Table_local_SHAP_for_plotting_only.csv"
)

output_plot_selected = os.path.join(
    output_dir,
    "Table_local_SHAP_for_plotting_only_2010_2014_2018_2022.csv"
)

# Step 3 output
output_wide = os.path.join(
    output_dir,
    "Table_local_SHAP_for_plotting_only_2010_2014_2018_2022_wide.csv"
)


# ============================================================
# 2. Load LightGBM point-level SHAP data
# ============================================================

# Read GEOID as string from the beginning
# to avoid potential GIS join problems
df = pd.read_csv(
    input_path,
    dtype={"GEOID": str}
)

df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_path)

print("\nData dimensions:")
print(df.shape)

print("\nFirst columns:")
print(df.columns.tolist()[:20])


# ============================================================
# 3. Define final 13 predictors
# ============================================================

feature_cols = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]


# ============================================================
# 4. Friendly variable names
# ============================================================

feature_name_map = {

    "PM25":
        "PM2.5 concentration",

    "tmax_c":
        "Mean daily maximum temperature",

    "ppt_mm":
        "Annual precipitation",

    "Below_poverty_percent":
        "Families below the poverty line",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher",

    "under5_percent":
        "Population under 5 years",

    "age5to14_percent":
        "Population aged 5–14 years",

    "age15to19_percent":
        "Population aged 15–19 years",

    "median_age":
        "Median age",

    "Pop_density":
        "Population density",

    "hispanic_percent":
        "Hispanic or Latino population",

    "asian_percent":
        "Asian population",

    "Forest_percent":
        "Forest land cover"
}


# ============================================================
# 5. Identify value and SHAP columns
# ============================================================

value_cols = [
    f"{feature}_value"
    for feature in feature_cols
]

shap_cols = [
    f"{feature}_SHAP"
    for feature in feature_cols
]


# ============================================================
# 6. Check required columns
# ============================================================

required_cols = (
    ["GEOID", "Year", "Autism_prev1000"]
    + value_cols
    + shap_cols
)

missing_cols = [
    c for c in required_cols
    if c not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print("\nAll required local SHAP columns are present.")


# ============================================================
# 7. Check duplicate GEOID-Year combinations
# ============================================================

n_duplicates = df.duplicated(
    subset=["GEOID", "Year"]
).sum()

print("\nDuplicate GEOID-Year records:")
print(n_duplicates)

if n_duplicates > 0:
    raise ValueError(
        "Duplicate GEOID-Year records found. "
        "Please check data before creating GIS files."
    )


# ============================================================
# 8. Build clean SHAP matrix
# ============================================================

# Rename:
# PM25_SHAP -> PM25
# Pop_density_SHAP -> Pop_density
# etc.

shap_df = df[
    shap_cols
].copy()

shap_df.columns = feature_cols

# Convert to numeric
for col in feature_cols:
    shap_df[col] = pd.to_numeric(
        shap_df[col],
        errors="coerce"
    )

if shap_df.isna().sum().sum() > 0:
    raise ValueError(
        "Missing SHAP values detected."
    )


# ============================================================
# STEP 1
# Generate local SHAP mapping information
# ============================================================


# ============================================================
# 9. Dominant predictor by absolute SHAP
# ============================================================

shap_abs = shap_df.abs()

top_abs_feature_raw = shap_abs.idxmax(
    axis=1
)

row_index = np.arange(
    len(shap_df)
)

column_index = shap_df.columns.get_indexer(
    top_abs_feature_raw
)

top_abs_value = shap_df.to_numpy()[
    row_index,
    column_index
]

top_abs_feature = (
    top_abs_feature_raw
    .map(feature_name_map)
)

top_abs_direction = np.where(
    top_abs_value > 0,
    "Positive",
    np.where(
        top_abs_value < 0,
        "Negative",
        "Neutral"
    )
)


# ============================================================
# 10. Strongest positive predictor
# ============================================================

pos_df = shap_df.where(
    shap_df > 0
)

top_pos_feature_raw = pos_df.idxmax(
    axis=1
)

top_pos_value = pos_df.max(
    axis=1
)

top_pos_feature = (
    top_pos_feature_raw
    .map(feature_name_map)
)


# ============================================================
# 11. Strongest negative predictor
# ============================================================

neg_df = shap_df.where(
    shap_df < 0
)

top_neg_feature_raw = neg_df.idxmin(
    axis=1
)

top_neg_value = neg_df.min(
    axis=1
)

top_neg_feature = (
    top_neg_feature_raw
    .map(feature_name_map)
)


# ============================================================
# 12. Build main local-SHAP mapping table
# ============================================================

local_map_df = pd.DataFrame({

    "GEOID":
        df["GEOID"],

    "Year":
        df["Year"],

    "Autism_prev1000":
        df["Autism_prev1000"],

    # --------------------------
    # Dominant absolute feature
    # --------------------------

    "Top_feature_abs_raw":
        top_abs_feature_raw,

    "Top_feature_abs":
        top_abs_feature,

    "Top_SHAP_abs_signed":
        top_abs_value,

    "Top_SHAP_abs_magnitude":
        np.abs(top_abs_value),

    "Top_SHAP_abs_direction":
        top_abs_direction,

    # --------------------------
    # Strongest positive feature
    # --------------------------

    "Top_feature_positive_raw":
        top_pos_feature_raw,

    "Top_feature_positive":
        top_pos_feature,

    "Top_SHAP_positive":
        top_pos_value,

    # --------------------------
    # Strongest negative feature
    # --------------------------

    "Top_feature_negative_raw":
        top_neg_feature_raw,

    "Top_feature_negative":
        top_neg_feature,

    "Top_SHAP_negative":
        top_neg_value
})


# ============================================================
# 13. Add raw predictor values
# ============================================================

for feature in feature_cols:

    local_map_df[
        feature
    ] = df[
        f"{feature}_value"
    ].values


# ============================================================
# 14. Add all local SHAP values
# ============================================================

for feature in feature_cols:

    local_map_df[
        f"SHAP_{feature}"
    ] = df[
        f"{feature}_SHAP"
    ].values


# ============================================================
# 15. Round numeric values
# ============================================================

round_cols = [

    "Autism_prev1000",

    "Top_SHAP_abs_signed",
    "Top_SHAP_abs_magnitude",

    "Top_SHAP_positive",
    "Top_SHAP_negative"
]

for col in round_cols:

    local_map_df[col] = (
        local_map_df[col]
        .round(4)
    )


# ============================================================
# 16. Save main mapping table
# ============================================================

local_map_df.to_csv(
    output_main,
    index=False
)


# ============================================================
# 17. Save raw SHAP matrix
# ============================================================

shap_export = pd.DataFrame({

    "GEOID":
        df["GEOID"],

    "Year":
        df["Year"]
})

for feature in feature_cols:

    shap_export[
        f"SHAP_{feature}"
    ] = df[
        f"{feature}_SHAP"
    ].values


shap_export.to_csv(
    output_shap_matrix,
    index=False
)


# ============================================================
# 18. Save friendly SHAP matrix
# ============================================================

friendly_export = pd.DataFrame({

    "GEOID":
        df["GEOID"],

    "Year":
        df["Year"]
})

for feature in feature_cols:

    friendly_name = (
        feature_name_map[feature]
    )

    friendly_export[
        f"SHAP_{friendly_name}"
    ] = df[
        f"{feature}_SHAP"
    ].values


friendly_export.to_csv(
    output_friendly,
    index=False
)


# ============================================================
# 19. Save selected years from full local table
# ============================================================

selected_years = [
    2010,
    2014,
    2018,
    2022
]

local_map_selected = local_map_df[
    local_map_df["Year"].isin(
        selected_years
    )
].copy()

local_map_selected.to_csv(
    output_selected,
    index=False
)


# ============================================================
# STEP 2
# Create plotting-only GIS files
# ============================================================


# ============================================================
# 20. Keep mapping fields only
# ============================================================

plot_cols = [

    "GEOID",
    "Year",

    # outcome reference
    "Autism_prev1000",

    # dominant feature
    "Top_feature_abs",
    "Top_SHAP_abs_signed",
    "Top_SHAP_abs_magnitude",
    "Top_SHAP_abs_direction",

    # positive contribution
    "Top_feature_positive",
    "Top_SHAP_positive",

    # negative contribution
    "Top_feature_negative",
    "Top_SHAP_negative"
]


plot_df = local_map_df[
    plot_cols
].copy()


# ============================================================
# 21. Save full plotting-ready dataset
# ============================================================

plot_df.to_csv(
    output_plot_all,
    index=False
)


# ============================================================
# 22. Save 2010 / 2014 / 2018 / 2022 only
# ============================================================

plot_df_selected = plot_df[
    plot_df["Year"].isin(
        selected_years
    )
].copy()

plot_df_selected = (
    plot_df_selected
    .sort_values(
        ["GEOID", "Year"]
    )
    .reset_index(drop=True)
)

plot_df_selected.to_csv(
    output_plot_selected,
    index=False
)


# ============================================================
# STEP 3
# Convert selected-year GIS data to WIDE format
# ============================================================


# ============================================================
# 23. Keep categorical dominant-feature fields
# ============================================================

wide_source = plot_df_selected[
    [
        "GEOID",
        "Year",
        "Top_feature_abs",
        "Top_feature_positive",
        "Top_feature_negative"
    ]
].copy()


# ============================================================
# 24. Pivot long -> wide
# ============================================================

wide_df = wide_source.pivot(
    index="GEOID",
    columns="Year",
    values=[
        "Top_feature_abs",
        "Top_feature_positive",
        "Top_feature_negative"
    ]
)


# Flatten MultiIndex columns
wide_df.columns = [

    f"{variable}_{year}"

    for variable, year
    in wide_df.columns
]

wide_df = (
    wide_df
    .reset_index()
)


# ============================================================
# 25. Reorder wide columns by year
# ============================================================

desired_cols = [
    "GEOID"
]

for year in selected_years:

    desired_cols.extend([

        f"Top_feature_abs_{year}",

        f"Top_feature_positive_{year}",

        f"Top_feature_negative_{year}"
    ])


desired_cols = [

    col
    for col in desired_cols
    if col in wide_df.columns
]

wide_df = wide_df[
    desired_cols
]


# ============================================================
# 26. Save final wide GIS file
# ============================================================

wide_df.to_csv(
    output_wide,
    index=False
)


# ============================================================
# 27. QC summaries
# ============================================================

print("\n" + "=" * 70)
print("LOCAL SHAP QC")
print("=" * 70)

print("\nTotal district-year records:")
print(len(local_map_df))

print("\nSelected-year records:")
print(len(plot_df_selected))

print("\nNumber of unique GEOIDs in wide table:")
print(wide_df["GEOID"].nunique())


print("\nDominant predictors by frequency:")

print(
    local_map_df[
        "Top_feature_abs"
    ]
    .value_counts()
)


print("\nDominant predictors in selected years:")

print(
    plot_df_selected
    .groupby(
        ["Year", "Top_feature_abs"]
    )
    .size()
    .reset_index(name="N")
    .to_string(index=False)
)


# ============================================================
# 28. Final saved-file summary
# ============================================================

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("\nSTEP 1:")
print(output_main)
print(output_shap_matrix)
print(output_friendly)
print(output_selected)

print("\nSTEP 2:")
print(output_plot_all)
print(output_plot_selected)

print("\nSTEP 3:")
print(output_wide)

print("\nDone.")
print("Existing LightGBM local SHAP values were reused.")
print("NO LightGBM model was retrained.")
print("NO SHAP values were recalculated.")

## 14. Add ASD prevalence to the mapping table

Merge prevalence values into the selected-year wide-format SHAP table while preserving GEOID as text.

In [ ]:
import pandas as pd
import os

# ============================================================
# 1. File paths
# ============================================================

base_dir = str(TABLES_DIR)

# Wide-format file with GEOID stored in a GIS-compatible format
wide_path = os.path.join(
    base_dir,
    "Table_local_SHAP_for_plotting_only_2010_2014_2018_2022_wide.csv"
)

# Long-format file containing autism prevalence
prev_path = os.path.join(
    base_dir,
    "Table_local_SHAP_for_plotting_only_2010_2014_2018_2022.csv"
)

# Output: retain the wide table and append prevalence columns
output_path = os.path.join(
    base_dir,
    "Table_local_SHAP_for_plotting_only_2010_2014_2018_2022_wide_with_prevalence.csv"
)


# ============================================================
# 2. Read data
# ============================================================

# Keep GEOID as text to preserve leading zeros and support GIS joins
wide_df = pd.read_csv(
    wide_path,
    dtype={"GEOID": str}
)

prev_df = pd.read_csv(
    prev_path,
    dtype={"GEOID": str}
)

wide_df.columns = wide_df.columns.str.strip()
prev_df.columns = prev_df.columns.str.strip()


# ============================================================
# 3. Basic checks
# ============================================================

print("=" * 70)
print("INPUT CHECK")
print("=" * 70)

print("Wide table rows:", len(wide_df))
print("Wide unique GEOIDs:", wide_df["GEOID"].nunique())

print("Prevalence table rows:", len(prev_df))
print("Prevalence unique GEOIDs:", prev_df["GEOID"].nunique())

print("\nWide GEOID dtype:", wide_df["GEOID"].dtype)
print("Prevalence GEOID dtype:", prev_df["GEOID"].dtype)


# ============================================================
# 4. Keep prevalence fields only
# ============================================================

target_years = [
    2010,
    2014,
    2018,
    2022
]

prev = prev_df[
    [
        "GEOID",
        "Year",
        "Autism_prev1000"
    ]
].copy()

prev["Year"] = pd.to_numeric(
    prev["Year"],
    errors="coerce"
)

prev["Autism_prev1000"] = pd.to_numeric(
    prev["Autism_prev1000"],
    errors="coerce"
)

prev = prev[
    prev["Year"].isin(target_years)
].copy()


# ============================================================
# 5. Check GEOID-Year duplicates
# ============================================================

dup_n = prev.duplicated(
    subset=["GEOID", "Year"]
).sum()

print("\nDuplicate GEOID-Year records:", dup_n)

if dup_n > 0:
    raise ValueError(
        "Duplicate GEOID-Year records found in prevalence table."
    )


# ============================================================
# 6. Convert prevalence from long to wide
# ============================================================

prev_wide = prev.pivot(
    index="GEOID",
    columns="Year",
    values="Autism_prev1000"
)

prev_wide.columns = [
    f"Autism_prev1000_{int(year)}"
    for year in prev_wide.columns
]

prev_wide = prev_wide.reset_index()


# ============================================================
# 7. Merge prevalence onto EXISTING wide SHAP table
# ============================================================

# IMPORTANT:
# wide_df stays as the base table.
# All its original fields and order are preserved.
merged_df = wide_df.merge(
    prev_wide,
    on="GEOID",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 8. Put prevalence fields at the END
# ============================================================

prevalence_cols = [
    f"Autism_prev1000_{year}"
    for year in target_years
    if f"Autism_prev1000_{year}" in merged_df.columns
]

original_cols = [
    col for col in wide_df.columns
]

merged_df = merged_df[
    original_cols + prevalence_cols
].copy()


# ============================================================
# 9. QC after merge
# ============================================================

print("\n" + "=" * 70)
print("MERGE QC")
print("=" * 70)

print("Final rows:", len(merged_df))
print("Final unique GEOIDs:", merged_df["GEOID"].nunique())

print("\nAdded prevalence columns:")
print(prevalence_cols)

print("\nMissing prevalence values:")

for col in prevalence_cols:
    print(
        col,
        merged_df[col].isna().sum()
    )


# ============================================================
# 10. Save
# ============================================================

merged_df.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("SAVED")
print("=" * 70)

print(output_path)

print("\nFinal columns:")
print(merged_df.columns.tolist())

print("\nPreview:")
print(merged_df.head())

## 15. Temporal distribution of dominant predictors

Summarize and visualize the annual proportion of districts dominated by each local SHAP predictor.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==============================
# 1. Path settings
# ==============================

tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)

# LightGBM local-SHAP plotting-ready data
input_file = os.path.join(
    tables_dir,
    "Table_local_SHAP_for_plotting_only.csv"
)

# Output percentage table
output_table = os.path.join(
    tables_dir,
    "factor_percentage_2decimal.xlsx"
)

# Output Figure 8
output_file = os.path.join(
    figures_dir,
    "Figure7_temporal_distribution_dominant_predictors.tiff"
)


# ==============================
# 2. Read local SHAP plotting data
# ==============================

df = pd.read_csv(
    input_file,
    dtype={"GEOID": str}
)

df.columns = df.columns.str.strip()

print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(input_file)

print("\nOriginal data shape:")
print(df.shape)


# ==============================
# 3. Check required columns
# ==============================

required_cols = [
    "GEOID",
    "Year",
    "Top_feature_abs"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# ==============================
# 4. Keep only needed columns
# ==============================

df = df[
    [
        "GEOID",
        "Year",
        "Top_feature_abs"
    ]
].copy()


# ==============================
# 5. Clean data
# ==============================

df = df.dropna(
    subset=[
        "GEOID",
        "Year",
        "Top_feature_abs"
    ]
).copy()

# GEOID stays as text
df["GEOID"] = (
    df["GEOID"]
    .astype(str)
    .str.strip()
)

# Year to numeric
df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
).astype("Int64")


# Keep only 2010–2022
df = df[
    (df["Year"] >= 2010) &
    (df["Year"] <= 2022)
].copy()

print("\nData used for temporal distribution:")
print(df.shape)


# ==============================
# 6. QC: annual sample size
# ==============================

annual_n = (
    df.groupby("Year")
    .size()
    .reset_index(name="N")
)

print("\nAnnual number of district-year records:")
print(
    annual_n.to_string(index=False)
)


# ==============================
# 7. Calculate annual percentages
# ==============================

count_table = (
    df.groupby(
        [
            "Year",
            "Top_feature_abs"
        ]
    )
    .size()
    .reset_index(name="Count")
)

total_by_year = (
    df.groupby("Year")
    .size()
    .reset_index(name="Total")
)

count_table = count_table.merge(
    total_by_year,
    on="Year",
    how="left"
)

count_table["Percentage"] = (
    count_table["Count"]
    / count_table["Total"]
    * 100
)

# Round to 2 decimals
count_table["Percentage"] = (
    count_table["Percentage"]
    .round(2)
)


# ==============================
# 8. Reshape to table format
#    Rows = predictor
#    Columns = year
# ==============================

factor_percentage = (
    count_table
    .pivot(
        index="Top_feature_abs",
        columns="Year",
        values="Percentage"
    )
    .fillna(0)
)


# ==============================
# 9. Rename predictor labels
# ==============================

factor_name_map = {
    "PM2.5 concentration":
        r"PM$_{2.5}$ concentration"
}

factor_percentage = (
    factor_percentage
    .rename(index=factor_name_map)
)

factor_percentage.index.name = "Factor"


# ==============================
# 10. Order predictors
#     by overall contribution frequency
# ==============================

factor_order = (
    factor_percentage
    .sum(axis=1)
    .sort_values(
        ascending=False
    )
    .index
)

factor_percentage = (
    factor_percentage
    .loc[factor_order]
)


# ==============================
# 11. Save percentage table
# ==============================

factor_percentage.to_excel(
    output_table
)

print("\nPercentage table saved to:")
print(output_table)


# ==============================
# 12. Prepare plotting data
# ==============================

# Rows = year
# Columns = predictors
df_plot = factor_percentage.T


# ==============================
# 13. Plot style
# ==============================

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 10.5,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.unicode_minus": True
})

# High-distinction palette
colors = sns.color_palette(
    "tab20",
    n_colors=df_plot.shape[1]
)


# ==============================
# 14. Plot
# ==============================

fig, ax = plt.subplots(
    figsize=(12, 6.5)
)

df_plot.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors,
    width=0.8,
    edgecolor="white",
    linewidth=0.4
)


# ==============================
# 15. Axis labels
# ==============================

ax.set_xlabel(
    "Year",
    fontsize=11
)

ax.set_ylabel(
    "Proportion of school districts (%)",
    fontsize=11
)

ax.set_ylim(
    0,
    100
)


# ==============================
# 16. Tick labels
# ==============================

ax.tick_params(
    axis="x",
    labelsize=10,
    rotation=0
)

ax.tick_params(
    axis="y",
    labelsize=10
)


# ==============================
# 17. Spines and grid
# ==============================

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)


# ==============================
# 18. Legend
# ==============================

ax.legend(
    title="Dominant predictor",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=10,
    title_fontsize=11,
    frameon=False,
    ncol=1
)


# ==============================
# 19. Layout
# ==============================

plt.tight_layout()


# ==============================
# 20. Save figure
# ==============================

plt.savefig(
    output_file,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()


# ==============================
# 21. Final output
# ==============================

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("\nTable:")
print(output_table)

print("\nFigure:")
print(output_file)

## 16. Local SHAP and LIME analysis

Generate Tables S8–S9 and Figures S5–S6 for the fixed set of 30 high-prevalence districts.

In [ ]:
# ============================================================
# FINAL LOCAL SHAP + LIME ANALYSIS
#
# Outputs:
#   Table S8
#   Table S9
#   Figure S5
#   Figure S6
#
# Final model:
#   LightGBM
#
# Reproducibility:
#   random_state = 42
#   LIME num_samples = 5000
#
# IMPORTANT:
#   1. Existing LightGBM local SHAP values are reused.
#   2. SHAP values are NOT recalculated.
#   3. LIME uses the SAME final LightGBM model.
#   4. GEOID is treated as text.
#   5. Four years share one common scale within each figure.
#   6. Predictor labels include full names and units.
#   7. Figure size/font style restored close to old version.
# ============================================================


# ============================================================
# 0. Packages
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42
LIME_NUM_SAMPLES = 5000

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


# ============================================================
# 2. File paths
# ============================================================

model_data_path = str(PROCESSED_DIR / 'autism_model_final.csv')

shap_data_path = str(TABLES_DIR / 'Table_SHAP_dependence_point_level_all13.csv')

tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)


# ============================================================
# 3. Output files
# ============================================================

table_s8_path = os.path.join(
    tables_dir,
    "Table_S8_top30_school_districts.csv"
)

table_s9_path = os.path.join(
    tables_dir,
    "Table_S9_LIME_mean_absolute_contributions.csv"
)

figure_s5_path = os.path.join(
    figures_dir,
    "Figure_S5_local_SHAP_heatmap.tiff"
)

figure_s6_path = os.path.join(
    figures_dir,
    "Figure_S6_local_LIME_heatmap.tiff"
)

top30_detail_path = os.path.join(
    tables_dir,
    "Table_S8_top30_school_districts_four_year_details.csv"
)

shap_selected_long_path = os.path.join(
    tables_dir,
    "Table_local_SHAP_top30_four_years_long.csv"
)

lime_all13_long_path = os.path.join(
    tables_dir,
    "Table_local_LIME_all13_top30_four_years_long.csv"
)

lime_prediction_check_path = os.path.join(
    tables_dir,
    "Table_LIME_prediction_check_top30_four_years.csv"
)

shap_ranking_check_path = os.path.join(
    tables_dir,
    "Table_SHAP_global_ranking_recalculated_from_existing_values.csv"
)


# ============================================================
# 4. Variables
# ============================================================

target_col = "Autism_prev1000"

feature_cols = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]

selected_years = [
    2010,
    2014,
    2018,
    2022
]


# ============================================================
# 5. GEOID cleaning
# ============================================================

def clean_geoid(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


# ============================================================
# 6. Full publication labels + units
# ============================================================

feature_label_mapping = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}

feature_full_name_mapping = feature_label_mapping.copy()


# ============================================================
# 7. Load modeling data
# ============================================================

df = pd.read_csv(
    model_data_path,
    dtype={"GEOID": str}
)

df.columns = df.columns.str.strip()

df["GEOID"] = clean_geoid(
    df["GEOID"]
)

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)


# ============================================================
# 8. Required columns
# ============================================================

required_cols = (
    ["GEOID", "Year", target_col]
    + feature_cols
)

missing_cols = [
    c
    for c in required_cols
    if c not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# ============================================================
# 9. Final modeling data
# ============================================================

model_data_all = (
    df[required_cols]
    .dropna()
    .copy()
)

model_data_all["Year"] = (
    model_data_all["Year"]
    .astype(int)
)

model_data_all.reset_index(
    drop=True,
    inplace=True
)

duplicate_n = (
    model_data_all
    .duplicated(
        subset=["GEOID", "Year"]
    )
    .sum()
)

if duplicate_n > 0:
    raise ValueError(
        f"{duplicate_n} duplicate GEOID-Year records found."
    )

X_all = model_data_all[
    feature_cols
].copy()

y_all = model_data_all[
    target_col
].copy()


# ============================================================
# 10. Four representative years
# ============================================================

plot_data = model_data_all[
    model_data_all["Year"].isin(
        selected_years
    )
].copy()


# ============================================================
# 11. Districts present in all four years
# ============================================================

geoid_year_counts = (
    plot_data
    .groupby("GEOID")["Year"]
    .nunique()
)

common_geoids = (
    geoid_year_counts[
        geoid_year_counts
        == len(selected_years)
    ]
    .index
    .tolist()
)

common_data = plot_data[
    plot_data["GEOID"].isin(
        common_geoids
    )
].copy()


# ============================================================
# 12. Table S8
# ============================================================

geoid_rank = (
    common_data
    .groupby("GEOID")[target_col]
    .mean()
    .sort_values(
        ascending=False
    )
)

top_30_geoids = (
    geoid_rank
    .head(30)
    .index
    .tolist()
)

top30_table = (
    geoid_rank
    .loc[top_30_geoids]
    .reset_index()
)

top30_table.columns = [
    "GEOID",
    "Mean autism prevalence (per 1,000 students)"
]

top30_table.insert(
    0,
    "Rank",
    range(
        1,
        len(top30_table) + 1
    )
)

top30_table[
    "Mean autism prevalence (per 1,000 students)"
] = (
    top30_table[
        "Mean autism prevalence (per 1,000 students)"
    ]
    .round(2)
)

top30_table["GEOID"] = clean_geoid(
    top30_table["GEOID"]
)

top30_table.to_csv(
    table_s8_path,
    index=False
)


# ============================================================
# 13. Selected 30 × 4 observations
# ============================================================

rank_map = dict(
    zip(
        top30_table["GEOID"],
        top30_table["Rank"]
    )
)

selected_data = plot_data[
    plot_data["GEOID"].isin(
        top_30_geoids
    )
].copy()

selected_data["Rank"] = (
    selected_data["GEOID"]
    .map(rank_map)
)

selected_data = (
    selected_data
    .sort_values(
        ["Rank", "Year"]
    )
    .reset_index(
        drop=True
    )
)

if len(selected_data) != 120:

    raise ValueError(
        "Expected 120 observations "
        "(30 districts × 4 years), "
        f"found {len(selected_data)}."
    )


top30_details = selected_data[
    [
        "Rank",
        "GEOID",
        "Year",
        target_col
    ]
].copy()

top30_details["GEOID"] = clean_geoid(
    top30_details["GEOID"]
)

top30_details.to_csv(
    top30_detail_path,
    index=False
)


# ============================================================
# 14. Load existing LightGBM SHAP
# ============================================================

shap_data = pd.read_csv(
    shap_data_path,
    dtype={"GEOID": str}
)

shap_data.columns = (
    shap_data.columns
    .str.strip()
)

shap_data["GEOID"] = clean_geoid(
    shap_data["GEOID"]
)

shap_data["Year"] = pd.to_numeric(
    shap_data["Year"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 15. SHAP columns
# ============================================================

shap_cols = [
    f"{f}_SHAP"
    for f in feature_cols
]

missing_shap_cols = [
    c
    for c in (
        ["GEOID", "Year"]
        + shap_cols
    )
    if c not in shap_data.columns
]

if missing_shap_cols:

    raise ValueError(
        f"Missing SHAP columns: {missing_shap_cols}"
    )


# ============================================================
# 16. Global SHAP ranking
# ============================================================

shap_rank_records = []

for f in feature_cols:

    vals = pd.to_numeric(
        shap_data[
            f"{f}_SHAP"
        ],
        errors="coerce"
    )

    shap_rank_records.append({

        "Feature":
            f,

        "MeanAbsSHAP":
            np.abs(vals).mean()
    })


shap_rank_df = (
    pd.DataFrame(
        shap_rank_records
    )
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

shap_rank_df.insert(
    0,
    "Rank",
    range(
        1,
        len(shap_rank_df) + 1
    )
)

shap_rank_df["Predictor"] = (
    shap_rank_df["Feature"]
    .map(
        feature_full_name_mapping
    )
)

shap_rank_df.to_csv(
    shap_ranking_check_path,
    index=False
)


# ============================================================
# 17. Top 6
# ============================================================

top_features = (
    shap_rank_df
    .head(6)["Feature"]
    .tolist()
)

print("\nTop 6 predictors:")

for i, f in enumerate(
    top_features,
    start=1
):

    print(
        i,
        feature_full_name_mapping[f]
    )


# ============================================================
# 18. Selected local SHAP
# ============================================================

shap_selected = shap_data[
    (
        shap_data["GEOID"]
        .isin(top_30_geoids)
    )
    &
    (
        shap_data["Year"]
        .isin(selected_years)
    )
].copy()

shap_selected["Rank"] = (
    shap_selected["GEOID"]
    .map(rank_map)
)

shap_selected = (
    shap_selected
    .sort_values(
        ["Rank", "Year"]
    )
    .reset_index(
        drop=True
    )
)

if len(shap_selected) != 120:

    raise ValueError(
        "SHAP data do not contain exactly "
        "120 selected observations."
    )


# ============================================================
# 19. SHAP long table
# ============================================================

shap_long_records = []

for _, row in shap_selected.iterrows():

    for f in feature_cols:

        value = float(
            row[
                f"{f}_SHAP"
            ]
        )

        shap_long_records.append({

            "Rank":
                int(row["Rank"]),

            "GEOID":
                str(row["GEOID"]),

            "Year":
                int(row["Year"]),

            "Feature":
                f,

            "Predictor":
                feature_full_name_mapping[f],

            "SHAP_Local_Contribution":
                value,

            "Abs_SHAP_Local_Contribution":
                abs(value)
        })


shap_long = pd.DataFrame(
    shap_long_records
)

shap_long["GEOID"] = clean_geoid(
    shap_long["GEOID"]
)

shap_long.to_csv(
    shap_selected_long_path,
    index=False
)


# ============================================================
# 20. Final LightGBM
# ============================================================

final_model = lgb.LGBMRegressor(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=-1,

    num_leaves=50,

    subsample=0.8,

    colsample_bytree=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbose=-1
)

final_model.fit(
    X_all,
    y_all
)


# ============================================================
# 21. LIME
# ============================================================

lime_explainer = LimeTabularExplainer(

    training_data=
        X_all.values,

    feature_names=
        feature_cols,

    mode=
        "regression",

    discretize_continuous=
        False,

    random_state=
        RANDOM_STATE
)


def predict_fn(arr):

    return final_model.predict(
        arr
    )


feature_index_map = {

    f: i

    for i, f
    in enumerate(
        feature_cols
    )
}


# ============================================================
# 22. Calculate LIME
# ============================================================

lime_records = []
prediction_check_records = []


for _, row in selected_data.iterrows():

    geoid = clean_geoid(
        pd.Series(
            [row["GEOID"]]
        )
    ).iloc[0]


    x_row = (
        row[
            feature_cols
        ]
        .values
        .astype(float)
    )


    exp = lime_explainer.explain_instance(

        data_row=x_row,

        predict_fn=predict_fn,

        num_features=len(
            feature_cols
        ),

        num_samples=
            LIME_NUM_SAMPLES
    )


    local_coef = dict(
        exp.local_exp[1]
    )


    x_row_scaled = (
        lime_explainer
        .scaler
        .transform(
            x_row.reshape(
                1,
                -1
            )
        )[0]
    )


    all_contributions = {}


    for f in feature_cols:

        idx = feature_index_map[f]

        coef = local_coef.get(
            idx,
            0.0
        )

        scaled_value = (
            x_row_scaled[idx]
        )

        local_contribution = (
            coef
            * scaled_value
        )

        all_contributions[
            f
        ] = local_contribution


        lime_records.append({

            "Rank":
                int(row["Rank"]),

            "GEOID":
                geoid,

            "Year":
                int(row["Year"]),

            "Autism_prev1000":
                float(
                    row[target_col]
                ),

            "Feature":
                f,

            "Predictor":
                feature_full_name_mapping[f],

            "LIME_Local_Coefficient":
                coef,

            "LIME_Scaled_Feature_Value":
                scaled_value,

            "LIME_Local_Contribution":
                local_contribution,

            "Abs_LIME_Local_Contribution":
                abs(local_contribution)
        })


    sum_all = sum(
        all_contributions.values()
    )


    lime_intercept = float(
        exp.intercept[1]
    )


    lime_prediction = (
        lime_intercept
        + sum_all
    )


    model_prediction = float(
        final_model.predict(
            x_row.reshape(
                1,
                -1
            )
        )[0]
    )


    prediction_check_records.append({

        "Rank":
            int(row["Rank"]),

        "GEOID":
            geoid,

        "Year":
            int(row["Year"]),

        "LightGBM_prediction":
            model_prediction,

        "LIME_intercept":
            lime_intercept,

        "Sum_all_LIME_local_contributions":
            sum_all,

        "LIME_local_prediction":
            lime_prediction,

        "Difference_LightGBM_minus_LIME_local":
            model_prediction
            - lime_prediction,

        "Absolute_difference":
            abs(
                model_prediction
                - lime_prediction
            )
    })


# ============================================================
# 23. Save LIME outputs
# ============================================================

lime_long = pd.DataFrame(
    lime_records
)

lime_prediction_check = pd.DataFrame(
    prediction_check_records
)

lime_long["GEOID"] = clean_geoid(
    lime_long["GEOID"]
)

lime_prediction_check[
    "GEOID"
] = clean_geoid(
    lime_prediction_check[
        "GEOID"
    ]
)

lime_long.to_csv(
    lime_all13_long_path,
    index=False
)

lime_prediction_check.to_csv(
    lime_prediction_check_path,
    index=False
)


# ============================================================
# 24. Table S9
# ============================================================

table_s9 = (
    lime_long
    .groupby(
        [
            "Feature",
            "Predictor"
        ],
        as_index=False
    )[
        "Abs_LIME_Local_Contribution"
    ]
    .mean()
    .rename(
        columns={
            "Abs_LIME_Local_Contribution":
                "Mean absolute LIME local contribution"
        }
    )
    .sort_values(
        "Mean absolute LIME local contribution",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


table_s9.insert(
    0,
    "Rank",
    range(
        1,
        len(table_s9) + 1
    )
)


table_s9_output = table_s9[
    [
        "Rank",
        "Predictor",
        "Mean absolute LIME local contribution"
    ]
].copy()


table_s9_output[
    "Mean absolute LIME local contribution"
] = (
    table_s9_output[
        "Mean absolute LIME local contribution"
    ]
    .round(2)
)


table_s9_output.to_csv(
    table_s9_path,
    index=False
)


# ============================================================
# 25. Build SHAP matrices
# ============================================================

shap_year_matrices = {}


for year in selected_years:

    year_data = shap_selected[
        shap_selected["Year"]
        == year
    ].copy()

    year_data = year_data.sort_values(
        "Rank"
    )


    matrix = pd.DataFrame(

        index=
            year_data["GEOID"]
            .astype(str)
            .values
    )


    for f in top_features:

        matrix[f] = (
            year_data[
                f"{f}_SHAP"
            ]
            .values
        )


    matrix = matrix.reindex(
        top_30_geoids
    )


    shap_year_matrices[
        year
    ] = matrix


# ============================================================
# 26. Build LIME matrices
# ============================================================

lime_year_matrices = {}


for year in selected_years:

    year_long = lime_long[
        (
            lime_long["Year"]
            == year
        )
        &
        (
            lime_long["Feature"]
            .isin(
                top_features
            )
        )
    ].copy()


    matrix = (
        year_long
        .pivot(
            index="GEOID",
            columns="Feature",
            values=
                "LIME_Local_Contribution"
        )
    )


    matrix = matrix.reindex(
        top_30_geoids
    )


    matrix = matrix[
        top_features
    ]


    lime_year_matrices[
        year
    ] = matrix


# ============================================================
# 27. Heatmap function
#
# UPDATED VISUAL SETTINGS:
#
# - Larger total figure height
# - Heatmap itself stays tall
# - Predictor labels restored to 22 pt
# - Year titles = 22 pt
# - GEOID labels = 17 pt
# - Colorbar full height, right side
#
# Analysis itself is unchanged.
# ============================================================

def plot_four_year_heatmap(

    year_matrices,

    selected_years,

    feature_label_mapping,

    output_path,

    colorbar_label

):

    # --------------------------------------------------------
    # Shared scale across all 4 years
    # --------------------------------------------------------

    global_vmin = min(
        matrix.min().min()
        for matrix
        in year_matrices.values()
    )

    global_vmax = max(
        matrix.max().max()
        for matrix
        in year_matrices.values()
    )

    max_abs = max(
        abs(global_vmin),
        abs(global_vmax)
    )

    if max_abs == 0:
        max_abs = 1.0


    print(
        f"\nShared color scale for "
        f"{colorbar_label}: "
        f"{-max_abs:.4f} to "
        f"{max_abs:.4f}"
    )


    # --------------------------------------------------------
    # Restore OLD font sizes
    # --------------------------------------------------------

    plt.rcParams.update({

        "font.family":
            "Arial",

        "font.size":
            17.0,

        "axes.labelsize":
            16.5,

        "axes.titlesize":
            17.5,

        "xtick.labelsize":
            22,

        "ytick.labelsize":
            17,

        "axes.unicode_minus":
            True
    })


    # --------------------------------------------------------
    # Slightly increase TOTAL height.
    #
    # Old = (20, 14)
    #
    # Extra bottom space is needed because labels now contain
    # full names + units.
    #
    # This prevents the actual heatmap from being compressed.
    # --------------------------------------------------------

    fig, axes = plt.subplots(

        nrows=1,

        ncols=len(
            selected_years
        ),

        figsize=(
            20,
            16
        ),

        sharey=True
    )


    # --------------------------------------------------------
    # Four heatmaps
    # --------------------------------------------------------

    for i, year in enumerate(
        selected_years
    ):

        matrix = (
            year_matrices[
                year
            ]
            .copy()
        )


        matrix.index = clean_geoid(
            pd.Series(
                matrix.index
            )
        ).values


        matrix.columns = [

            feature_label_mapping[
                col
            ]

            for col
            in matrix.columns
        ]


        # Only right-most panel gets colorbar
        show_cbar = (
            i
            == len(
                selected_years
            ) - 1
        )


        heatmap = sns.heatmap(

            matrix,

            cmap=
                "coolwarm",

            ax=
                axes[i],

            cbar=
                show_cbar,

            vmin=
                -max_abs,

            vmax=
                max_abs,

            center=
                0,

            xticklabels=
                True,

            yticklabels=
                True,

            annot=
                False,

            linewidths=
                0.2,

            linecolor=
                "white",

            cbar_kws={
                "shrink": 1.0,
                "aspect": 30,
                "pad": 0.03
            }
        )


        # ----------------------------------------------------
        # Year
        # ----------------------------------------------------

        axes[i].set_title(

            f"{year}",

            fontsize=22,

            pad=10
        )


        # ----------------------------------------------------
        # Predictor labels
        #
        # Restored to OLD size = 22
        # ----------------------------------------------------

        axes[i].tick_params(

            axis="x",

            labelrotation=90,

            labelsize=22
        )


        # ----------------------------------------------------
        # School district labels
        # ----------------------------------------------------

        axes[i].tick_params(

            axis="y",

            labelsize=17
        )


        for label in (
            axes[i]
            .get_yticklabels()
        ):

            label.set_fontsize(
                17
            )

            label.set_fontweight(
                "medium"
            )


        # ----------------------------------------------------
        # Y-axis title
        # ----------------------------------------------------

        if i == 0:

            axes[i].set_ylabel(

                "School districts",

                fontsize=20
            )

        else:

            axes[i].set_ylabel(
                ""
            )


        axes[i].set_xlabel(
            ""
        )


        # ----------------------------------------------------
        # Shared colorbar
        # ----------------------------------------------------

        if show_cbar:

            cbar = (
                heatmap
                .collections[0]
                .colorbar
            )


            ticks = np.linspace(

                -max_abs,

                max_abs,

                5
            )


            cbar.set_ticks(
                ticks
            )


            cbar.set_ticklabels([

                f"{t:.2f}"

                for t
                in ticks
            ])


            cbar.ax.tick_params(
                labelsize=18
            )


            cbar.set_label(

                colorbar_label,

                fontsize=20
            )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Do NOT use tight_layout().
    #
    # tight_layout() was compressing the heatmaps vertically
    # because the x-axis labels are now much longer.
    #
    # Instead explicitly reserve space below for labels.
    # --------------------------------------------------------

    fig.subplots_adjust(

        left=0.075,

        right=0.93,

        top=0.93,

        bottom=0.28,

        wspace=0.08
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    plt.savefig(

        output_path,

        dpi=600,

        format="tiff",

        bbox_inches="tight"
    )


    plt.show()


# ============================================================
# 28. Figure S5
# ============================================================

plot_four_year_heatmap(

    year_matrices=
        shap_year_matrices,

    selected_years=
        selected_years,

    feature_label_mapping=
        feature_label_mapping,

    output_path=
        figure_s5_path,

    colorbar_label=
        "SHAP local contribution"
)


# ============================================================
# 29. Figure S6
# ============================================================

plot_four_year_heatmap(

    year_matrices=
        lime_year_matrices,

    selected_years=
        selected_years,

    feature_label_mapping=
        feature_label_mapping,

    output_path=
        figure_s6_path,

    colorbar_label=
        "LIME local contribution"
)


# ============================================================
# 30. QC
# ============================================================

print("\n" + "=" * 75)
print("REPRODUCIBILITY / QC")
print("=" * 75)

print(
    "Random state:",
    RANDOM_STATE
)

print(
    "LIME num_samples:",
    LIME_NUM_SAMPLES
)

print(
    "Selected districts:",
    len(top_30_geoids)
)

print(
    "Selected observations:",
    len(selected_data)
)

print(
    "\nMaximum absolute "
    "LightGBM-LIME prediction difference:"
)

print(
    lime_prediction_check[
        "Absolute_difference"
    ].max()
)

print(
    "\nMean absolute "
    "LightGBM-LIME prediction difference:"
)

print(
    lime_prediction_check[
        "Absolute_difference"
    ].mean()
)


# ============================================================
# 31. Outputs
# ============================================================

print("\n" + "=" * 75)
print("FILES SAVED")
print("=" * 75)

print("\nTable S8:")
print(table_s8_path)

print("\nTable S9:")
print(table_s9_path)

print("\nFigure S5:")
print(figure_s5_path)

print("\nFigure S6:")
print(figure_s6_path)

print("\nAdditional QC files:")
print(top30_detail_path)
print(shap_selected_long_path)
print(lime_all13_long_path)
print(lime_prediction_check_path)
print(shap_ranking_check_path)

print("\nDone.")

## 17. Environmental-signature classification

Classify district-year observations using the Top-3 SHAP signature rules and generate Figure S7.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ============================================================
# Figure S7
# Top-3 SHAP Environmental Signature Classification
# Based on final LightGBM local SHAP results
# ============================================================

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)

input_file = os.path.join(
    tables_dir,
    "Table_local_SHAP_matrix_raw.csv"
)

assignment_out = os.path.join(
    tables_dir,
    "top3_signature_assignments.csv"
)

mean_profile_out = os.path.join(
    tables_dir,
    "top3_signature_mean_SHAP_profiles.csv"
)

standardized_profile_out = os.path.join(
    tables_dir,
    "top3_signature_standardized_profiles.csv"
)

temporal_out = os.path.join(
    tables_dir,
    "top3_signature_temporal_percent.csv"
)

figure_out = os.path.join(
    figures_dir,
    "Figure_S7_environmental_signature_profiles_temporal_evolution.tiff"
)


# ------------------------------------------------------------
# 2. Load LightGBM local SHAP data
# ------------------------------------------------------------

df = pd.read_csv(
    input_file,
    dtype={"GEOID": str}
)

df.columns = df.columns.str.strip()

if "GEOID" in df.columns:
    df["GEOID"] = (
        df["GEOID"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

if "Year" in df.columns:
    df["Year"] = pd.to_numeric(
        df["Year"],
        errors="coerce"
    ).astype("Int64")


print("=" * 75)
print("INPUT DATA")
print("=" * 75)

print(input_file)

print("\nData shape:")
print(df.shape)


# ------------------------------------------------------------
# 3. Final 13 SHAP variables
# ------------------------------------------------------------

expected_shap_cols = [
    "SHAP_PM25",
    "SHAP_tmax_c",
    "SHAP_ppt_mm",
    "SHAP_Below_poverty_percent",
    "SHAP_Bachelor_plus_percent",
    "SHAP_under5_percent",
    "SHAP_age5to14_percent",
    "SHAP_age15to19_percent",
    "SHAP_median_age",
    "SHAP_Pop_density",
    "SHAP_hispanic_percent",
    "SHAP_asian_percent",
    "SHAP_Forest_percent"
]

missing_shap_cols = [
    col
    for col in expected_shap_cols
    if col not in df.columns
]

if missing_shap_cols:
    raise ValueError(
        f"Missing expected SHAP columns: {missing_shap_cols}"
    )

shap_cols = expected_shap_cols.copy()


# ------------------------------------------------------------
# 4. Calculate Top-3 SHAP contribution ratio
# ------------------------------------------------------------

abs_shap = df[shap_cols].abs()

total_abs_shap = abs_shap.sum(axis=1)

top3_abs_shap = (
    np.sort(
        abs_shap.values,
        axis=1
    )[:, -3:]
    .sum(axis=1)
)

df["Top3_SHAP_Ratio"] = np.where(
    total_abs_shap > 0,
    top3_abs_shap / total_abs_shap,
    np.nan
)

print(
    "\nMean proportion of total absolute SHAP explained by Top 3 predictors:"
)

print(
    df["Top3_SHAP_Ratio"].mean()
)


# ------------------------------------------------------------
# 5. Identify Top-3 predictors for each district-year
# ------------------------------------------------------------

top3_predictors = (
    abs_shap
    .apply(
        lambda row:
            row
            .sort_values(ascending=False)
            .head(3)
            .index
            .tolist(),
        axis=1
    )
)

df["Top1_SHAP_Predictor"] = top3_predictors.apply(
    lambda x: x[0]
)

df["Top2_SHAP_Predictor"] = top3_predictors.apply(
    lambda x: x[1]
)

df["Top3_SHAP_Predictor"] = top3_predictors.apply(
    lambda x: x[2]
)


# ------------------------------------------------------------
# 6. Define predictor groups
# ------------------------------------------------------------

urban_demographic = {
    "SHAP_Pop_density",
    "SHAP_hispanic_percent",
    "SHAP_Bachelor_plus_percent",
    "SHAP_asian_percent"
}

environmental_climate = {
    "SHAP_PM25",
    "SHAP_Forest_percent",
    "SHAP_tmax_c",
    "SHAP_ppt_mm"
}


# ------------------------------------------------------------
# 7. Assign environmental signatures
# ------------------------------------------------------------

def assign_signature(top3):

    top3_set = set(top3)

    urban_count = len(
        top3_set.intersection(
            urban_demographic
        )
    )

    env_count = len(
        top3_set.intersection(
            environmental_climate
        )
    )

    if urban_count >= 2:

        return (
            "S1",
            "Urban-Demographic Signature"
        )

    elif env_count >= 2:

        return (
            "S2",
            "Environmental-Climate Signature"
        )

    else:

        return (
            "S3",
            "Mixed Socioenvironmental Signature"
        )


signature_results = (
    top3_predictors
    .apply(
        assign_signature
    )
)

df["Signature_Label"] = signature_results.apply(
    lambda x: x[0]
)

df["Signature_Name"] = signature_results.apply(
    lambda x: x[1]
)

signature_order = [
    "S1",
    "S2",
    "S3"
]


# ------------------------------------------------------------
# 8. Summarize signature counts
# ------------------------------------------------------------

signature_summary = (
    df
    .groupby(
        [
            "Signature_Label",
            "Signature_Name"
        ]
    )
    .size()
    .reset_index(
        name="N"
    )
)

signature_summary["Percent"] = (
    100
    * signature_summary["N"]
    / len(df)
)

print("\n" + "=" * 75)
print("SIGNATURE SUMMARY")
print("=" * 75)

print(
    signature_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 9. Mean SHAP profiles by signature
# ------------------------------------------------------------

mean_profiles = (
    df
    .groupby(
        [
            "Signature_Label",
            "Signature_Name"
        ]
    )[shap_cols]
    .mean()
    .reset_index()
)

mean_profiles.to_csv(
    mean_profile_out,
    index=False
)


# ------------------------------------------------------------
# 10. Top 8 SHAP variables for visualization
# ------------------------------------------------------------

top_n = 8

mean_abs_shap = (
    df[shap_cols]
    .abs()
    .mean()
    .sort_values(
        ascending=False
    )
)

top_shap_cols = (
    mean_abs_shap
    .head(top_n)
    .index
    .tolist()
)

print("\nTop 8 SHAP variables used in Figure S7:")

print(
    mean_abs_shap.head(
        top_n
    )
)


# ------------------------------------------------------------
# 11. Standardize signature mean profiles
# ------------------------------------------------------------

standardized_profiles = mean_profiles.copy()

profile_values = mean_profiles[
    top_shap_cols
]

profile_sd = profile_values.std(
    ddof=0
)

profile_sd = profile_sd.replace(
    0,
    np.nan
)

standardized_profiles[
    top_shap_cols
] = (
    profile_values
    - profile_values.mean()
) / profile_sd

standardized_profiles.to_csv(
    standardized_profile_out,
    index=False
)


# ------------------------------------------------------------
# 12. Temporal evolution
# ------------------------------------------------------------

if "Year" not in df.columns:
    raise ValueError(
        "The dataset must include a 'Year' column."
    )

temporal_counts = (
    df
    .groupby(
        [
            "Year",
            "Signature_Label"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)

temporal_counts["Percent"] = (
    temporal_counts
    .groupby("Year")["Count"]
    .transform(
        lambda x:
            100 * x / x.sum()
    )
)

temporal_wide = (
    temporal_counts
    .pivot(
        index="Year",
        columns="Signature_Label",
        values="Percent"
    )
    .fillna(0)
    .reindex(
        columns=signature_order
    )
)

temporal_wide.to_csv(
    temporal_out
)


print("\n" + "=" * 75)
print("TEMPORAL SIGNATURE DISTRIBUTION (%)")
print("=" * 75)

print(
    temporal_wide.round(2)
)


# ------------------------------------------------------------
# 13. Save complete signature assignment table
# ------------------------------------------------------------

df.to_csv(
    assignment_out,
    index=False
)


# ------------------------------------------------------------
# 14. Friendly labels for Figure S7
# ------------------------------------------------------------

figure_label_map = {

    "SHAP_PM25":
        r"PM$_{2.5}$",

    "SHAP_Pop_density":
        "Population density",

    "SHAP_hispanic_percent":
        "Hispanic/Latino",

    "SHAP_Bachelor_plus_percent":
        "Bachelor’s degree",

    "SHAP_Forest_percent":
        "Forest cover",

    "SHAP_tmax_c":
        "Max temperature",

    "SHAP_age15to19_percent":
        "Age 15–19",

    "SHAP_Below_poverty_percent":
        "Below poverty",

    "SHAP_median_age":
        "Median age",

    "SHAP_age5to14_percent":
        "Age 5–14",

    "SHAP_under5_percent":
        "Age <5",

    "SHAP_ppt_mm":
        "Precipitation",

    "SHAP_asian_percent":
        "Asian population"
}


# ------------------------------------------------------------
# 15. Combined Figure S7
# ------------------------------------------------------------

heatmap_data = (
    standardized_profiles
    .set_index(
        "Signature_Label"
    )[top_shap_cols]
    .reindex(
        signature_order
    )
)

heatmap_labels = [
    figure_label_map.get(
        col,
        col
    )
    for col in top_shap_cols
]


fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(16, 6),
    gridspec_kw={
        "width_ratios": [
            1.1,
            1.4
        ]
    }
)

fig.subplots_adjust(
    wspace=0.45
)


# ------------------------------------------------------------
# Panel A: Signature profiles
# ------------------------------------------------------------

ax1 = axes[0]


# ============================================================
# ACADEMICALLY CONSISTENT COLOR SCALE
#
# Keep the original viridis palette
# but force the numeric range to be symmetric around zero.
#
# This preserves the old visual style while ensuring that
# zero is exactly at the midpoint of the color scale.
# ============================================================

max_abs = np.nanmax(
    np.abs(
        heatmap_data.values
    )
)

if not np.isfinite(max_abs) or max_abs == 0:
    max_abs = 1.0


im = ax1.imshow(
    heatmap_data.values,
    aspect="auto",
    cmap="viridis",
    vmin=-max_abs,
    vmax=max_abs
)


ax1.set_xticks(
    np.arange(
        len(top_shap_cols)
    )
)

ax1.set_xticklabels(
    heatmap_labels,
    rotation=45,
    ha="right",
    fontsize=12
)

ax1.set_yticks(
    np.arange(
        len(heatmap_data.index)
    )
)

ax1.set_yticklabels(
    heatmap_data.index,
    fontsize=12
)

ax1.set_title(
    "(a) Signature profiles",
    fontsize=14
)


cbar = fig.colorbar(
    im,
    ax=ax1,
    fraction=0.046,
    pad=0.08
)

# Explicit symmetric ticks
cbar_ticks = np.linspace(
    -max_abs,
    max_abs,
    5
)

cbar.set_ticks(
    cbar_ticks
)

cbar.set_ticklabels(
    [
        f"{x:.2f}"
        for x in cbar_ticks
    ]
)

cbar.set_label(
    "Mean standardized SHAP value",
    fontsize=12
)

cbar.ax.tick_params(
    labelsize=12
)


# ------------------------------------------------------------
# Panel B: Temporal evolution
# ------------------------------------------------------------

ax2 = axes[1]

ax2.stackplot(
    temporal_wide.index,
    [
        temporal_wide[col]
        for col in signature_order
    ],
    labels=signature_order
)

ax2.set_title(
    "(b) Temporal evolution",
    fontsize=14
)

ax2.set_xlabel(
    "Year",
    fontsize=12
)

ax2.set_ylabel(
    "Percentage of school districts (%)",
    fontsize=12
)

ax2.set_ylim(
    0,
    100
)

ax2.tick_params(
    axis="both",
    labelsize=9
)


legend_labels = [
    "S1: Urban-Demographic Signature",
    "S2: Environmental-Climate Signature",
    "S3: Mixed Socioenvironmental Signature"
]

handles, _ = (
    ax2
    .get_legend_handles_labels()
)

ax2.legend(
    handles,
    legend_labels,
    title="Environmental Signatures",
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        -0.15
    ),
    ncol=1,
    frameon=False,
    fontsize=12,
    title_fontsize=12,
    handlelength=2.0,
    labelspacing=0.4
)


# ------------------------------------------------------------
# 16. Layout and save
# ------------------------------------------------------------

plt.tight_layout()

plt.subplots_adjust(
    bottom=0.18,
    wspace=0.25
)

plt.savefig(
    figure_out,
    dpi=300,
    format="tiff",
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 17. Final output summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FILES SAVED")
print("=" * 75)

print("\nSignature assignments:")
print(assignment_out)

print("\nMean SHAP profiles:")
print(mean_profile_out)

print("\nStandardized profiles:")
print(standardized_profile_out)

print("\nTemporal distribution:")
print(temporal_out)

print("\nFigure S7:")
print(figure_out)

## 18. Install spatial-analysis dependencies

Run this one-time setup cell only if GeoPandas, libpysal, or esda are not already installed.

In [ ]:
%pip install geopandas libpysal esda

## 19. Verify spatial-analysis packages

Confirm that the geospatial packages required for Local Moran's I are available.

In [ ]:
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran_Local

print("GeoPandas version:", gpd.__version__)
print("Spatial packages imported successfully.")

## 20. Table 1: Local Moran's I spatial clustering

Quantify clustering and spatial outliers in 2022 local SHAP contributions.

In [ ]:
# ============================================================
# TABLE 1
# Spatial clustering of 2022 local SHAP contributions
# using Local Moran's I
#
# Final model:
#   LightGBM
#
# Spatial weights:
#   Queen contiguity
#   Row-standardized
#
# Significance:
#   permutation pseudo-p < 0.05
#
# Reproducibility:
#   permutations = 999
#   seed = 42
# ============================================================


# ============================================================
# 0. Packages
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

from libpysal.weights import Queen
from esda.moran import Moran_Local


# ============================================================
# 1. Reproducibility settings
# ============================================================

RANDOM_STATE = 42
N_PERMUTATIONS = 999
SIGNIFICANCE_LEVEL = 0.05


# ============================================================
# 2. Paths
# ============================================================

# ------------------------------------------------------------
# LightGBM local SHAP matrix
# ------------------------------------------------------------

shap_path = TABLES_DIR / 'Table_local_SHAP_matrix_raw.csv'


# ------------------------------------------------------------
# New York Unified School District shapefile
# ------------------------------------------------------------

shapefile_path = SPATIAL_DIR / 'tl_2020_36_unsd.shp'


# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

tables_dir = TABLES_DIR


# Detailed district-level Local Moran outputs
moran_dir = (
    tables_dir
    / "Spatial_Clustering_2022"
)

moran_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Final manuscript Table 1
table1_path = (
    tables_dir
    / "Table_1_local_Moran_spatial_clustering_2022.csv"
)


# Full QC table
summary_full_path = (
    tables_dir
    / "Table_1_local_Moran_spatial_clustering_2022_full.csv"
)


# Join QC
join_qc_path = (
    tables_dir
    / "Table_1_GEOID_join_QC_2022.csv"
)


# ============================================================
# 3. GEOID cleaning function
# ============================================================

def clean_geoid(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


# ============================================================
# 4. Predictors included in Table 1
#
# Keep the SAME seven predictors as the old Table 1.
# ============================================================

predictors = {

    "PM₂.₅":
        "SHAP_PM25",

    "Population density":
        "SHAP_Pop_density",

    "Bachelor's degree attainment":
        "SHAP_Bachelor_plus_percent",

    "Hispanic population":
        "SHAP_hispanic_percent",

    "Forest cover":
        "SHAP_Forest_percent",

    "Precipitation":
        "SHAP_ppt_mm",

    "Maximum temperature":
        "SHAP_tmax_c"
}


# Safe short names for detailed CSV files
file_names = {

    "PM₂.₅":
        "PM25",

    "Population density":
        "Popdensity",

    "Bachelor's degree attainment":
        "Bachelor",

    "Hispanic population":
        "Hispanic",

    "Forest cover":
        "Forest",

    "Precipitation":
        "PPT",

    "Maximum temperature":
        "Tmax"
}


# ============================================================
# 5. Check input files
# ============================================================

if not shap_path.exists():

    raise FileNotFoundError(
        f"SHAP file not found:\n{shap_path}"
    )


if not shapefile_path.exists():

    raise FileNotFoundError(
        f"Shapefile not found:\n{shapefile_path}"
    )


# ============================================================
# 6. Load LightGBM local SHAP data
# ============================================================

shap_df = pd.read_csv(
    shap_path,
    dtype={"GEOID": str}
)

shap_df.columns = (
    shap_df
    .columns
    .str.strip()
)

shap_df["GEOID"] = clean_geoid(
    shap_df["GEOID"]
)

shap_df["Year"] = pd.to_numeric(
    shap_df["Year"],
    errors="coerce"
)


print("=" * 80)
print("LIGHTGBM LOCAL SHAP INPUT")
print("=" * 80)

print("File:")
print(shap_path)

print("\nDimensions:")
print(shap_df.shape)

print("\nExample GEOIDs:")
print(
    shap_df["GEOID"]
    .head(10)
    .tolist()
)


# ============================================================
# 7. Check required SHAP columns
# ============================================================

required_cols = (
    ["GEOID", "Year"]
    + list(
        predictors.values()
    )
)


missing_cols = [
    col
    for col in required_cols
    if col not in shap_df.columns
]


if missing_cols:

    raise ValueError(
        "Missing required SHAP columns:\n"
        + "\n".join(
            missing_cols
        )
    )


# ============================================================
# 8. Select 2022 only
# ============================================================

shap_2022 = (
    shap_df[
        shap_df["Year"] == 2022
    ]
    .copy()
)


# One row per school district
duplicate_n = (
    shap_2022
    .duplicated(
        subset=["GEOID"]
    )
    .sum()
)


if duplicate_n > 0:

    raise ValueError(
        f"{duplicate_n} duplicate GEOIDs "
        "found in 2022 SHAP data."
    )


print("\n2022 SHAP records:")
print(
    len(shap_2022)
)


# ============================================================
# 9. Load school district shapefile
# ============================================================

gdf = gpd.read_file(
    shapefile_path
)


print("\n" + "=" * 80)
print("SCHOOL DISTRICT BOUNDARY DATA")
print("=" * 80)

print("Shapefile:")
print(shapefile_path)

print("\nDimensions:")
print(gdf.shape)

print("\nAvailable fields:")
print(
    gdf.columns.tolist()
)


# ============================================================
# 10. Automatically identify GEOID field
# ============================================================

preferred_geoid_fields = [
    "GEOID",
    "GEOID20",
    "GEOID10"
]


geoid_field = None


for candidate in preferred_geoid_fields:

    if candidate in gdf.columns:

        geoid_field = candidate

        break


if geoid_field is None:

    possible_geoid_fields = [
        col
        for col in gdf.columns
        if "geoid" in col.lower()
    ]

    if len(possible_geoid_fields) == 1:

        geoid_field = (
            possible_geoid_fields[0]
        )

    else:

        raise ValueError(
            "Could not uniquely identify the GEOID field.\n"
            f"Possible fields: {possible_geoid_fields}"
        )


print("\nGEOID field used from shapefile:")
print(
    geoid_field
)


gdf["GEOID"] = clean_geoid(
    gdf[
        geoid_field
    ]
)


# ============================================================
# 11. Basic geometry cleaning
# ============================================================

# Remove missing geometries
gdf = gdf[
    gdf.geometry.notna()
].copy()


# Remove empty geometries
gdf = gdf[
    ~gdf.geometry.is_empty
].copy()


invalid_n = (
    ~gdf.geometry.is_valid
).sum()


print("\nInvalid geometries before repair:")
print(
    invalid_n
)


if invalid_n > 0:

    try:

        gdf["geometry"] = (
            gdf.geometry
            .make_valid()
        )

    except Exception:

        gdf["geometry"] = (
            gdf.geometry
            .buffer(0)
        )


# ============================================================
# 12. Check GEOID uniqueness in shapefile
# ============================================================

shape_duplicate_n = (
    gdf
    .duplicated(
        subset=["GEOID"]
    )
    .sum()
)


if shape_duplicate_n > 0:

    raise ValueError(
        f"{shape_duplicate_n} duplicate GEOIDs "
        "found in the shapefile."
    )


# ============================================================
# 13. GEOID join QC
# ============================================================

shape_geoids = set(
    gdf["GEOID"]
)

shap_geoids = set(
    shap_2022["GEOID"]
)


matched_geoids = (
    shape_geoids
    .intersection(
        shap_geoids
    )
)

shap_not_in_shape = sorted(
    shap_geoids
    - shape_geoids
)

shape_not_in_shap = sorted(
    shape_geoids
    - shap_geoids
)


print("\n" + "=" * 80)
print("GEOID JOIN QC")
print("=" * 80)

print(
    "2022 SHAP GEOIDs:",
    len(shap_geoids)
)

print(
    "Boundary GEOIDs:",
    len(shape_geoids)
)

print(
    "Matched GEOIDs:",
    len(matched_geoids)
)

print(
    "SHAP GEOIDs without polygon:",
    len(shap_not_in_shape)
)

print(
    "Boundary GEOIDs without SHAP record:",
    len(shape_not_in_shap)
)


# Save join QC
join_qc_rows = []


for geoid in sorted(
    shap_geoids.union(
        shape_geoids
    )
):

    join_qc_rows.append({

        "GEOID":
            geoid,

        "In_SHAP_2022":
            geoid in shap_geoids,

        "In_Shapefile":
            geoid in shape_geoids,

        "Matched":
            (
                geoid in shap_geoids
                and
                geoid in shape_geoids
            )
    })


pd.DataFrame(
    join_qc_rows
).to_csv(
    join_qc_path,
    index=False
)


# ============================================================
# 14. Join 2022 SHAP to polygons
# ============================================================

analysis_gdf = (
    gdf[
        [
            "GEOID",
            "geometry"
        ]
    ]
    .merge(

        shap_2022[
            ["GEOID"]
            + list(
                predictors.values()
            )
        ],

        on="GEOID",

        how="inner",

        validate="one_to_one"
    )
)


analysis_gdf = gpd.GeoDataFrame(
    analysis_gdf,
    geometry="geometry",
    crs=gdf.crs
)


print("\nSuccessfully joined districts:")
print(
    len(analysis_gdf)
)


# ============================================================
# 15. Final missing-value check
# ============================================================

missing_analysis = (
    analysis_gdf[
        list(
            predictors.values()
        )
    ]
    .isna()
    .sum()
)


print("\nMissing SHAP values after join:")
print(
    missing_analysis
)


if missing_analysis.sum() > 0:

    raise ValueError(
        "Missing SHAP values remain after the spatial join."
    )


# ============================================================
# 16. Build Queen contiguity weights
#
# Neighbor definition:
# share an edge OR a vertex
# ============================================================

w = Queen.from_dataframe(
    analysis_gdf,
    ids=analysis_gdf[
        "GEOID"
    ].tolist()
)


# Row-standardized spatial weights
w.transform = "R"


print("\n" + "=" * 80)
print("SPATIAL WEIGHTS")
print("=" * 80)

print(
    "Number of districts:",
    w.n
)

print(
    "Connected components:",
    w.n_components
)

print(
    "Number of islands:",
    len(w.islands)
)


if len(w.islands) > 0:

    print("\nIsland GEOIDs:")
    print(
        w.islands
    )


# ============================================================
# 17. Local Moran's I
# ============================================================

summary_rows = []


for predictor_name, shap_col in predictors.items():

    print("\n" + "-" * 80)

    print(
        "Analyzing:",
        predictor_name
    )


    # --------------------------------------------------------
    # Predictor-specific SHAP values
    # --------------------------------------------------------

    values = pd.to_numeric(
        analysis_gdf[
            shap_col
        ],
        errors="coerce"
    ).to_numpy(
        dtype=float
    )


    if np.isnan(
        values
    ).any():

        raise ValueError(
            f"Missing values detected for "
            f"{predictor_name}."
        )


    # --------------------------------------------------------
    # Local Moran
    # --------------------------------------------------------

    local_moran = Moran_Local(

        values,

        w,

        permutations=
            N_PERMUTATIONS,

        seed=
            RANDOM_STATE
    )


    # --------------------------------------------------------
    # Significant Local Moran locations
    # --------------------------------------------------------

    significant = (
        local_moran.p_sim
        < SIGNIFICANCE_LEVEL
    )


    # --------------------------------------------------------
    # PySAL Moran quadrants
    #
    # q = 1 → HH
    # q = 2 → LH
    # q = 3 → LL
    # q = 4 → HL
    # --------------------------------------------------------

    cluster_type = np.full(
        len(values),
        "Not Significant",
        dtype=object
    )


    cluster_type[
        significant
        & (
            local_moran.q == 1
        )
    ] = "HH"


    cluster_type[
        significant
        & (
            local_moran.q == 2
        )
    ] = "LH"


    cluster_type[
        significant
        & (
            local_moran.q == 3
        )
    ] = "LL"


    cluster_type[
        significant
        & (
            local_moran.q == 4
        )
    ] = "HL"


    # --------------------------------------------------------
    # District-level output
    # --------------------------------------------------------

    local_output = pd.DataFrame({

        "GEOID":
            analysis_gdf[
                "GEOID"
            ].values,

        "SHAP_value":
            values,

        "Local_Moran_I":
            local_moran.Is,

        "Pseudo_p_value":
            local_moran.p_sim,

        "Quadrant":
            local_moran.q,

        "COType":
            cluster_type
    })


    local_output_path = (
        moran_dir
        / (
            file_names[
                predictor_name
            ]
            + "_Local_Moran_2022.csv"
        )
    )


    local_output.to_csv(
        local_output_path,
        index=False
    )


    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    counts = (
        pd.Series(
            cluster_type
        )
        .value_counts()
    )


    hh = int(
        counts.get(
            "HH",
            0
        )
    )

    ll = int(
        counts.get(
            "LL",
            0
        )
    )

    hl = int(
        counts.get(
            "HL",
            0
        )
    )

    lh = int(
        counts.get(
            "LH",
            0
        )
    )

    ns = int(
        counts.get(
            "Not Significant",
            0
        )
    )


    total = len(
        cluster_type
    )


    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------

    summary_rows.append({

        "Predictor":
            predictor_name,

        "HH (%)":
            round(
                hh
                / total
                * 100,
                1
            ),

        "LL (%)":
            round(
                ll
                / total
                * 100,
                1
            ),

        "Spatial Clustering (%)":
            round(
                (
                    hh
                    + ll
                )
                / total
                * 100,
                1
            ),

        "Spatial Outliers (%)":
            round(
                (
                    hl
                    + lh
                )
                / total
                * 100,
                1
            ),

        "N":
            total,

        "HH (N)":
            hh,

        "LL (N)":
            ll,

        "HL (N)":
            hl,

        "LH (N)":
            lh,

        "Not Significant (N)":
            ns
    })


    print(
        f"HH = {hh}, "
        f"LL = {ll}, "
        f"HL = {hl}, "
        f"LH = {lh}, "
        f"Not significant = {ns}"
    )


# ============================================================
# 18. Full summary table
# ============================================================

summary_full = pd.DataFrame(
    summary_rows
)


summary_full = (
    summary_full
    .sort_values(
        "Spatial Clustering (%)",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


summary_full.to_csv(
    summary_full_path,
    index=False
)


# ============================================================
# 19. Final manuscript Table 1
# ============================================================

table1 = summary_full[
    [
        "Predictor",
        "HH (%)",
        "LL (%)",
        "Spatial Clustering (%)",
        "Spatial Outliers (%)",
        "N"
    ]
].copy()


table1.to_csv(
    table1_path,
    index=False
)


# ============================================================
# 20. Print final Table 1
# ============================================================

print("\n" + "=" * 80)
print("FINAL TABLE 1")
print("=" * 80)

print(
    table1.to_string(
        index=False
    )
)


# ============================================================
# 21. Final file summary
# ============================================================

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print("\nFinal manuscript Table 1:")
print(
    table1_path
)

print("\nFull QC summary:")
print(
    summary_full_path
)

print("\nGEOID join QC:")
print(
    join_qc_path
)

print("\nDistrict-level Local Moran outputs:")
print(
    moran_dir
)

print("\nAnalysis completed.")

## 21. Overall environmental-signature summary

Calculate the overall proportions of the three environmental signatures.

In [ ]:
import pandas as pd
import os

# ============================================================
# 1. Input / output paths
# ============================================================

tables_dir = str(TABLES_DIR)

input_file = os.path.join(
    tables_dir,
    "top3_signature_assignments.csv"
)

output_file = os.path.join(
    tables_dir,
    "top3_signature_overall_summary.csv"
)


# ============================================================
# 2. Load assignment data
# ============================================================

df = pd.read_csv(input_file)


# ============================================================
# 3. Summarize overall signature distribution
# ============================================================

summary = (
    df.groupby(
        ["Signature_Label", "Signature_Name"]
    )
    .size()
    .reset_index(name="N")
)

summary["Percent"] = (
    summary["N"]
    / summary["N"].sum()
    * 100
)

summary["Percent"] = summary["Percent"].round(1)


# ============================================================
# 4. Order S1, S2, S3
# ============================================================

signature_order = ["S1", "S2", "S3"]

summary["Signature_Label"] = pd.Categorical(
    summary["Signature_Label"],
    categories=signature_order,
    ordered=True
)

summary = (
    summary
    .sort_values("Signature_Label")
    .reset_index(drop=True)
)


# ============================================================
# 5. Save
# ============================================================

summary.to_csv(
    output_file,
    index=False
)

print(summary)

print("\nSaved to:")
print(output_file)

## 22. Sensitivity analysis: 2013–2022

Refit the final LightGBM model using 2013–2022 and recompute SHAP importance.

In [ ]:
import pandas as pd
import numpy as np
import os
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


# ============================================================
# 0. Reproducibility / warnings
# ============================================================

RANDOM_STATE = 42

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)

np.random.seed(
    RANDOM_STATE
)


# ============================================================
# 1. Paths
# ============================================================

# Final complete-case modeling dataset
file_path = str(PROCESSED_DIR / 'autism_model_final.csv')

# Output folders
tables_dir = str(TABLES_DIR)

figures_dir = str(FIGURES_DIR)


# ------------------------------------------------------------
# Output tables
# ------------------------------------------------------------

performance_out = os.path.join(
    tables_dir,
    "Table_model_performance_LightGBM_2013_2022_sensitivity.csv"
)

shap_summary_out = os.path.join(
    tables_dir,
    "Table_SHAP_global_importance_direction_2013_2022_sensitivity.csv"
)

shap_matrix_out = os.path.join(
    tables_dir,
    "Table_SHAP_values_matrix_2013_2022_sensitivity.csv"
)


# ------------------------------------------------------------
# Output figures
# ------------------------------------------------------------

figure_s8_out = os.path.join(
    figures_dir,
    "Figure_S8_SHAP_2013_2022_sensitivity.tiff"
)

standard_summary_out = os.path.join(
    figures_dir,
    "Figure_SHAP_summary_standard_2013_2022_sensitivity.tiff"
)

standard_bar_out = os.path.join(
    figures_dir,
    "Figure_SHAP_bar_standard_2013_2022_sensitivity.tiff"
)


# ============================================================
# 2. Load final dataset
# ============================================================

df = pd.read_csv(
    file_path,
    dtype={"GEOID": str}
)

df.columns = (
    df.columns
    .str.strip()
)


# Clean GEOID as identifier
df["GEOID"] = (
    df["GEOID"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)


# Ensure Year numeric
df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)


# ============================================================
# 3. Define outcome and final 13 predictors
#    Sensitivity analysis = 2013–2022
# ============================================================

target_col = "Autism_prev1000"

feature_cols = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]


required_cols = (
    ["GEOID", "Year", target_col]
    + feature_cols
)


missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# ============================================================
# 4. Keep only 2013–2022
# ============================================================

model_data = (
    df[
        required_cols
    ]
    .copy()
)


model_data = (
    model_data[
        (model_data["Year"] >= 2013)
        &
        (model_data["Year"] <= 2022)
    ]
    .dropna()
    .copy()
    .reset_index(drop=True)
)


model_data["Year"] = (
    model_data["Year"]
    .astype(int)
)


X = model_data[
    feature_cols
].copy()

y = model_data[
    target_col
].copy()


# ============================================================
# 5. Basic QC
# ============================================================

print("=" * 75)
print("SENSITIVITY ANALYSIS DATA")
print("=" * 75)

print(
    "Sensitivity analysis period: 2013–2022"
)

print(
    "Year range:",
    model_data["Year"].min(),
    "-",
    model_data["Year"].max()
)

print(
    "Number of observations:",
    model_data.shape[0]
)

print(
    "\nObservations by year:"
)

print(
    model_data[
        "Year"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nX shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "\nMissing values in predictors:"
)

print(
    X.isna().sum()
)

print(
    "\nInfinite values in predictors:"
)

print(
    np.isinf(
        X
    ).sum()
)


# ============================================================
# 6. Train-test split
#
# Same random_state as main analysis
# ============================================================

X_train, X_valid, y_train, y_valid = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE
    )
)


# ============================================================
# 7. FINAL LightGBM model
#
# Same parameters used in the main analysis
# ============================================================

final_model = lgb.LGBMRegressor(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=-1,

    num_leaves=50,

    subsample=0.8,

    colsample_bytree=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbose=-1
)


# ============================================================
# 8. Fit on training data and evaluate
# ============================================================

final_model.fit(
    X_train,
    y_train
)

y_pred = final_model.predict(
    X_valid
)


rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        y_pred
    )
)

r2 = r2_score(
    y_valid,
    y_pred
)


print("\n" + "=" * 75)
print("LIGHTGBM SENSITIVITY MODEL PERFORMANCE")
print("=" * 75)

print(
    f"Validation RMSE: {rmse:.4f}"
)

print(
    f"Validation R²: {r2:.4f}"
)


model_perf = pd.DataFrame({

    "Metric": [
        "Validation RMSE",
        "Validation R2"
    ],

    "Value": [
        rmse,
        r2
    ]
})


model_perf.to_csv(
    performance_out,
    index=False
)


# ============================================================
# 9. Refit LightGBM on ALL 2013–2022 data
#    for SHAP interpretation
# ============================================================

final_model.fit(
    X,
    y
)


# ============================================================
# 10. Compute SHAP values
# ============================================================

explainer = shap.TreeExplainer(
    final_model
)

shap_values = explainer.shap_values(
    X
)


# Ensure expected matrix format
shap_values = np.asarray(
    shap_values
)


if shap_values.ndim != 2:

    raise ValueError(
        f"Unexpected SHAP dimensions: {shap_values.shape}"
    )


shap_df = pd.DataFrame(
    shap_values,
    columns=feature_cols,
    index=X.index
)


# ============================================================
# 11. Export SHAP matrix
# ============================================================

shap_matrix_export = pd.concat(
    [
        model_data[
            [
                "GEOID",
                "Year"
            ]
        ].reset_index(
            drop=True
        ),

        shap_df.reset_index(
            drop=True
        )
    ],
    axis=1
)


# Rename SHAP value columns explicitly
shap_matrix_export = (
    shap_matrix_export
    .rename(
        columns={
            col:
                f"SHAP_{col}"
            for col in feature_cols
        }
    )
)


shap_matrix_export.to_csv(
    shap_matrix_out,
    index=False
)


# ============================================================
# 12. Publication-friendly feature names
# ============================================================

feature_name_map = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}


X_renamed = X.rename(
    columns=feature_name_map
)

shap_df_renamed = shap_df.rename(
    columns=feature_name_map
)


display_features = [
    feature_name_map[
        feature
    ]
    for feature in feature_cols
]


# ============================================================
# 13. Numeric SHAP summary
# ============================================================

results = []


for raw_feature in feature_cols:

    display_feature = (
        feature_name_map[
            raw_feature
        ]
    )


    mean_abs_shap = (
        np.abs(
            shap_df[
                raw_feature
            ]
        )
        .mean()
    )


    mean_shap = (
        shap_df[
            raw_feature
        ]
        .mean()
    )


    corr = np.corrcoef(
        X[
            raw_feature
        ],
        shap_df[
            raw_feature
        ]
    )[0, 1]


    if pd.isna(
        corr
    ):

        direction = "Unclear"

    elif np.isclose(
        corr,
        0,
        atol=1e-6
    ):

        direction = "Neutral"

    elif corr > 0:

        direction = "Positive"

    else:

        direction = "Negative"


    results.append({

        "Feature":
            display_feature,

        "MeanAbsSHAP":
            mean_abs_shap,

        "MeanSHAP":
            mean_shap,

        "Feature_SHAP_Correlation":
            corr,

        "Overall_Direction":
            direction
    })


shap_summary = pd.DataFrame(
    results
)


shap_summary = (
    shap_summary
    .sort_values(
        by="MeanAbsSHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


shap_summary[
    "MeanAbsSHAP"
] = (
    shap_summary[
        "MeanAbsSHAP"
    ]
    .round(4)
)


shap_summary[
    "MeanSHAP"
] = (
    shap_summary[
        "MeanSHAP"
    ]
    .round(4)
)


shap_summary[
    "Feature_SHAP_Correlation"
] = (
    shap_summary[
        "Feature_SHAP_Correlation"
    ]
    .round(4)
)


print("\n" + "=" * 75)
print("SHAP GLOBAL IMPORTANCE + DIRECTION")
print("2013–2022 SENSITIVITY ANALYSIS")
print("=" * 75)

print(
    shap_summary.to_string(
        index=False
    )
)


shap_summary.to_csv(
    shap_summary_out,
    index=False
)


# ============================================================
# 14. Figure S8
#     Combined SHAP summary + importance
# ============================================================

feature_order = (
    shap_summary[
        "Feature"
    ]
    .tolist()
)

n_feat = len(
    feature_order
)

y_positions = (
    np.arange(
        n_feat
    )[::-1]
)


plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        11.5
})


fig, (
    ax1,
    ax2
) = plt.subplots(

    1,
    2,

    figsize=(
        16,
        6.0
    ),

    gridspec_kw={
        "width_ratios": [
            2.5,
            1.5
        ],
        "wspace":
            0.10
    }
)


# ============================================================
# 14a. Panel A
# SHAP summary-like scatter plot
# ============================================================

sc = None


for i, feature in enumerate(
    feature_order
):

    y0 = y_positions[
        i
    ]


    shap_vals = (
        shap_df_renamed[
            feature
        ]
        .values
    )


    feat_vals = (
        X_renamed[
            feature
        ]
        .values
    )


    valid = ~(
        np.isnan(
            shap_vals
        )
        |
        np.isnan(
            feat_vals
        )
    )


    shap_vals = (
        shap_vals[
            valid
        ]
    )


    feat_vals = (
        feat_vals[
            valid
        ]
    )


    # Clip feature values for stable coloring
    vmin = np.nanpercentile(
        feat_vals,
        5
    )

    vmax = np.nanpercentile(
        feat_vals,
        95
    )


    feat_vals_clip = np.clip(
        feat_vals,
        vmin,
        vmax
    )


    if vmax > vmin:

        feat_color = (
            feat_vals_clip
            - vmin
        ) / (
            vmax
            - vmin
        )

    else:

        feat_color = np.full_like(
            feat_vals_clip,
            0.5,
            dtype=float
        )


    # Fixed jitter = reproducible figure
    rng = np.random.default_rng(
        RANDOM_STATE
        + i
    )

    jitter = rng.normal(
        0,
        0.08,
        size=len(
            shap_vals
        )
    )


    sc = ax1.scatter(

        shap_vals,

        np.full_like(
            shap_vals,
            y0,
            dtype=float
        )
        + jitter,

        c=
            feat_color,

        cmap=
            "coolwarm",

        vmin=
            0,

        vmax=
            1,

        s=
            12,

        alpha=
            0.75,

        edgecolors=
            "none"
    )


ax1.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)


ax1.set_yticks(
    y_positions
)

ax1.set_yticklabels(
    feature_order,
    fontsize=11
)

ax1.set_xlabel(
    "SHAP value (effect on predicted autism prevalence)",
    fontsize=13
)

ax1.set_ylabel(
    ""
)

ax1.set_title(
    "(a) SHAP summary",
    fontsize=14
)

ax1.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax1.tick_params(
    axis="x",
    labelsize=11
)


cbar = plt.colorbar(
    sc,
    ax=ax1,
    pad=0.01
)

cbar.set_label(
    "Feature value",
    fontsize=11
)

ticks = np.linspace(
    0,
    1,
    6
)

cbar.set_ticks(
    ticks
)

cbar.set_ticklabels(
    [
        f"{t:.1f}"
        for t in ticks
    ]
)

cbar.ax.tick_params(
    labelsize=10
)


ax1.spines[
    "top"
].set_visible(
    False
)

ax1.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 14b. Panel B
# Mean absolute SHAP feature importance
# ============================================================

bar_data = (
    shap_summary
    .set_index(
        "Feature"
    )
    .loc[
        feature_order
    ]
)

bar_values = (
    bar_data[
        "MeanAbsSHAP"
    ]
    .values
)


bar_colors = (
    plt.cm.viridis(
        np.linspace(
            0.35,
            0.92,
            n_feat
        )
    )
)


ax2.barh(

    y_positions,

    bar_values,

    height=
        0.52,

    color=
        bar_colors
)


xmax = (
    bar_values.max()
)

right_margin = (
    xmax
    * 0.20
)

text_offset = (
    xmax
    * 0.02
)


ax2.set_xlim(
    0,
    xmax
    + right_margin
)


for y_pos, value in zip(
    y_positions,
    bar_values
):

    ax2.text(

        value
        + text_offset,

        y_pos,

        f"{value:.2f}",

        va=
            "center",

        ha=
            "left",

        fontsize=
            10.5,

        clip_on=
            True
    )


ax2.set_yticks(
    y_positions
)

ax2.set_yticklabels(
    []
)

ax2.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=13
)

ax2.set_ylabel(
    ""
)

ax2.set_title(
    "(b) SHAP feature importance",
    fontsize=14
)

ax2.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax2.tick_params(
    axis="x",
    labelsize=11
)


ax2.spines[
    "top"
].set_visible(
    False
)

ax2.spines[
    "right"
].set_visible(
    False
)


# Keep both panels aligned
ax1.set_ylim(
    -0.45,
    n_feat
    - 0.55
)

ax2.set_ylim(
    -0.45,
    n_feat
    - 0.55
)


plt.tight_layout()


# ============================================================
# 15. Save Figure S8
# ============================================================

plt.savefig(

    figure_s8_out,

    dpi=
        300,

    format=
        "tiff",

    bbox_inches=
        "tight"
)

plt.show()


# ============================================================
# 16. Optional standard SHAP summary plot
# ============================================================

plt.figure()

shap.summary_plot(

    shap_values,

    X_renamed,

    show=False
)

plt.tight_layout()

plt.savefig(

    standard_summary_out,

    dpi=300,

    format="tiff",

    bbox_inches="tight"
)

plt.show()


# ============================================================
# 17. Optional standard SHAP bar plot
# ============================================================

plt.figure()

shap.summary_plot(

    shap_values,

    X_renamed,

    plot_type=
        "bar",

    show=
        False
)

plt.tight_layout()

plt.savefig(

    standard_bar_out,

    dpi=300,

    format="tiff",

    bbox_inches="tight"
)

plt.show()


# ============================================================
# 18. Final output summary
# ============================================================

print("\n" + "=" * 75)
print("FILES SAVED")
print("=" * 75)

print("\nModel performance:")
print(
    performance_out
)

print("\nSHAP summary:")
print(
    shap_summary_out
)

print("\nSHAP matrix:")
print(
    shap_matrix_out
)

print("\nFigure S8:")
print(
    figure_s8_out
)

print("\nStandard SHAP summary:")
print(
    standard_summary_out
)

print("\nStandard SHAP bar:")
print(
    standard_bar_out
)

print("\nSensitivity analysis completed.")

## 23. Table S10: SHAP rank stability

Compare main-analysis and 2013–2022 SHAP rankings using Spearman and Kendall rank correlations.

In [ ]:
import os
import pandas as pd
from scipy.stats import spearmanr, kendalltau

# ============================================================
# 1. Paths
# ============================================================

tables_dir = str(TABLES_DIR)

# Main analysis: 2010–2022
main_file = os.path.join(
    tables_dir,
    "Table_SHAP_global_importance_direction_final.csv"
)

# Sensitivity analysis: 2013–2022
sens_file = os.path.join(
    tables_dir,
    "Table_SHAP_global_importance_direction_2013_2022_sensitivity.csv"
)

# Final Table S10
output_file = os.path.join(
    tables_dir,
    "Table_S10_SHAP_rank_comparison_main_vs_2013_2022_sensitivity.csv"
)


# ============================================================
# 2. Read SHAP results
# ============================================================

main = pd.read_csv(main_file)
sens = pd.read_csv(sens_file)

main.columns = main.columns.str.strip()
sens.columns = sens.columns.str.strip()


# ============================================================
# 3. Check required columns
# ============================================================

required_cols = [
    "Feature",
    "MeanAbsSHAP",
    "Overall_Direction"
]

for name, df in [
    ("Main analysis", main),
    ("Sensitivity analysis", sens)
]:
    missing = [
        col
        for col in required_cols
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} is missing columns: {missing}"
        )


# ============================================================
# 4. Add ranks
# ============================================================

main = (
    main
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

sens = (
    sens
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

main["Main_Rank_2010_2022"] = (
    main.index + 1
)

sens["Sensitivity_Rank_2013_2022"] = (
    sens.index + 1
)


# ============================================================
# 5. Keep key columns
# ============================================================

main_keep = (
    main[
        [
            "Feature",
            "Main_Rank_2010_2022",
            "MeanAbsSHAP",
            "Overall_Direction"
        ]
    ]
    .rename(
        columns={
            "MeanAbsSHAP":
                "Main_MeanAbsSHAP_2010_2022",

            "Overall_Direction":
                "Main_Direction_2010_2022"
        }
    )
)


sens_keep = (
    sens[
        [
            "Feature",
            "Sensitivity_Rank_2013_2022",
            "MeanAbsSHAP",
            "Overall_Direction"
        ]
    ]
    .rename(
        columns={
            "MeanAbsSHAP":
                "Sensitivity_MeanAbsSHAP_2013_2022",

            "Overall_Direction":
                "Sensitivity_Direction_2013_2022"
        }
    )
)


# ============================================================
# 6. Merge main and sensitivity results
# ============================================================

comparison = pd.merge(
    main_keep,
    sens_keep,
    on="Feature",
    how="outer",
    validate="one_to_one"
)


# ============================================================
# 7. Calculate rank change
#
# Positive Rank_Change:
# sensitivity rank number became larger = moved DOWN
#
# Negative Rank_Change:
# sensitivity rank number became smaller = moved UP
# ============================================================

comparison["Rank_Change"] = (
    comparison[
        "Sensitivity_Rank_2013_2022"
    ]
    -
    comparison[
        "Main_Rank_2010_2022"
    ]
)


def rank_change_label(x):

    if pd.isna(x):

        return ""

    elif x == 0:

        return "No change"

    elif x > 0:

        return f"Down {int(x)}"

    else:

        return f"Up {abs(int(x))}"


comparison[
    "Rank_Change_Label"
] = (
    comparison[
        "Rank_Change"
    ]
    .apply(
        rank_change_label
    )
)


# ============================================================
# 8. Sort by MAIN analysis ranking
# ============================================================

comparison = (
    comparison
    .sort_values(
        "Main_Rank_2010_2022"
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. Round SHAP values
# ============================================================

comparison[
    "Main_MeanAbsSHAP_2010_2022"
] = (
    comparison[
        "Main_MeanAbsSHAP_2010_2022"
    ]
    .round(4)
)


comparison[
    "Sensitivity_MeanAbsSHAP_2013_2022"
] = (
    comparison[
        "Sensitivity_MeanAbsSHAP_2013_2022"
    ]
    .round(4)
)


# ============================================================
# 10. Rank correlations
# ============================================================

valid = (
    comparison
    .dropna(
        subset=[
            "Main_Rank_2010_2022",
            "Sensitivity_Rank_2013_2022"
        ]
    )
    .copy()
)


spearman_rho, spearman_p = spearmanr(
    valid[
        "Main_Rank_2010_2022"
    ],
    valid[
        "Sensitivity_Rank_2013_2022"
    ]
)


kendall_tau, kendall_p = kendalltau(
    valid[
        "Main_Rank_2010_2022"
    ],
    valid[
        "Sensitivity_Rank_2013_2022"
    ]
)


# ============================================================
# 11. Add summary rows
# ============================================================

summary = pd.DataFrame({

    "Feature": [
        "Spearman rank correlation",
        "Kendall tau"
    ],

    "Main_Rank_2010_2022": [
        "",
        ""
    ],

    "Main_MeanAbsSHAP_2010_2022": [
        "",
        ""
    ],

    "Main_Direction_2010_2022": [
        "",
        ""
    ],

    "Sensitivity_Rank_2013_2022": [
        "",
        ""
    ],

    "Sensitivity_MeanAbsSHAP_2013_2022": [
        "",
        ""
    ],

    "Sensitivity_Direction_2013_2022": [
        "",
        ""
    ],

    "Rank_Change": [
        "",
        ""
    ],

    "Rank_Change_Label": [

        f"rho = {spearman_rho:.3f}, "
        f"p = {spearman_p:.4g}",

        f"tau = {kendall_tau:.3f}, "
        f"p = {kendall_p:.4g}"
    ]
})


comparison_out = pd.concat(
    [
        comparison,
        summary
    ],
    ignore_index=True
)


# ============================================================
# 12. Save full comparison output
# ============================================================

comparison_out.to_csv(
    output_file,
    index=False
)


# ============================================================
# 13. Print
# ============================================================

print("=" * 80)
print("SHAP RANK COMPARISON")
print("MAIN 2010–2022 vs SENSITIVITY 2013–2022")
print("=" * 80)

print(
    comparison.to_string(
        index=False
    )
)


print("\nSpearman rank correlation:")

print(
    f"rho = {spearman_rho:.3f}, "
    f"p = {spearman_p:.4g}"
)


print("\nKendall tau:")

print(
    f"tau = {kendall_tau:.3f}, "
    f"p = {kendall_p:.4g}"
)


print("\nSaved to:")

print(
    output_file
)

## 24. Figure S9: sensitivity analysis including year

Add calendar year as a predictor and assess whether the leading non-temporal predictors remain important.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# Figure S9
# Sensitivity analysis: adding Year as a predictor
# Final model: LightGBM
#
# Layout is matched to Figure S8:
#   Panel A : Panel B = 2.5 : 1.5
# ============================================================


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# ============================================================
# 2. Paths
# ============================================================

model_data_path = PROCESSED_DIR / 'autism_model_final.csv'

tables_dir = TABLES_DIR

figures_dir = FIGURES_DIR

tables_dir.mkdir(
    parents=True,
    exist_ok=True
)

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. Outcome and predictors
# ============================================================

TARGET = "Autism_prev1000"

BASE_FEATURES = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]

# Sensitivity analysis:
# original 13 predictors + Year
FEATURES = (
    BASE_FEATURES
    + ["Year"]
)


# ============================================================
# 4. Final LightGBM parameters
#
# Same fixed parameters as the main analysis
# ============================================================

FINAL_LGBM_PARAMS = dict(

    n_estimators=500,

    learning_rate=0.1,

    max_depth=-1,

    num_leaves=50,

    subsample=0.8,

    colsample_bytree=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,

    verbose=-1
)


# ============================================================
# 5. Publication-friendly variable names
# ============================================================

feature_name_map = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)",

    "Year":
        "Year"
}


# ============================================================
# 6. Load data
# ============================================================

df = pd.read_csv(
    model_data_path
)

df.columns = (
    df.columns
    .str.strip()
)


required_cols = (
    [TARGET]
    + FEATURES
)


missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]


if missing_cols:

    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# Convert modeling fields to numeric
data = (
    df[
        required_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .dropna()
    .copy()
)


X = data[
    FEATURES
].copy()

y = data[
    TARGET
].copy()


print("=" * 75)
print("FIGURE S9 DATA")
print("=" * 75)

print(
    "Data shape:",
    X.shape
)

print(
    "Predictors:",
    len(FEATURES)
)


# ============================================================
# 7. Train/test split
#
# Fixed random state for reproducibility
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE
    )
)


# ============================================================
# 8. Fit final LightGBM
# ============================================================

model = lgb.LGBMRegressor(
    **FINAL_LGBM_PARAMS
)


model.fit(
    X_train,
    y_train
)


# ============================================================
# 9. Test-set performance
# ============================================================

y_pred = model.predict(
    X_test
)


r2 = r2_score(
    y_test,
    y_pred
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)


mae = mean_absolute_error(
    y_test,
    y_pred
)


performance = pd.DataFrame({

    "Metric": [
        "R2",
        "RMSE",
        "MAE"
    ],

    "Value": [
        r2,
        rmse,
        mae
    ]
})


performance_path = (
    tables_dir
    / "Table_S9_year_predictor_model_performance.csv"
)


performance.to_csv(
    performance_path,
    index=False
)


print("\nModel performance:")

print(
    performance
)


# ============================================================
# 10. SHAP values
#
# Same test data used for evaluation
# ============================================================

explainer = shap.TreeExplainer(
    model
)


shap_values = explainer.shap_values(
    X_test
)


if isinstance(
    shap_values,
    list
):

    shap_values = (
        shap_values[0]
    )


shap_values = np.asarray(
    shap_values
)


# ============================================================
# 11. SHAP importance table
# ============================================================

importance = pd.DataFrame({

    "Feature":
        FEATURES,

    "Display":
        [
            feature_name_map[
                f
            ]
            for f in FEATURES
        ],

    "MeanAbsSHAP":
        np.abs(
            shap_values
        ).mean(
            axis=0
        ),

    "MeanSHAP":
        shap_values.mean(
            axis=0
        )
})


importance = (
    importance
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


importance.insert(

    0,

    "Rank",

    range(
        1,
        len(
            importance
        ) + 1
    )
)


importance_path = (
    tables_dir
    / "Table_SHAP_importance_with_year_sensitivity.csv"
)


importance.to_csv(
    importance_path,
    index=False
)


print("\nSHAP importance:")

print(
    importance.to_string(
        index=False
    )
)


# ============================================================
# 12. Prepare plotting data
# ============================================================

feature_order = (
    importance[
        "Feature"
    ]
    .tolist()
)


display_order = (
    importance[
        "Display"
    ]
    .tolist()
)


n_feat = len(
    feature_order
)


y_positions = (
    np.arange(
        n_feat
    )[::-1]
)


shap_df = pd.DataFrame(

    shap_values,

    columns=
        FEATURES,

    index=
        X_test.index
)


# ============================================================
# 13. Figure style
#
# MATCH Figure S8
# ============================================================

plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        11.5,

    "axes.titlesize":
        14,

    "axes.labelsize":
        13,

    "xtick.labelsize":
        11,

    "ytick.labelsize":
        11
})


# ============================================================
# IMPORTANT:
#
# Figure S8 layout:
#
# figsize = (16, 6.0)
# Panel A : Panel B = 2.5 : 1.5
#
# ============================================================

fig, (
    ax1,
    ax2
) = plt.subplots(

    1,
    2,

    figsize=(
        16,
        6.0
    ),

    gridspec_kw={
        "width_ratios": [
            2.5,
            1.5
        ],

        # Leave enough space between the two main panels.
        # Colorbar is attached to Panel A independently.
        "wspace":
            0.24
    }
)


# ============================================================
# 14. PANEL A
# SHAP summary
# ============================================================

scatter_handle = None


for i, feature in enumerate(
    feature_order
):

    y0 = (
        y_positions[i]
    )


    shap_vals = (
        shap_df[
            feature
        ]
        .to_numpy()
    )


    feat_vals = (
        X_test[
            feature
        ]
        .to_numpy()
    )


    valid = ~(
        np.isnan(
            shap_vals
        )
        |
        np.isnan(
            feat_vals
        )
    )


    shap_vals = (
        shap_vals[
            valid
        ]
    )


    feat_vals = (
        feat_vals[
            valid
        ]
    )


    # ----------------------------------------
    # Feature-value coloring
    # ----------------------------------------

    vmin = np.nanpercentile(
        feat_vals,
        5
    )

    vmax = np.nanpercentile(
        feat_vals,
        95
    )


    feat_vals_clip = np.clip(
        feat_vals,
        vmin,
        vmax
    )


    if vmax > vmin:

        feat_color = (
            feat_vals_clip
            - vmin
        ) / (
            vmax
            - vmin
        )

    else:

        feat_color = np.full(
            len(
                feat_vals_clip
            ),
            0.5,
            dtype=float
        )


    # ----------------------------------------
    # Fixed jitter
    # ----------------------------------------

    rng = np.random.default_rng(
        RANDOM_STATE
        + i
    )


    jitter = rng.normal(
        0,
        0.08,
        size=len(
            shap_vals
        )
    )


    scatter_handle = ax1.scatter(

        shap_vals,

        np.full_like(
            shap_vals,
            y0,
            dtype=float
        )
        + jitter,

        c=
            feat_color,

        cmap=
            "coolwarm",

        vmin=
            0,

        vmax=
            1,

        s=
            12,

        alpha=
            0.75,

        edgecolors=
            "none",

        rasterized=
            True
    )


# Zero reference line
ax1.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)


# Y labels
ax1.set_yticks(
    y_positions
)


ax1.set_yticklabels(
    display_order,
    fontsize=11
)


# Axis title
ax1.set_xlabel(
    "SHAP value (effect on predicted autism prevalence)",
    fontsize=13
)


ax1.set_ylabel(
    ""
)


# Panel title
ax1.set_title(
    "(a) SHAP summary",
    fontsize=14,
    pad=8
)


# Grid
ax1.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.30
)


ax1.set_axisbelow(
    True
)


# Remove top/right borders
ax1.spines[
    "top"
].set_visible(
    False
)


ax1.spines[
    "right"
].set_visible(
    False
)


# Exact vertical alignment
ax1.set_ylim(
    -0.45,
    n_feat
    - 0.55
)


# ============================================================
# 15. PANEL A COLORBAR
#
# IMPORTANT:
# make_axes_locatable makes the colorbar:
#
#   - exactly the same height as Panel A
#   - independent of Panel B
#   - no overlap with Panel B
# ============================================================

divider = make_axes_locatable(
    ax1
)


cax = divider.append_axes(

    "right",

    # Width of colorbar relative to Panel A
    size="2.8%",

    # Physical gap between Panel A and colorbar
    pad=0.10
)


cbar = fig.colorbar(

    scatter_handle,

    cax=cax
)


cbar.set_label(
    "Feature value",
    fontsize=11
)


cbar.set_ticks(
    np.linspace(
        0,
        1,
        6
    )
)


cbar.set_ticklabels(
    [
        "0.0",
        "0.2",
        "0.4",
        "0.6",
        "0.8",
        "1.0"
    ]
)


cbar.ax.tick_params(
    labelsize=10
)


# ============================================================
# 16. PANEL B
# SHAP feature importance
# ============================================================

bar_values = (
    importance[
        "MeanAbsSHAP"
    ]
    .to_numpy()
)


bar_colors = (
    plt.cm.viridis(
        np.linspace(
            0.35,
            0.92,
            n_feat
        )
    )
)


ax2.barh(

    y_positions,

    bar_values,

    height=
        0.52,

    color=
        bar_colors
)


# ============================================================
# Add numeric SHAP values
# ============================================================

xmax = (
    bar_values.max()
)


right_margin = (
    xmax
    * 0.20
)


text_offset = (
    xmax
    * 0.02
)


ax2.set_xlim(
    0,
    xmax
    + right_margin
)


for y_pos, value in zip(

    y_positions,

    bar_values
):

    ax2.text(

        value
        + text_offset,

        y_pos,

        f"{value:.2f}",

        va=
            "center",

        ha=
            "left",

        fontsize=
            10.5,

        clip_on=
            False
    )


# ============================================================
# Panel B Y axis
#
# DO NOT repeat predictor labels.
# The rows align directly with Panel A.
# ============================================================

ax2.set_yticks(
    y_positions
)


ax2.set_yticklabels(
    []
)


ax2.tick_params(
    axis="y",
    length=0
)


# ============================================================
# Panel B labels
# ============================================================

ax2.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=13
)


ax2.set_ylabel(
    ""
)


ax2.set_title(
    "(b) SHAP feature importance",
    fontsize=14,
    pad=8
)


# Grid
ax2.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.30
)


ax2.set_axisbelow(
    True
)


# Remove top/right borders
ax2.spines[
    "top"
].set_visible(
    False
)


ax2.spines[
    "right"
].set_visible(
    False
)


# Exact vertical alignment with Panel A
ax2.set_ylim(
    -0.45,
    n_feat
    - 0.55
)


# ============================================================
# 17. FINAL LAYOUT
#
# Do NOT use tight_layout after make_axes_locatable,
# because it can disturb the colorbar/panel spacing.
# ============================================================

fig.subplots_adjust(

    # Enough room for long predictor labels
    left=
        0.305,

    # Enough room for numeric labels in Panel B
    right=
        0.965,

    bottom=
        0.13,

    top=
        0.92,

    # Main-panel spacing
    wspace=
        0.24
)


# ============================================================
# 18. Save Figure S9
# ============================================================

figure_path = (
    figures_dir
    / "Figure_S9_year_as_predictor_sensitivity.tiff"
)


plt.savefig(

    figure_path,

    dpi=
        300,

    format=
        "tiff",

    bbox_inches=
        "tight"
)


plt.show()


# ============================================================
# 19. Final outputs
# ============================================================

print("\n" + "=" * 75)
print("FILES SAVED")
print("=" * 75)


print("\nModel performance:")
print(
    performance_path
)


print("\nSHAP importance:")
print(
    importance_path
)


print("\nFigure S9:")
print(
    figure_path
)

## 25. Figure S10: lagged environmental predictors

Evaluate 1-year and 3-year lag specifications for PM2.5, climate, precipitation, and forest cover.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# Figure S10
# SHAP-based sensitivity analysis using lagged
# environmental predictors
#
# Panel (a): 1-year lag
# Panel (b): 3-year lag
#
# Final model: LightGBM
# Plot style matched to revised Figures S8/S9
# ============================================================


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# ============================================================
# 2. Paths
# ============================================================

processed_dir = PROCESSED_DIR


model_data_path = (
    processed_dir
    / "autism_model_final.csv"
)


observed_data_path = (
    processed_dir
    / "autism_observed_data.csv"
)


tables_dir = TABLES_DIR


figures_dir = FIGURES_DIR


tables_dir.mkdir(
    parents=True,
    exist_ok=True
)

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. Outcome / identifiers / predictors
# ============================================================

TARGET = "Autism_prev1000"

ID_COL = "GEOID"

YEAR_COL = "Year"


BASE_FEATURES = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]


# Only these four predictors are lagged
ENV_FEATURES = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Forest_percent"
]


# ============================================================
# 4. Final LightGBM parameters
#    Same as primary analysis
# ============================================================

FINAL_LGBM_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)


# ============================================================
# 5. SHORT publication labels for Figure S10
#
# Units intentionally omitted to preserve plotting space.
# Lag annotations are added ONLY to lagged environmental
# predictors.
# ============================================================

SHORT_NAMES = {

    "PM25":
        r"PM$_{2.5}$ concentration",

    "tmax_c":
        "Maximum temperature",

    "ppt_mm":
        "Annual precipitation",

    "Below_poverty_percent":
        "Families below poverty",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher",

    "under5_percent":
        "Population under 5 years",

    "age5to14_percent":
        "Population aged 5–14 years",

    "age15to19_percent":
        "Population aged 15–19 years",

    "median_age":
        "Median age",

    "Pop_density":
        "Population density",

    "hispanic_percent":
        "Hispanic or Latino population",

    "asian_percent":
        "Asian population",

    "Forest_percent":
        "Forest cover"
}


# ============================================================
# 6. Helper functions
# ============================================================

def clean_geoid(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


def calc_rmse(
    y_true,
    y_pred
):

    return float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )
    )


def get_shap_values(
    model,
    X
):

    explainer = shap.TreeExplainer(
        model
    )

    shap_values = explainer.shap_values(
        X
    )

    if isinstance(
        shap_values,
        list
    ):
        shap_values = shap_values[0]

    return np.asarray(
        shap_values
    )


# ============================================================
# 7. Load final modeling dataset
# ============================================================

main_df = pd.read_csv(
    model_data_path,
    dtype={
        ID_COL: str
    }
)


main_df.columns = (
    main_df.columns
    .str.strip()
)


main_df[
    ID_COL
] = clean_geoid(
    main_df[
        ID_COL
    ]
)


main_df[
    YEAR_COL
] = pd.to_numeric(
    main_df[
        YEAR_COL
    ],
    errors="coerce"
)


# ============================================================
# 8. Load observed/non-imputed annual data
# ============================================================

observed_df = pd.read_csv(
    observed_data_path,
    dtype={
        ID_COL: str
    }
)


observed_df.columns = (
    observed_df.columns
    .str.strip()
)


observed_df[
    ID_COL
] = clean_geoid(
    observed_df[
        ID_COL
    ]
)


observed_df[
    YEAR_COL
] = pd.to_numeric(
    observed_df[
        YEAR_COL
    ],
    errors="coerce"
)


# ============================================================
# 9. Validate required columns
# ============================================================

main_required = (
    [
        ID_COL,
        YEAR_COL,
        TARGET
    ]
    + BASE_FEATURES
)


observed_required = (
    [
        ID_COL,
        YEAR_COL
    ]
    + ENV_FEATURES
)


missing_main = [
    c
    for c in main_required
    if c not in main_df.columns
]


missing_observed = [
    c
    for c in observed_required
    if c not in observed_df.columns
]


if missing_main:

    raise ValueError(
        "Missing columns in autism_model_final.csv:\n"
        + str(missing_main)
    )


if missing_observed:

    raise ValueError(
        "Missing columns in autism_observed_data.csv:\n"
        + str(missing_observed)
    )


# ============================================================
# 10. Current-year analysis rows
# ============================================================

current_data = (
    main_df[
        main_required
    ]
    .copy()
)


# ============================================================
# 11. Build exact-year lag dataset
#
# Example:
# current 2018 + 1-year lag -> 2017 environmental exposure
# current 2018 + 3-year lag -> 2015 environmental exposure
#
# Exact GEOID + Year matching avoids erroneous lagging when
# intermediate district-years are absent.
# ============================================================

def create_exact_lag_dataset(
    lag_years
):

    current = (
        current_data
        .copy()
    )


    lookup = (
        observed_df[
            [
                ID_COL,
                YEAR_COL
            ]
            + ENV_FEATURES
        ]
        .copy()
    )


    # Ensure unique district-year environmental records
    duplicate_n = (
        lookup
        .duplicated(
            subset=[
                ID_COL,
                YEAR_COL
            ]
        )
        .sum()
    )


    if duplicate_n > 0:

        raise ValueError(
            f"{duplicate_n} duplicate GEOID-Year records "
            "found in observed environmental data."
        )


    # Shift source year forward
    lookup[
        YEAR_COL
    ] = (
        lookup[
            YEAR_COL
        ]
        + lag_years
    )


    # Rename lagged environmental variables
    lag_rename = {

        feature:
            f"{feature}_lag{lag_years}"

        for feature
        in ENV_FEATURES
    }


    lookup = lookup.rename(
        columns=lag_rename
    )


    # Remove current-year environmental predictors
    current = current.drop(
        columns=ENV_FEATURES
    )


    # Merge exact GEOID-Year lag exposure
    lagged = current.merge(

        lookup,

        on=[
            ID_COL,
            YEAR_COL
        ],

        how="left",

        validate="one_to_one"
    )


    # Build predictor list
    features = []


    for feature in BASE_FEATURES:

        if feature in ENV_FEATURES:

            features.append(
                f"{feature}_lag{lag_years}"
            )

        else:

            features.append(
                feature
            )


    # Numeric conversion
    numeric_cols = (
        [
            YEAR_COL,
            TARGET
        ]
        + features
    )


    lagged[
        numeric_cols
    ] = (
        lagged[
            numeric_cols
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )


    # Complete cases
    lagged = (
        lagged
        .dropna(
            subset=[
                TARGET
            ]
            + features
        )
        .copy()
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # Short display labels
    # ========================================================

    display_map = {}


    for feature in BASE_FEATURES:

        if feature in ENV_FEATURES:

            lag_feature = (
                f"{feature}_lag{lag_years}"
            )


            if lag_years == 1:

                display_map[
                    lag_feature
                ] = (
                    SHORT_NAMES[
                        feature
                    ]
                    + " (1-year lag)"
                )

            else:

                display_map[
                    lag_feature
                ] = (
                    SHORT_NAMES[
                        feature
                    ]
                    + f" ({lag_years}-year lag)"
                )


        else:

            display_map[
                feature
            ] = (
                SHORT_NAMES[
                    feature
                ]
            )


    return (
        lagged,
        features,
        display_map
    )


# ============================================================
# 12. Run one lag sensitivity analysis
# ============================================================

def run_lag_analysis(
    lag_years
):

    (
        data,
        features,
        display_map
    ) = create_exact_lag_dataset(
        lag_years
    )


    X = (
        data[
            features
        ]
        .copy()
    )


    y = (
        data[
            TARGET
        ]
        .copy()
    )


    print("\n" + "=" * 75)

    print(
        f"{lag_years}-YEAR LAG ANALYSIS"
    )

    print("=" * 75)


    print(
        "Number of observations:",
        len(data)
    )


    print(
        "Year range:",
        int(
            data[
                YEAR_COL
            ].min()
        ),
        "-",
        int(
            data[
                YEAR_COL
            ].max()
        )
    )


    # ========================================================
    # Train/test performance
    # ========================================================

    (
        X_train,
        X_test,
        y_train,
        y_test
    ) = train_test_split(

        X,
        y,

        test_size=
            0.20,

        random_state=
            RANDOM_STATE
    )


    model_eval = lgb.LGBMRegressor(
        **FINAL_LGBM_PARAMS
    )


    model_eval.fit(
        X_train,
        y_train
    )


    y_pred = model_eval.predict(
        X_test
    )


    performance = {

        "Lag_years":
            lag_years,

        "N":
            len(data),

        "Year_min":
            int(
                data[
                    YEAR_COL
                ].min()
            ),

        "Year_max":
            int(
                data[
                    YEAR_COL
                ].max()
            ),

        "R2":
            r2_score(
                y_test,
                y_pred
            ),

        "RMSE":
            calc_rmse(
                y_test,
                y_pred
            ),

        "MAE":
            mean_absolute_error(
                y_test,
                y_pred
            )
    }


    print("\nPerformance:")

    print(
        performance
    )


    # ========================================================
    # Full-data model for SHAP interpretation
    # ========================================================

    model_full = lgb.LGBMRegressor(
        **FINAL_LGBM_PARAMS
    )


    model_full.fit(
        X,
        y
    )


    shap_values = get_shap_values(
        model_full,
        X
    )


    # ========================================================
    # Importance table
    # ========================================================

    importance = pd.DataFrame({

        "Feature":
            features,

        "Display":
            [
                display_map[
                    f
                ]
                for f in features
            ],

        "MeanAbsSHAP":
            np.abs(
                shap_values
            ).mean(
                axis=0
            ),

        "MeanSHAP":
            shap_values.mean(
                axis=0
            )
    })


    importance = (
        importance
        .sort_values(
            "MeanAbsSHAP",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    importance.insert(

        0,

        "Rank",

        range(
            1,
            len(
                importance
            ) + 1
        )
    )


    return {

        "lag_years":
            lag_years,

        "data":
            data,

        "X":
            X,

        "features":
            features,

        "display_map":
            display_map,

        "shap_values":
            shap_values,

        "importance":
            importance,

        "performance":
            performance
    }


# ============================================================
# 13. Run analyses
# ============================================================

lag1 = run_lag_analysis(
    1
)


lag3 = run_lag_analysis(
    3
)


# ============================================================
# 14. Save tables
# ============================================================

performance_path = (
    tables_dir
    / "Table_S10_lagged_predictor_model_performance.csv"
)


pd.DataFrame(
    [
        lag1[
            "performance"
        ],
        lag3[
            "performance"
        ]
    ]
).to_csv(
    performance_path,
    index=False
)


lag1_importance_path = (
    tables_dir
    / "Table_S10_SHAP_importance_lag1.csv"
)


lag3_importance_path = (
    tables_dir
    / "Table_S10_SHAP_importance_lag3.csv"
)


lag1[
    "importance"
].to_csv(
    lag1_importance_path,
    index=False
)


lag3[
    "importance"
].to_csv(
    lag3_importance_path,
    index=False
)


# Save exact eligible observations for reproducibility
lag1_data_path = (
    tables_dir
    / "Table_S10_eligible_records_lag1.csv"
)


lag3_data_path = (
    tables_dir
    / "Table_S10_eligible_records_lag3.csv"
)


lag1[
    "data"
].to_csv(
    lag1_data_path,
    index=False
)


lag3[
    "data"
].to_csv(
    lag3_data_path,
    index=False
)


# ============================================================
# 15. Plot style
#
# Match revised Figure S8 / Figure S9 style
# ============================================================

plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        11.5,

    "axes.titlesize":
        14,

    "axes.labelsize":
        13,

    "xtick.labelsize":
        11,

    "ytick.labelsize":
        11
})


# ============================================================
# 16. Draw SHAP summary panel
# ============================================================

def draw_shap_panel(
    ax,
    result,
    panel_title
):

    X = (
        result[
            "X"
        ]
    )


    shap_values = (
        result[
            "shap_values"
        ]
    )


    importance = (
        result[
            "importance"
        ]
    )


    display_map = (
        result[
            "display_map"
        ]
    )


    feature_order = (
        importance[
            "Feature"
        ]
        .tolist()
    )


    display_order = [
        display_map[
            feature
        ]
        for feature
        in feature_order
    ]


    n_features = len(
        feature_order
    )


    y_positions = (
        np.arange(
            n_features
        )[::-1]
    )


    shap_df = pd.DataFrame(
        shap_values,
        columns=X.columns,
        index=X.index
    )


    scatter_handle = None


    # ========================================================
    # Draw each predictor row
    # ========================================================

    for i, feature in enumerate(
        feature_order
    ):

        y0 = (
            y_positions[i]
        )


        shap_vals = (
            shap_df[
                feature
            ]
            .to_numpy()
        )


        feature_vals = (
            X[
                feature
            ]
            .to_numpy()
        )


        valid = ~(
            np.isnan(
                shap_vals
            )
            |
            np.isnan(
                feature_vals
            )
        )


        shap_vals = (
            shap_vals[
                valid
            ]
        )


        feature_vals = (
            feature_vals[
                valid
            ]
        )


        # ----------------------------------------
        # Feature-value normalization
        # ----------------------------------------

        low = np.nanpercentile(
            feature_vals,
            5
        )


        high = np.nanpercentile(
            feature_vals,
            95
        )


        feature_clip = np.clip(
            feature_vals,
            low,
            high
        )


        if high > low:

            feature_color = (
                feature_clip
                - low
            ) / (
                high
                - low
            )

        else:

            feature_color = np.full(
                len(
                    feature_clip
                ),
                0.5,
                dtype=float
            )


        # ----------------------------------------
        # Fixed jitter
        # ----------------------------------------

        rng = np.random.default_rng(

            RANDOM_STATE
            + i
            + result[
                "lag_years"
            ] * 100
        )


        jitter = rng.normal(
            0,
            0.08,
            size=len(
                shap_vals
            )
        )


        scatter_handle = ax.scatter(

            shap_vals,

            np.full_like(
                shap_vals,
                y0,
                dtype=float
            )
            + jitter,

            c=
                feature_color,

            cmap=
                "coolwarm",

            vmin=
                0,

            vmax=
                1,

            s=
                12,

            alpha=
                0.75,

            edgecolors=
                "none",

            rasterized=
                True
        )


    # ========================================================
    # Zero line
    # ========================================================

    ax.axvline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1
    )


    # ========================================================
    # Predictor labels
    # ========================================================

    ax.set_yticks(
        y_positions
    )


    ax.set_yticklabels(
        display_order,
        fontsize=10.8
    )


    # ========================================================
    # X axis
    # ========================================================

    ax.set_xlabel(
        "SHAP value (effect on predicted autism prevalence)",
        fontsize=12.5
    )


    ax.set_ylabel(
        ""
    )


    # ========================================================
    # Panel title
    # ========================================================

    ax.set_title(
        panel_title,
        fontsize=14,
        pad=8
    )


    # ========================================================
    # Grid / frame
    # ========================================================

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.8,
        alpha=0.30
    )


    ax.set_axisbelow(
        True
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    ax.set_ylim(
        -0.45,
        n_features - 0.55
    )


    return scatter_handle


# ============================================================
# 17. Figure S10
#
# Keep horizontal comparison:
# 1-year vs 3-year lag
# ============================================================

fig, (
    ax1,
    ax2
) = plt.subplots(

    1,
    2,

    figsize=(
        18.6,
        6.3
    ),

    gridspec_kw={

        "width_ratios": [
            1,
            1
        ],

        "wspace":
            0.36
    }
)


# ============================================================
# 18. Panel A
# ============================================================

sc1 = draw_shap_panel(

    ax1,

    lag1,

    "(a) 1-year lag"
)


# ============================================================
# 19. Panel B
# ============================================================

sc2 = draw_shap_panel(

    ax2,

    lag3,

    "(b) 3-year lag"
)


# ============================================================
# 20. Full-height shared feature-value colorbar
#
# Match Figure S9 approach:
# colorbar height exactly equals Panel B.
# ============================================================

divider = make_axes_locatable(
    ax2
)


cax = divider.append_axes(

    "right",

    size=
        "2.7%",

    pad=
        0.10
)


cbar = fig.colorbar(
    sc2,
    cax=cax
)


cbar.set_label(
    "Feature value",
    fontsize=11
)


cbar.set_ticks(
    np.linspace(
        0,
        1,
        6
    )
)


cbar.set_ticklabels(
    [
        "0.0",
        "0.2",
        "0.4",
        "0.6",
        "0.8",
        "1.0"
    ]
)


cbar.ax.tick_params(
    labelsize=10
)


# ============================================================
# 21. Final layout
#
# Shorter labels allow much more actual SHAP plotting area.
# ============================================================

fig.subplots_adjust(

    left=
        0.155,

    right=
        0.965,

    bottom=
        0.13,

    top=
        0.92,

    wspace=
        0.36
)


# ============================================================
# 22. Save Figure S10
# ============================================================

figure_path = (
    figures_dir
    / "Figure_S10_lagged_environmental_predictors_sensitivity.tiff"
)


plt.savefig(

    figure_path,

    dpi=
        300,

    format=
        "tiff",

    bbox_inches=
        "tight"
)


plt.show()


# ============================================================
# 23. Output summary
# ============================================================

print("\n" + "=" * 75)
print("FIGURE S10 COMPLETED")
print("=" * 75)


print("\nModel performance:")
print(
    performance_path
)


print("\n1-year lag SHAP importance:")
print(
    lag1_importance_path
)


print("\n3-year lag SHAP importance:")
print(
    lag3_importance_path
)


print("\n1-year eligible observations:")
print(
    lag1_data_path
)


print("\n3-year eligible observations:")
print(
    lag3_data_path
)


print("\nFigure S10:")
print(
    figure_path
)

## 26. Figure S11: nested spatial-block cross-validation

Evaluate geographic generalizability with nested fivefold spatial-block validation and out-of-fold SHAP values.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap
import geopandas as gpd

from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import (
    GroupKFold,
    RandomizedSearchCV
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# FIGURE S11
# Nested fivefold spatial-block cross-validation
#
# Final modeling framework: LightGBM
#
# Panel (a): Out-of-fold SHAP summary
# Panel (b): Out-of-fold SHAP importance
# ============================================================


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)


# ============================================================
# 2. Paths
# ============================================================

model_data_path = PROCESSED_DIR / 'autism_model_final.csv'


shapefile_path = SPATIAL_DIR / 'tl_2020_36_unsd.shp'


tables_dir = TABLES_DIR


figures_dir = FIGURES_DIR


tables_dir.mkdir(
    parents=True,
    exist_ok=True
)

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. Variables
# ============================================================

TARGET = "Autism_prev1000"

ID_COL = "GEOID"

YEAR_COL = "Year"


FEATURES = [
    "PM25",
    "tmax_c",
    "ppt_mm",
    "Below_poverty_percent",
    "Bachelor_plus_percent",
    "under5_percent",
    "age5to14_percent",
    "age15to19_percent",
    "median_age",
    "Pop_density",
    "hispanic_percent",
    "asian_percent",
    "Forest_percent"
]


# ============================================================
# 4. Publication-friendly names
# ============================================================

FEATURE_NAMES = {

    "PM25":
        r"PM$_{2.5}$ concentration (µg/m³)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "under5_percent":
        "Population under 5 years (%)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "Pop_density":
        "Population density (persons/km²)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "asian_percent":
        "Asian population (%)",

    "Forest_percent":
        "Forest land cover (%)"
}


# ============================================================
# 5. Analysis settings
# ============================================================

BLOCK_SIZE_KM = 50

OUTER_FOLDS = 5

INNER_FOLDS = 4

N_RANDOM_SEARCH = 50


# ============================================================
# 6. Helper functions
# ============================================================

def clean_geoid(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


def calc_rmse(
    y_true,
    y_pred
):

    return float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )
    )


def get_shap_values(
    model,
    X
):

    explainer = shap.TreeExplainer(
        model
    )

    values = explainer.shap_values(
        X
    )

    if isinstance(
        values,
        list
    ):
        values = values[0]

    return np.asarray(
        values
    )


# ============================================================
# 7. Load final modeling data
# ============================================================

df = pd.read_csv(
    model_data_path,
    dtype={
        ID_COL: str
    }
)


df.columns = (
    df.columns
    .str.strip()
)


df[
    ID_COL
] = clean_geoid(
    df[
        ID_COL
    ]
)


df[
    YEAR_COL
] = pd.to_numeric(
    df[
        YEAR_COL
    ],
    errors="coerce"
)


required_cols = (
    [
        ID_COL,
        YEAR_COL,
        TARGET
    ]
    + FEATURES
)


missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]


if missing_cols:

    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


model_data = (
    df[
        required_cols
    ]
    .copy()
)


numeric_cols = (
    [
        YEAR_COL,
        TARGET
    ]
    + FEATURES
)


model_data[
    numeric_cols
] = (
    model_data[
        numeric_cols
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


model_data = (
    model_data
    .dropna(
        subset=[
            ID_COL,
            YEAR_COL,
            TARGET
        ]
        + FEATURES
    )
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. Load school-district boundaries
# ============================================================

geo = gpd.read_file(
    shapefile_path
)


# ------------------------------------------------------------
# Identify GEOID field
# ------------------------------------------------------------

geoid_candidates = [
    "GEOID",
    "GEOID20",
    "GEOID10"
]


shape_geoid = None


for candidate in geoid_candidates:

    if candidate in geo.columns:

        shape_geoid = candidate
        break


if shape_geoid is None:

    possible = [
        col
        for col in geo.columns
        if "geoid" in col.lower()
    ]

    if len(possible) != 1:

        raise ValueError(
            "Could not uniquely identify the GEOID field "
            f"in the shapefile. Possible fields: {possible}"
        )

    shape_geoid = possible[0]


geo[
    ID_COL
] = clean_geoid(
    geo[
        shape_geoid
    ]
)


# ============================================================
# 9. Geometry QC
# ============================================================

geo = (
    geo[
        geo.geometry.notna()
    ]
    .copy()
)


geo = (
    geo[
        ~geo.geometry.is_empty
    ]
    .copy()
)


# ============================================================
# 10. Project to EPSG:5070
#
# Appropriate projected CRS for CONUS distance calculations.
# ============================================================

geo_proj = geo.to_crs(
    "EPSG:5070"
)


centroids = (
    geo_proj
    .geometry
    .centroid
)


coordinate_table = pd.DataFrame({

    ID_COL:
        geo_proj[
            ID_COL
        ].values,

    "x_km":
        centroids.x.to_numpy()
        / 1000,

    "y_km":
        centroids.y.to_numpy()
        / 1000
})


coordinate_table = (
    coordinate_table
    .drop_duplicates(
        subset=[
            ID_COL
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. Join spatial coordinates to final modeling data
# ============================================================

spatial_data = model_data.merge(

    coordinate_table,

    on=
        ID_COL,

    how=
        "inner",

    validate=
        "many_to_one"
)


print("=" * 80)
print("SPATIAL JOIN QC")
print("=" * 80)


print(
    "Original modeling observations:",
    len(
        model_data
    )
)


print(
    "Spatially matched observations:",
    len(
        spatial_data
    )
)


print(
    "Matched school districts:",
    spatial_data[
        ID_COL
    ].nunique()
)


# ============================================================
# 12. Create 50-km spatial blocks
# ============================================================

spatial_data[
    "block_x"
] = np.floor(

    (
        spatial_data[
            "x_km"
        ]
        -
        spatial_data[
            "x_km"
        ].min()
    )

    /
    BLOCK_SIZE_KM

).astype(int)


spatial_data[
    "block_y"
] = np.floor(

    (
        spatial_data[
            "y_km"
        ]
        -
        spatial_data[
            "y_km"
        ].min()
    )

    /
    BLOCK_SIZE_KM

).astype(int)


spatial_data[
    "Spatial_Block"
] = (

    spatial_data[
        "block_x"
    ].astype(str)

    + "_"

    + spatial_data[
        "block_y"
    ].astype(str)
)


n_blocks = (
    spatial_data[
        "Spatial_Block"
    ]
    .nunique()
)


print(
    "Number of 50-km spatial blocks:",
    n_blocks
)


if n_blocks < OUTER_FOLDS:

    raise ValueError(
        "Insufficient spatial blocks for fivefold outer CV."
    )


# ============================================================
# 13. Save spatial-block assignments
#
# Useful for exact reproducibility.
# ============================================================

block_assignment_path = (
    tables_dir
    / "Table_S11_spatial_block_assignments.csv"
)


spatial_data[
    [
        ID_COL,
        YEAR_COL,
        "x_km",
        "y_km",
        "block_x",
        "block_y",
        "Spatial_Block"
    ]
].to_csv(
    block_assignment_path,
    index=False
)


# ============================================================
# 14. Modeling objects
#
# IMPORTANT:
# Year and x/y are NOT predictors.
#
# They are used only to organize spatial validation.
#
# Model contains exactly the same 13 substantive predictors
# as the primary analysis.
# ============================================================

X = spatial_data[
    FEATURES
].copy()


y = spatial_data[
    TARGET
].copy()


groups = spatial_data[
    "Spatial_Block"
].copy()


# ============================================================
# 15. LightGBM estimator
#
# deterministic=True + force_col_wise=True improves
# reproducibility across repeated runs.
# ============================================================

base_model = lgb.LGBMRegressor(

    objective=
        "regression",

    random_state=
        RANDOM_STATE,

    deterministic=
        True,

    force_col_wise=
        True,

    n_jobs=
        1,

    verbosity=
        -1,

    # Needed if subsample is tuned
    subsample_freq=
        1
)


# ============================================================
# 16. LightGBM hyperparameter search space
#
# Adapted from the original nested-CV framework,
# but for LightGBM.
# ============================================================

parameter_space = {

    "n_estimators":
        randint(
            200,
            701
        ),

    "learning_rate":
        loguniform(
            0.02,
            0.15
        ),

    "num_leaves":
        randint(
            20,
            81
        ),

    "max_depth":
        [
            -1,
            4,
            6,
            8,
            10
        ],

    "min_child_samples":
        randint(
            10,
            61
        ),

    "subsample":
        uniform(
            0.70,
            0.30
        ),

    "colsample_bytree":
        uniform(
            0.70,
            0.30
        ),

    "reg_alpha":
        loguniform(
            1e-4,
            1
        ),

    "reg_lambda":
        loguniform(
            0.1,
            10
        )
}


# ============================================================
# 17. Nested spatial-block CV
# ============================================================

outer_cv = GroupKFold(
    n_splits=
        OUTER_FOLDS
)


oof_predictions = np.full(
    len(
        spatial_data
    ),
    np.nan
)


oof_shap = np.full(

    shape=(
        len(
            spatial_data
        ),
        len(
            FEATURES
        )
    ),

    fill_value=
        np.nan
)


fold_results = []

best_parameter_rows = []


# ============================================================
# 18. Outer folds
# ============================================================

for fold, (
    train_idx,
    test_idx
) in enumerate(

    outer_cv.split(
        X,
        y,
        groups=
            groups
    ),

    start=
        1
):


    print("\n" + "=" * 80)

    print(
        f"OUTER FOLD {fold}"
    )

    print("=" * 80)


    X_train = (
        X.iloc[
            train_idx
        ]
        .copy()
    )


    X_test = (
        X.iloc[
            test_idx
        ]
        .copy()
    )


    y_train = (
        y.iloc[
            train_idx
        ]
        .copy()
    )


    y_test = (
        y.iloc[
            test_idx
        ]
        .copy()
    )


    train_groups = (
        groups.iloc[
            train_idx
        ]
        .copy()
    )


    test_groups = (
        groups.iloc[
            test_idx
        ]
        .copy()
    )


    train_geoids = set(
        spatial_data.iloc[
            train_idx
        ][
            ID_COL
        ]
    )


    test_geoids = set(
        spatial_data.iloc[
            test_idx
        ][
            ID_COL
        ]
    )


    # --------------------------------------------------------
    # Strict spatial separation checks
    # --------------------------------------------------------

    block_overlap = set(
        train_groups
    ).intersection(
        set(
            test_groups
        )
    )


    if block_overlap:

        raise ValueError(
            f"Outer fold {fold}: spatial block overlap detected."
        )


    district_overlap = (
        train_geoids
        .intersection(
            test_geoids
        )
    )


    if district_overlap:

        raise ValueError(
            f"Outer fold {fold}: school district overlap detected."
        )


    print(
        "Training observations:",
        len(
            train_idx
        )
    )


    print(
        "Validation observations:",
        len(
            test_idx
        )
    )


    print(
        "Training spatial blocks:",
        train_groups.nunique()
    )


    print(
        "Validation spatial blocks:",
        test_groups.nunique()
    )


    # ========================================================
    # 19. Inner grouped CV
    # ========================================================

    if (
        train_groups.nunique()
        < INNER_FOLDS
    ):

        raise ValueError(
            f"Outer fold {fold} has insufficient training "
            "blocks for fourfold inner CV."
        )


    inner_cv = GroupKFold(
        n_splits=
            INNER_FOLDS
    )


    # ========================================================
    # 20. Hyperparameter tuning
    # ========================================================

    search = RandomizedSearchCV(

        estimator=
            base_model,

        param_distributions=
            parameter_space,

        n_iter=
            N_RANDOM_SEARCH,

        scoring=
            "r2",

        cv=
            inner_cv,

        random_state=
            RANDOM_STATE
            + fold,

        n_jobs=
            -1,

        refit=
            True,

        verbose=
            0
    )


    search.fit(

        X_train,

        y_train,

        groups=
            train_groups
    )


    best_model = (
        search.best_estimator_
    )


    # ========================================================
    # 21. Outer-fold prediction
    # ========================================================

    predictions = best_model.predict(
        X_test
    )


    oof_predictions[
        test_idx
    ] = predictions


    fold_r2 = r2_score(
        y_test,
        predictions
    )


    fold_rmse = calc_rmse(
        y_test,
        predictions
    )


    fold_mae = mean_absolute_error(
        y_test,
        predictions
    )


    fold_results.append({

        "Fold":
            fold,

        "Training_observations":
            len(
                train_idx
            ),

        "Validation_observations":
            len(
                test_idx
            ),

        "Training_districts":
            len(
                train_geoids
            ),

        "Validation_districts":
            len(
                test_geoids
            ),

        "Training_blocks":
            train_groups.nunique(),

        "Validation_blocks":
            test_groups.nunique(),

        "R2":
            fold_r2,

        "RMSE":
            fold_rmse,

        "MAE":
            fold_mae
    })


    best_parameter_rows.append({

        "Fold":
            fold,

        "Best_inner_CV_R2":
            search.best_score_,

        **search.best_params_
    })


    print(
        f"Outer R² = {fold_r2:.4f}"
    )


    print(
        f"Outer RMSE = {fold_rmse:.4f}"
    )


    print(
        f"Outer MAE = {fold_mae:.4f}"
    )


    # ========================================================
    # 22. Out-of-fold SHAP
    #
    # Only validation observations receive SHAP values.
    # ========================================================

    fold_shap = get_shap_values(
        best_model,
        X_test
    )


    oof_shap[
        test_idx,
        :
    ] = fold_shap


# ============================================================
# 23. Verify complete out-of-fold predictions / SHAP
# ============================================================

if np.isnan(
    oof_predictions
).any():

    raise ValueError(
        "Some observations lack out-of-fold predictions."
    )


if np.isnan(
    oof_shap
).any():

    raise ValueError(
        "Some observations lack out-of-fold SHAP values."
    )


# ============================================================
# 24. Save fold-specific results
# ============================================================

fold_results_df = pd.DataFrame(
    fold_results
)


fold_results_path = (
    tables_dir
    / "Table_S11_nested_spatial_CV_fold_metrics.csv"
)


fold_results_df.to_csv(
    fold_results_path,
    index=False
)


best_parameters_df = pd.DataFrame(
    best_parameter_rows
)


best_parameters_path = (
    tables_dir
    / "Table_S11_nested_spatial_CV_best_parameters.csv"
)


best_parameters_df.to_csv(
    best_parameters_path,
    index=False
)


# ============================================================
# 25. Performance summary
# ============================================================

metric_summary = pd.DataFrame({

    "Metric": [
        "R2",
        "RMSE",
        "MAE"
    ],

    "Mean": [

        fold_results_df[
            "R2"
        ].mean(),

        fold_results_df[
            "RMSE"
        ].mean(),

        fold_results_df[
            "MAE"
        ].mean()
    ],

    "SD": [

        fold_results_df[
            "R2"
        ].std(
            ddof=1
        ),

        fold_results_df[
            "RMSE"
        ].std(
            ddof=1
        ),

        fold_results_df[
            "MAE"
        ].std(
            ddof=1
        )
    ]
})


# Add pooled OOF metrics
pooled_r2 = r2_score(
    y,
    oof_predictions
)


pooled_rmse = calc_rmse(
    y,
    oof_predictions
)


pooled_mae = mean_absolute_error(
    y,
    oof_predictions
)


metric_summary[
    "Pooled_OOF"
] = [
    pooled_r2,
    pooled_rmse,
    pooled_mae
]


metric_summary_path = (
    tables_dir
    / "Table_S11_nested_spatial_CV_metric_summary.csv"
)


metric_summary.to_csv(
    metric_summary_path,
    index=False
)


print("\n" + "=" * 80)
print("NESTED SPATIAL CV PERFORMANCE SUMMARY")
print("=" * 80)


print(
    metric_summary.to_string(
        index=False
    )
)


# ============================================================
# 26. Save OOF predictions
# ============================================================

prediction_output = pd.DataFrame({

    ID_COL:
        spatial_data[
            ID_COL
        ],

    YEAR_COL:
        spatial_data[
            YEAR_COL
        ],

    "Spatial_Block":
        spatial_data[
            "Spatial_Block"
        ],

    "Observed_ASD_prevalence":
        y,

    "Predicted_ASD_prevalence":
        oof_predictions,

    "Residual":
        y
        - oof_predictions
})


prediction_path = (
    tables_dir
    / "Table_S11_nested_spatial_CV_OOF_predictions.csv"
)


prediction_output.to_csv(
    prediction_path,
    index=False
)


# ============================================================
# 27. OOF SHAP importance
# ============================================================

importance = pd.DataFrame({

    "Feature":
        FEATURES,

    "Display":
        [
            FEATURE_NAMES[
                feature
            ]
            for feature
            in FEATURES
        ],

    "MeanAbsSHAP":
        np.abs(
            oof_shap
        ).mean(
            axis=0
        ),

    "MeanSHAP":
        oof_shap.mean(
            axis=0
        )
})


importance = (
    importance
    .sort_values(
        "MeanAbsSHAP",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


importance.insert(

    0,

    "Rank",

    range(
        1,
        len(
            importance
        )
        + 1
    )
)


importance_path = (
    tables_dir
    / "Table_S11_nested_spatial_CV_SHAP_importance.csv"
)


importance.to_csv(
    importance_path,
    index=False
)


print("\nOOF SHAP importance:")

print(
    importance.to_string(
        index=False
    )
)


# ============================================================
# 28. Prepare plotting data
# ============================================================

feature_order = (
    importance[
        "Feature"
    ]
    .tolist()
)


display_order = (
    importance[
        "Display"
    ]
    .tolist()
)


n_features = len(
    feature_order
)


y_positions = (
    np.arange(
        n_features
    )[::-1]
)


shap_df = pd.DataFrame(

    oof_shap,

    columns=
        FEATURES,

    index=
        X.index
)


# ============================================================
# 29. Figure style
#
# Match revised Figures S8/S9
# ============================================================

plt.rcParams.update({

    "font.family":
        "Arial",

    "axes.unicode_minus":
        True,

    "font.size":
        11.5,

    "axes.titlesize":
        14,

    "axes.labelsize":
        13,

    "xtick.labelsize":
        11,

    "ytick.labelsize":
        11
})


# ============================================================
# 30. Figure S11
#
# Same A:B layout as Figure S8/S9
# ============================================================

fig, (
    ax1,
    ax2
) = plt.subplots(

    1,
    2,

    figsize=(
        16,
        6.0
    ),

    gridspec_kw={

        "width_ratios": [
            2.5,
            1.5
        ],

        "wspace":
            0.24
    }
)


# ============================================================
# 31. Panel A
# Out-of-fold SHAP summary
# ============================================================

scatter_handle = None


for i, feature in enumerate(
    feature_order
):

    y0 = (
        y_positions[
            i
        ]
    )


    shap_vals = (
        shap_df[
            feature
        ]
        .to_numpy()
    )


    feature_vals = (
        X[
            feature
        ]
        .to_numpy()
    )


    valid = ~(
        np.isnan(
            shap_vals
        )
        |
        np.isnan(
            feature_vals
        )
    )


    shap_vals = (
        shap_vals[
            valid
        ]
    )


    feature_vals = (
        feature_vals[
            valid
        ]
    )


    # --------------------------------------------------------
    # Robust within-feature color scaling
    # --------------------------------------------------------

    low = np.nanpercentile(
        feature_vals,
        5
    )


    high = np.nanpercentile(
        feature_vals,
        95
    )


    feature_clip = np.clip(
        feature_vals,
        low,
        high
    )


    if high > low:

        feature_color = (

            feature_clip
            - low

        ) / (

            high
            - low
        )

    else:

        feature_color = np.full(
            len(
                feature_clip
            ),
            0.5,
            dtype=float
        )


    # --------------------------------------------------------
    # Reproducible vertical jitter
    # --------------------------------------------------------

    rng = np.random.default_rng(
        RANDOM_STATE
        + i
    )


    jitter = rng.normal(
        0,
        0.08,
        size=len(
            shap_vals
        )
    )


    scatter_handle = ax1.scatter(

        shap_vals,

        np.full_like(
            shap_vals,
            y0,
            dtype=float
        )
        + jitter,

        c=
            feature_color,

        cmap=
            "coolwarm",

        vmin=
            0,

        vmax=
            1,

        s=
            12,

        alpha=
            0.75,

        edgecolors=
            "none",

        rasterized=
            True
    )


# Zero line
ax1.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)


# Y labels
ax1.set_yticks(
    y_positions
)


ax1.set_yticklabels(
    display_order,
    fontsize=11
)


# X label
ax1.set_xlabel(
    "SHAP value (effect on predicted autism prevalence)",
    fontsize=13
)


ax1.set_ylabel(
    ""
)


# Panel title
ax1.set_title(
    "(a) Out-of-fold SHAP summary",
    fontsize=14,
    pad=8
)


# Grid
ax1.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.30
)


ax1.set_axisbelow(
    True
)


ax1.spines[
    "top"
].set_visible(
    False
)


ax1.spines[
    "right"
].set_visible(
    False
)


ax1.set_ylim(
    -0.45,
    n_features - 0.55
)


# ============================================================
# 32. Full-height colorbar for Panel A
# ============================================================

divider = make_axes_locatable(
    ax1
)


cax = divider.append_axes(

    "right",

    size=
        "2.8%",

    pad=
        0.10
)


cbar = fig.colorbar(

    scatter_handle,

    cax=
        cax
)


cbar.set_label(
    "Feature value",
    fontsize=11
)


cbar.set_ticks(
    np.linspace(
        0,
        1,
        6
    )
)


cbar.set_ticklabels(
    [
        "0.0",
        "0.2",
        "0.4",
        "0.6",
        "0.8",
        "1.0"
    ]
)


cbar.ax.tick_params(
    labelsize=10
)


# ============================================================
# 33. Panel B
# Mean absolute OOF SHAP importance
# ============================================================

bar_values = (
    importance[
        "MeanAbsSHAP"
    ]
    .to_numpy()
)


bar_colors = (
    plt.cm.viridis(
        np.linspace(
            0.35,
            0.92,
            n_features
        )
    )
)


ax2.barh(

    y_positions,

    bar_values,

    height=
        0.52,

    color=
        bar_colors
)


# ============================================================
# Numeric labels
# ============================================================

xmax = (
    bar_values.max()
)


right_margin = (
    xmax
    * 0.20
)


text_offset = (
    xmax
    * 0.02
)


ax2.set_xlim(
    0,
    xmax
    + right_margin
)


for y_pos, value in zip(

    y_positions,

    bar_values
):

    ax2.text(

        value
        + text_offset,

        y_pos,

        f"{value:.2f}",

        va=
            "center",

        ha=
            "left",

        fontsize=
            10.5
    )


# Do not repeat variable names
ax2.set_yticks(
    y_positions
)


ax2.set_yticklabels(
    []
)


ax2.tick_params(
    axis="y",
    length=0
)


ax2.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=13
)


ax2.set_ylabel(
    ""
)


ax2.set_title(
    "(b) SHAP feature importance",
    fontsize=14,
    pad=8
)


ax2.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.30
)


ax2.set_axisbelow(
    True
)


ax2.spines[
    "top"
].set_visible(
    False
)


ax2.spines[
    "right"
].set_visible(
    False
)


ax2.set_ylim(
    -0.45,
    n_features - 0.55
)


# ============================================================
# 34. Final layout
# ============================================================

fig.subplots_adjust(

    left=
        0.305,

    right=
        0.965,

    bottom=
        0.13,

    top=
        0.92,

    wspace=
        0.24
)


# ============================================================
# 35. Save Figure S11
# ============================================================

figure_path = (
    figures_dir
    / "Figure_S11_nested_spatial_CV_SHAP.tiff"
)


plt.savefig(

    figure_path,

    dpi=
        300,

    format=
        "tiff",

    bbox_inches=
        "tight"
)


plt.show()


# ============================================================
# 36. Final output summary
# ============================================================

print("\n" + "=" * 80)
print("FIGURE S11 COMPLETED")
print("=" * 80)


print("\nSpatial-block assignments:")
print(
    block_assignment_path
)


print("\nFold metrics:")
print(
    fold_results_path
)


print("\nBest LightGBM parameters by fold:")
print(
    best_parameters_path
)


print("\nPerformance summary:")
print(
    metric_summary_path
)


print("\nOut-of-fold predictions:")
print(
    prediction_path
)


print("\nSHAP importance:")
print(
    importance_path
)


print("\nFigure S11:")
print(
    figure_path
)

## 27. Table S11: annual mean values

Summarize annual mean ASD prevalence and predictor values for 2010–2022.

In [ ]:
import pandas as pd
from pathlib import Path


# ============================================================
# Table S11
# Annual mean values of autism prevalence and predictors
# New final analytical dataset
# ============================================================


# ============================================================
# 1. Paths
# ============================================================

input_file = PROCESSED_DIR / 'autism_model_final.csv'


output_dir = TABLES_DIR


output_dir.mkdir(
    parents=True,
    exist_ok=True
)


output_file = (
    output_dir
    / "Table_S11_annual_mean_values_2010_2022.csv"
)


# ============================================================
# 2. Load final analytical dataset
# ============================================================

df = pd.read_csv(
    input_file,
    dtype={
        "GEOID": str
    }
)


df.columns = (
    df.columns
    .str.strip()
)


# ============================================================
# 3. Variables
# ============================================================

target = "Autism_prev1000"


variables = [
    target,
    "PM25",
    "Pop_density",
    "Bachelor_plus_percent",
    "hispanic_percent",
    "Forest_percent",
    "tmax_c",
    "Below_poverty_percent",
    "age15to19_percent",
    "median_age",
    "age5to14_percent",
    "ppt_mm",
    "asian_percent",
    "under5_percent"
]


# ============================================================
# 4. Check required columns
# ============================================================

required_cols = (
    ["Year"]
    + variables
)


missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]


if missing_cols:

    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# ============================================================
# 5. Convert variables to numeric
# ============================================================

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)


for col in variables:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# 6. Keep study years only
# ============================================================

df = (
    df[
        (df["Year"] >= 2010)
        &
        (df["Year"] <= 2022)
    ]
    .copy()
)


df["Year"] = (
    df["Year"]
    .astype(int)
)


# ============================================================
# 7. QC: observations by year
# ============================================================

n_by_year = (
    df.groupby("Year")
    .size()
)


print("=" * 75)
print("OBSERVATIONS BY YEAR")
print("=" * 75)

print(
    n_by_year
)


# ============================================================
# 8. Calculate annual arithmetic means
#
# Each value is the mean across school-district observations
# available in that year.
# ============================================================

annual_means = (
    df.groupby("Year")[variables]
    .mean()
)


# Force complete year order
years = list(
    range(
        2010,
        2023
    )
)


annual_means = (
    annual_means
    .reindex(years)
)


# ============================================================
# 9. Transpose:
#
# rows    = variables
# columns = years
# ============================================================

table_s11 = (
    annual_means
    .T
)


# ============================================================
# 10. Publication-friendly variable names
# ============================================================

variable_name_map = {

    "Autism_prev1000":
        "Autism prevalence (‰)",

    "PM25":
        "PM2.5 concentration (µg/m³)",

    "Pop_density":
        "Population density (persons/km²)",

    "Bachelor_plus_percent":
        "Bachelor’s degree or higher (%)",

    "hispanic_percent":
        "Hispanic or Latino population (%)",

    "Forest_percent":
        "Forest land cover (%)",

    "tmax_c":
        "Mean daily maximum temperature (°C)",

    "Below_poverty_percent":
        "Families below the poverty line (%)",

    "age15to19_percent":
        "Population aged 15–19 years (%)",

    "median_age":
        "Median age (years)",

    "age5to14_percent":
        "Population aged 5–14 years (%)",

    "ppt_mm":
        "Annual precipitation (mm)",

    "asian_percent":
        "Asian population (%)",

    "under5_percent":
        "Population under 5 years (%)"
}


table_s11 = table_s11.rename(
    index=variable_name_map
)


table_s11.index.name = (
    "Variable"
)


# ============================================================
# 11. Round to two decimals
# ============================================================

table_s11 = (
    table_s11
    .round(2)
)


# ============================================================
# 12. Save Table S11
# ============================================================

table_s11.to_csv(
    output_file
)


# ============================================================
# 13. Print
# ============================================================

print("\n" + "=" * 75)
print("UPDATED TABLE S11")
print("=" * 75)


print(
    table_s11.to_string()
)


print("\nSaved to:")

print(
    output_file
)

## 28. Pooled out-of-fold performance

Calculate pooled R², RMSE, and MAE from the spatial-block out-of-fold predictions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ============================================================
# 1. File path
# ============================================================

file_path = str(TABLES_DIR / 'Table_S11_nested_spatial_CV_OOF_predictions.csv')


# ============================================================
# 2. Read OOF predictions
# ============================================================

df = pd.read_csv(file_path)


# ============================================================
# 3. Check required columns
# ============================================================

required_cols = [
    "Observed_ASD_prevalence",
    "Predicted_ASD_prevalence"
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


# ============================================================
# 4. Remove missing values, if any
# ============================================================

data = df[
    required_cols
].dropna().copy()


y_true = data[
    "Observed_ASD_prevalence"
]

y_pred = data[
    "Predicted_ASD_prevalence"
]


# ============================================================
# 5. Calculate pooled out-of-fold performance
# ============================================================

pooled_r2 = r2_score(
    y_true,
    y_pred
)

pooled_rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

pooled_mae = mean_absolute_error(
    y_true,
    y_pred
)


# ============================================================
# 6. Print results
# ============================================================

print("=" * 60)
print("POOLED OUT-OF-FOLD PERFORMANCE")
print("=" * 60)

print(
    f"N = {len(data)}"
)

print(
    f"Pooled OOF R² = {pooled_r2:.4f}"
)

print(
    f"Pooled OOF RMSE = {pooled_rmse:.4f}"
)

print(
    f"Pooled OOF MAE = {pooled_mae:.4f}"
)